# SiLU forward and derivative in one kernel

SiLU (also called swish) is the activation inside the SwiGLU MLPs of most current LLMs:

$$\mathrm{silu}(x) = x\,\sigma(x), \qquad \sigma(x) = \frac{1}{1 + e^{-x}}, \qquad \frac{d}{dx}\mathrm{silu}(x) = \sigma(x)\bigl(1 + x\,(1 - \sigma(x))\bigr)$$

Write `silu_fwd_bwd(x)` that computes **both** the activation and its derivative in a single Triton kernel: one read of `x`, two writes.

## Inputs
- `x`: contiguous `float32` CUDA tensor, any shape, at least one element.

## Output
- A tuple `(y, dy)` of two new `float32` CUDA tensors with the shape of `x`: `y = silu(x)` and `dy = silu'(x)`.

## Requirements
- Compute in `float32`. Large inputs of either sign (e.g. $\pm 100$) must give finite results: `silu(100) = 100`, `silu(-100) ≈ 0`, and the derivative tends to 1 and 0.
- Flat 1-D grid, `BLOCK` a `tl.constexpr` power of two, masked loads and stores.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def silu_fwd_bwd(x):
    """x: float32 CUDA tensor. Return (silu(x), d silu / dx) from one Triton kernel."""
    raise NotImplementedError


@triton.jit
def _silu_kernel(x_ptr, y_ptr, dy_ptr, n, BLOCK: tl.constexpr):
    # TODO: load a block of x, compute sigmoid once, store y and dy
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [0.0, 1.0, -1.0, 2.0], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [0.0, 0.7310585975646973, -0.2689414322376251, 1.7615940570831299], "dtype": "float32"}, {"__nd__": [0.5, 0.9276705384254456, 0.07232948392629623, 1.0907843112945557], "dtype": "float32"}]}}, {"a": [{"__nd__": [[3.0, -3.0], [0.5, -0.5]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [[2.857722282409668, -0.14227762818336487], [0.3112296760082245, -0.1887703388929367]], "dtype": "float32"}, {"__nd__": [[1.0881041288375854, -0.08810409903526306], [0.7399612069129944, 0.260038822889328]], "dtype": "float32"}]}}, {"a": [{"__nd__": [100.0, -100.0, 30.0, -30.0, 88.0, -88.0], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [100.0, -0.0, 30.0, -2.8072871718215486e-12, 88.0, -5.3280493069724375e-37], "dtype": "float32"}, {"__nd__": [1.0, -0.0, 1.0, -2.7137107665164972e-12, 1.0, -5.2675033481724166e-37], "dtype": "float32"}]}}, {"a": [{"__nd__": [-1.2784645557403564], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [-0.27846455574035645], "dtype": "float32"}, {"__nd__": [0.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [0.1367710679769516, 5.438990116119385, 4.898884296417236, -2.0412282943725586, -1.191878080368042, -2.10953688621521, 2.2789053916931152, -0.22425775229930878, 2.9875423908233643, -7.389299392700195, 6.266195297241211, -0.3857286274433136, 2.7215137481689453, -0.5462653636932373, -1.5163942575454712, 1.852440595626831, 3.2980542182922363, -0.8101195096969604, -0.6111447215080261, 2.7427945137023926, -3.4813625812530518, -6.057534217834473, 1.5799274444580078, -2.6822633743286133, -7.681362152099609, -3.2562146186828613, -1.8703901767730713, -4.772809982299805, -5.969855308532715, 0.14655131101608276, 3.5889971256256104, -0.9325283169746399, -2.974384069442749, 1.5399752855300903, 2.868943214416504, -1.2000423669815063, 2.17867112159729, 4.171502113342285, -0.8278257250785828, -3.2540621757507324, 1.3906023502349854, 0.9901829361915588, 4.395251274108887, -5.1383233070373535, -2.646451711654663, -3.352667808532715, -6.936059474945068, 0.5057381987571716, 2.1112167835235596, -2.955160140991211, 5.542588233947754, 3.2876973152160645, 2.5095059871673584, 1.6068283319473267, 3.822678327560425, -5.3279194831848145, 2.455718517303467, 2.411107301712036, -7.070874214172363, 1.388120412826538, -1.001685380935669, 3.1260907649993896, -1.7562487125396729, -0.07296343147754669, 1.371406078338623, -3.505046844482422, 2.3943865299224854, -0.41985276341438293, 1.9699305295944214, -2.0873501300811768, 4.34480619430542, 2.4208078384399414, -0.712100088596344, 2.5278286933898926, 5.039020538330078, 7.16470193862915, -6.294305324554443, 3.532527208328247, 1.860274076461792, -0.37544313073158264, -4.026659965515137, 5.028754234313965, -5.046951770782471, 2.267781972885132, 5.20747184753418, -6.398677349090576, -1.210071325302124, -5.236867427825928, 0.9762164354324341, 6.05750036239624, 8.094223976135254, -7.112457752227783, -2.2997961044311523, 2.814180374145508, 6.317490577697754, 1.684842586517334, -2.9846079349517822, 1.1885260343551636, -0.06647680699825287, -0.8149628639221191, -2.9378843307495117, 1.5490390062332153, 1.2315185070037842, -0.37193650007247925, -0.8867517709732056, -5.1396660804748535, -1.9447026252746582, 4.825798511505127, -0.7622330188751221, -5.7587890625, 5.337771415710449, 2.1210615634918213, 8.432348251342773, 0.2500486671924591, -1.8455382585525513, -5.790578842163086, 5.295353412628174, 10.278045654296875, -3.2837398052215576, -2.588350534439087, 2.3847551345825195, -3.3213818073272705, -1.0822334289550781, -1.3993507623672485, 0.7678355574607849, 4.379273891448975, 0.08827035874128342, 3.675640344619751, -1.6795555353164673, 1.3111941814422607, -8.552902221679688, -5.799792289733887, 3.1836535930633545, -2.360597610473633, 2.31965970993042, 2.169377088546753, 5.2891154289245605, 3.247436285018921, 4.067965507507324, -0.4466853141784668, -2.7931406497955322, -2.9262351989746094, -1.9521757364273071, -4.519316673278809, -2.18977427482605, -0.37027621269226074, 1.0064477920532227, -1.3555631637573242, -7.69473123550415, -0.2891318202018738, 0.9013831615447998, 4.337902545928955, 2.311455488204956, -2.5742428302764893, -2.895106315612793, 8.042389869689941, 3.0265517234802246, 7.325759410858154, 8.517778396606445, -3.272334575653076, 1.5410772562026978, 1.8326680660247803, 2.2384488582611084, 2.1676089763641357, 0.8082185387611389, 0.6965169906616211, -6.009999752044678, -0.6616047620773315, -2.990478754043579, 0.5043492317199707, -1.8701850175857544, 2.4740829467773438, 3.2763028144836426, 1.2347694635391235, 1.2646725177764893, 0.3717867434024811, -1.791191577911377, -0.6580050587654114, -1.9825842380523682, 1.5519485473632812, 0.0564536489546299, 2.325164794921875, -5.314593315124512, 3.5511157512664795, -3.0507850646972656, -2.937128782272339, -0.7897719144821167, -2.2535383701324463, 1.1645008325576782, -2.296658992767334, -4.277910232543945, -3.383211135864258, 5.247933387756348, 0.17703332006931305, -4.670121192932129, 0.03350961208343506, -6.223785877227783, 7.158266067504883, -6.097912788391113, 1.9150108098983765, 2.1745026111602783, -5.8000311851501465, 1.201106071472168, 3.9886882305145264, 1.8703011274337769, 1.0440651178359985, 3.79638671875, 0.6436181664466858, 1.3457010984420776, -0.5068461894989014, 2.5272228717803955, -3.7655365467071533, 3.167052984237671, -2.0212061405181885, -4.362613677978516, 1.4609957933425903, 6.771821975708008, 3.846712827682495, -2.0625994205474854, 2.792238712310791, -1.8168364763259888, -0.49607783555984497, 0.35811397433280945, -9.26887321472168, 0.766459047794342, -4.118436813354492, -2.7895944118499756, -5.897561550140381, -6.066232681274414, -3.773057699203491, 3.302389144897461, 6.664165496826172, -0.10082041472196579, 4.36716365814209, -1.0559226274490356, -7.647891521453857, 0.5996664762496948, 1.782593011856079, -1.7144662141799927, 1.2088969945907593, 2.290225028991699, -3.453630208969116, -5.907336235046387, -0.885016679763794, -0.8442421555519104, -1.4106528759002686, 3.9489870071411133, 6.902355194091797, -1.6665880680084229, 2.795546770095825, 3.7637245655059814, 2.852625608444214, 4.189945220947266, -1.5569758415222168, 2.9592254161834717, 8.097928047180176, 3.2146623134613037, -2.479574680328369, 2.430850028991699, 5.049520969390869, 1.4759451150894165, -2.2536027431488037, 6.1723222732543945, -4.999068260192871, 2.013700246810913, -0.06470229476690292, -4.774662971496582, 4.877798557281494, 1.3915504217147827, -4.6671929359436035, 2.4069135189056396, -1.726401686668396, -7.611154079437256, -2.7556204795837402, 1.0542057752609253, 2.542811632156372, 0.6622596383094788, 0.18066003918647766, 1.879960060119629, -0.843904435634613, 4.707818508148193, 0.5945665836334229, 1.175686240196228, 2.163195848464966, -4.228675365447998, -2.836907386779785, 6.114316463470459, 1.341234803199768, -1.1304643154144287, 1.365934133529663, -1.926488995552063, 1.1142652034759521, -2.6634843349456787, 1.104075312614441, -6.310810089111328, 5.323678970336914, -2.0518314838409424, -6.324965000152588, -0.8953667879104614, -1.4839848279953003, 0.6095936894416809, -4.581002235412598, 1.58986234664917, 14.494270324707031, -5.114670753479004, 1.3508622646331787, -1.2428754568099976, 0.7704331874847412, -7.247236251831055, -4.604037761688232, 1.838638186454773, -0.15367208421230316, 6.540952682495117, -2.8191895484924316, 0.6979981064796448, 11.652485847473145, -3.0622458457946777, -3.6153340339660645, -0.14052033424377441, -0.17825071513652802, 3.37666916847229, 0.5183035135269165, -3.0269572734832764, 0.8456861972808838, 10.554387092590332, 5.087094783782959, -11.128676414489746, -0.5680676698684692, -3.5003621578216553, 2.4890501499176025, -0.6930539011955261, 7.730782985687256, 3.724942207336426, 3.5616116523742676, 0.7462842464447021, -0.16119061410427094, 1.3607196807861328, 5.181118965148926, 2.137559175491333, -1.4045302867889404, 5.100338459014893, 0.7684414982795715, -0.31046023964881897, -3.1655073165893555, -2.2639691829681396, -2.544187068939209, -11.169428825378418, 4.067384719848633, 2.2659103870391846, -0.8570823669433594, 4.183935165405273, 1.7519510984420776, 1.4862699508666992, -9.899054527282715, -0.9320636987686157, 2.128756523132324, 6.438747406005859, 7.361179351806641, 5.703352928161621, -4.646438121795654, -8.790685653686523, 2.2818782329559326, 0.11046334356069565, 3.850213050842285, -0.5035374760627747, 0.8790198564529419, -5.999956130981445, 1.7920386791229248, -1.779593825340271, 1.3999414443969727, 1.837623953819275, -0.4688885807991028, 1.2672045230865479, -0.05442812293767929, 3.0367980003356934, 2.006971836090088, -2.1888654232025146, -2.35160756111145, 6.974955081939697, -0.4063819944858551, 3.967231035232544, 1.5735429525375366, -2.2564499378204346, -2.9958951473236084, 0.48050275444984436, 1.0266176462173462, 2.4574549198150635, -5.0677337646484375, -2.3787331581115723, -0.3794648051261902, 6.402997970581055, -9.440713882446289, 1.7564465999603271, -2.1601834297180176, 1.7255094051361084, -1.967915654182434, 1.4582042694091797, 6.258552074432373, 1.4167859554290771, 1.7335997819900513, 1.4914315938949585, -3.001254081726074, 2.1720714569091797, -0.833386242389679, -3.246854543685913, -0.15419170260429382, -0.9091538786888123, 3.0139191150665283, -0.1713569015264511, -2.1177682876586914, 5.823886394500732, 3.9535510540008545, 5.078917980194092, 2.4306695461273193, -0.4387959837913513, -1.6763736009597778, -1.11735999584198, 3.790762424468994, 0.4476131200790405, -5.93137264251709, -0.15633119642734528, -5.664416790008545, 0.6634633541107178, -4.81527853012085, -5.055744647979736, 0.4743178188800812, -0.3128573000431061, -6.6470465660095215, 2.9928948879241943, 8.652386665344238, -5.925502777099609, 7.2793989181518555, -2.751157760620117, -1.9763624668121338, -0.7155488729476929, -5.781108856201172, 5.196508884429932, 1.970774531364441, -3.005816698074341, -1.1083086729049683, -6.0654377937316895, -1.3910309076309204, 3.7448525428771973, -6.861159324645996, -1.7449612617492676, 0.587166965007782, 4.192957401275635, 5.397322177886963, -1.8290117979049683, -2.9612274169921875, 1.8015986680984497, -1.453019142150879, -1.2400774955749512, 4.32073450088501, 6.758951663970947, 1.0634702444076538, -0.7688590884208679, 2.285047769546509, -7.550063133239746, -0.07299625128507614, -4.242307186126709, 9.557711601257324, 2.8807997703552246, 0.5654793977737427, -0.7774584889411926, -10.059823989868164, 2.8512179851531982, 6.517475605010986, -2.9152796268463135, 2.7336180210113525, 2.8901727199554443, -6.192510604858398, 3.7429614067077637, 5.419124126434326, 0.9139204621315002, -0.2527279853820801, 3.1518540382385254, -6.163015842437744, 0.37312281131744385, 4.217367649078369, -4.969741344451904, 3.970284938812256, -1.672918677330017, -8.323307991027832, -0.4229918122291565, -3.8875863552093506, 1.27703857421875, -4.688564777374268, 4.201103687286377, -2.9553213119506836, 1.4254299402236938, -2.375919818878174, -1.291621208190918, 4.767040252685547, 0.8611137866973877, -1.5801283121109009, -5.0060319900512695, 4.639805316925049, 2.9312446117401123, -1.3061423301696777, 0.9732440710067749, -4.11830997467041, -4.059379577636719, -3.304750919342041, 1.4137089252471924, 1.183383584022522, -2.9812116622924805, 2.9510984420776367, 3.1387741565704346, 0.6916326284408569, 4.153554439544678, -1.1494498252868652, -0.4520845413208008, 1.891434669494629, -0.13617603480815887, 3.1755967140197754, -5.010672569274902, 5.703060150146484, -3.1250858306884766, -0.6587914228439331, -5.510170936584473, 2.4008893966674805, 5.095405578613281, 4.2841877937316895, 3.3334999084472656, -6.228572845458984, -3.1286721229553223, -1.8502519130706787, -3.7389371395111084, 4.490780830383301, 0.7379372715950012, -0.09791121631860733, 1.9501429796218872, -3.80647611618042, -3.6334028244018555, -3.525847911834717, 1.9904837608337402, 1.3663854598999023, 2.3123698234558105, 4.31353235244751, -5.429203510284424, -2.2419774532318115, 0.590152382850647, -5.394972324371338, -2.1287009716033936, 4.056115627288818, 4.410576343536377, 4.306827545166016, 0.17207074165344238, -0.8478701710700989, 4.318098545074463, -1.4905840158462524, 1.352955937385559, 5.3011155128479, -1.4702831506729126, -2.442209243774414, -0.037485092878341675, 2.393516778945923, 1.9296361207962036, 0.6234310865402222, 2.2256805896759033, -6.39839506149292, -3.4370639324188232, 0.6529044508934021, -3.714646100997925, -5.366697788238525, -4.831279277801514, -3.6726555824279785, -0.713899552822113, -4.090503215789795, 0.9395953416824341, 6.784896373748779, -0.5905052423477173, 5.098795413970947, 1.1537269353866577, -4.4792938232421875, -1.0256383419036865, -3.9273390769958496, 3.0717415809631348, -4.612381458282471, 5.290290832519531, 2.3127517700195312, -8.089240074157715, 3.1373655796051025, 6.21206521987915, -3.162094831466675, 3.9410088062286377, -7.193526744842529, 1.0086264610290527, -6.645557403564453, 1.9295639991760254, -7.151851177215576, -10.53148078918457, 1.584372639656067, 9.151558876037598, -2.382678270339966, 3.320662260055542, 0.4428443908691406, -1.2940189838409424, 2.4615371227264404, 6.569621562957764, -4.940512180328369, 2.2555878162384033, 1.3805416822433472, -5.69738245010376, 6.458843231201172, 7.7974348068237305, -3.8110878467559814, 6.951601982116699, 0.22458447515964508, -0.652993381023407, 6.013795852661133, 0.9097314476966858, 1.292703628540039, 6.070699691772461, 5.15287971496582, 3.5725977420806885, -5.602199554443359, -6.2895612716674805, 4.5045647621154785, -3.706878185272217, 0.593934178352356, -4.296288013458252, -0.7347460985183716, -5.114780426025391, 6.52069616317749, -4.075649738311768, -0.4879770576953888, -0.7527979016304016, 4.8243913650512695, -5.679590225219727, -0.1333681344985962, -0.8369370698928833, 2.119702100753784, 1.8880671262741089, -1.8549861907958984, 0.9583439826965332, -1.498524785041809, 1.872460961341858, -0.4103206694126129, -2.6222763061523438, -0.11022809147834778, 1.4551358222961426, 9.063529968261719, -1.9649274349212646, 1.2266322374343872, 5.852147102355957, -8.312674522399902, -6.9721527099609375, -4.754637241363525, -0.45694616436958313, 9.759906768798828, 1.5654678344726562, 4.832301139831543, -0.8881574273109436, -9.854894638061523, -9.25455379486084, 4.074476718902588, 0.7394381761550903, -5.193996906280518, 4.469282627105713, -6.155819892883301, 4.53696870803833, 5.805363178253174, -3.059293508529663, -4.432467937469482, 3.3773081302642822, -4.381154537200928, 3.4475457668304443, -4.7809553146362305, 3.230314254760742, 0.37546777725219727, -1.7311135530471802, -3.5827386379241943, 0.950299084186554, -7.570642471313477, -1.1038566827774048, -1.5846166610717773, 2.713439702987671, -5.0152812004089355, 6.853752613067627, -1.2463231086730957, -4.552670001983643, 5.544314861297607, 6.651724338531494, -0.75108802318573, -3.56429123878479, 0.0642118752002716, -2.7616469860076904, -1.2481341361999512, 6.233717441558838, 1.5676039457321167, 0.5815752744674683, 1.3840488195419312, -0.23235923051834106, 3.366271495819092, 3.517160415649414, -1.4788669347763062, 0.42035356163978577, 9.654430389404297, 1.686396598815918, -0.743413507938385, 8.198766708374023, 0.03773021697998047, -4.38920783996582, 2.6983234882354736, -3.599304676055908, -1.9888900518417358, 1.305301547050476, -0.5070676803588867, -3.0539464950561523, -2.203826427459717, -2.3735039234161377, 4.664745807647705, -2.884000778198242, -2.2888762950897217, 8.153417587280273, -2.215057373046875, 5.95767068862915, 5.2629289627075195, -0.611006498336792, -5.677565574645996, 1.812056541442871, 2.6907005310058594, 3.871217966079712, 0.28757694363594055, 4.818230152130127, 5.561434268951416, -4.217528343200684, 5.717953205108643, 0.6981694102287292, -2.2629761695861816, -1.2745108604431152, 7.510288715362549, 3.936119794845581, -7.15618371963501, 0.3338300287723541, -0.5593997836112976, -0.702349066734314, -2.0526208877563477, -5.8252058029174805, 3.446577310562134, -3.758759021759033, -1.4801644086837769, -2.409541130065918, -2.9112954139709473, 2.9729466438293457, 1.5350966453552246, -2.6966464519500732, -3.779242753982544, -1.8776576519012451, 3.8680968284606934, 0.503685474395752, 0.9323347806930542, -1.3126546144485474, -1.0315216779708862, 0.8661749362945557, -7.373700141906738, 6.448232650756836, -3.150574207305908, -2.2732574939727783, 5.353988170623779, 0.8897349238395691, 0.1683342158794403, -2.3332772254943848, -6.244677543640137, 0.7397814393043518, 4.781045436859131, -3.598620891571045, 5.451019287109375, -4.508568286895752, -2.936044454574585, 2.023566246032715, 0.0002609133953228593, 2.763551950454712, -2.966952323913574, -4.127046585083008, -7.468250751495361, -4.814845085144043, -4.304430961608887, -5.707862377166748, 1.2338447570800781, -2.655540704727173, -7.953180313110352, 9.933930397033691, -3.339414596557617, -5.022037506103516, 4.959153175354004, 4.661735534667969, -4.605363845825195, -5.895927429199219, -4.000267028808594, 4.517943382263184, -1.2450430393218994, -3.2347915172576904, -0.11348965018987656, 4.816765785217285, -5.558201313018799, -1.5352071523666382, -5.533902645111084, 8.688621520996094, 7.27327823638916, 3.8046483993530273, -0.5336063504219055, 2.6365065574645996, -8.40805435180664, 1.105707049369812, -5.344660758972168, -5.299091339111328, -2.818903684616089, -1.400866985321045, 2.296876907348633, 2.8027150630950928, 1.0506902933120728, -3.77734637260437, 2.703947067260742, -0.4837879240512848, 4.7209153175354, -3.668522596359253, -11.052900314331055, -1.9068368673324585, 8.605053901672363, 1.8759493827819824, -0.777316689491272, 3.5345451831817627, 2.0127298831939697, 1.8453651666641235, -2.054182529449463, 0.3046841621398926, 0.8047680258750916, -2.5706470012664795, -4.474920749664307, 0.5560906529426575, 3.87380051612854, 2.589217185974121, 1.9759103059768677, 2.0172598361968994, 10.336010932922363, 1.0395349264144897, 4.168152332305908, 7.828400135040283, 0.2122524231672287, 1.0581202507019043, 4.1761860847473145, 6.750653266906738, 1.252421259880066, 1.4520713090896606, 0.3633583188056946, -4.822396755218506, 4.470671653747559, -1.0099949836730957, -6.8750104904174805, -0.576546847820282, -6.9081196784973145, -4.018784046173096, 0.08433593809604645, -3.667703151702881, 4.301290988922119, -5.856557369232178, 1.0765315294265747, 2.2570724487304688, 5.869886875152588, 3.7198870182037354, 0.9648178219795227, 0.3546941876411438, -0.20012779533863068, 4.589901447296143, -2.591193914413452, 2.1209542751312256, -0.3365079164505005, 1.0015941858291626, -1.0095455646514893, 8.581499099731445, -2.58693265914917, 1.0552339553833008, 1.1475512981414795, 2.476947069168091, 1.8470127582550049, 1.7441374063491821, 3.973992347717285, -5.973062038421631, 0.08693695813417435, -4.886726379394531, 0.9783831238746643, -4.68242883682251, -3.7912709712982178, -0.19298498332500458, 5.635383129119873, 2.3599367141723633, 1.2372868061065674, 5.664585590362549, -3.7290430068969727, -8.316091537475586, -1.6428017616271973, -1.0598392486572266, 4.992150783538818, 8.610191345214844, -5.265335559844971, 9.96362590789795, 6.312252998352051, 3.4091062545776367, 1.5272321701049805, 9.759543418884277, 0.37749966979026794, -1.8394118547439575, 0.6119140386581421, -5.421327114105225, 1.9439361095428467, -0.3832545578479767, 1.3783963918685913, 1.4969409704208374, -4.179163455963135, 5.321286678314209, -4.002683639526367, 3.609449625015259, -6.31630277633667, 1.5435091257095337, -3.222256660461426, 0.2289784997701645, -2.5468573570251465, -8.673900604248047, -4.399075031280518, 0.3349153399467468, 2.556579351425171, -5.815744876861572, -8.4393892288208, -3.427396297454834, -0.7841569781303406, -2.4712698459625244, -3.1769821643829346, 3.4641096591949463, 3.925265073776245, 3.0287742614746094, -3.690178155899048, 0.36675217747688293, -0.7129285931587219, -5.648152828216553, 3.5916030406951904, 0.4134490489959717, -1.8780573606491089, 5.731741905212402, 3.116827964782715, 3.4855167865753174, 4.861575603485107, 2.9326610565185547, -5.000097274780273, -1.494598388671875, 4.073136806488037, -3.0155320167541504, 0.20745974779129028, 1.649872064590454, 5.824659824371338, 5.253272533416748, 3.6513147354125977, -0.6886249780654907, 3.7160277366638184, 2.1682963371276855, -0.5919160842895508, 5.485564231872559, 2.0324130058288574, -2.9706313610076904, 5.117656230926514, -0.9685616493225098, 2.425847291946411, -6.047229290008545, -0.028351955115795135, -3.7367637157440186, -3.844102621078491, -7.596767902374268, 8.617923736572266, -0.556251049041748, 0.995521605014801, 4.4739670753479, -0.4460131525993347, 2.094522476196289, -5.396482944488525, 0.8246738314628601, -1.617155909538269, 2.040158987045288, 0.7612664103507996, -5.505842685699463, -7.371341705322266, -4.02432107925415, 2.983631134033203, 1.7346394062042236, 1.6462081670761108, -8.044708251953125, -10.72612190246582, -2.46514630317688, -1.3022998571395874, -2.386636734008789, 1.4440975189208984, -0.6602855324745178, -4.72097110748291, -3.498189926147461, 0.5924770832061768, 0.30958500504493713, -0.279262900352478, 9.086281776428223, -1.0351203680038452, 0.4951372444629669, -3.970677614212036, -1.4241161346435547, 2.107797861099243, 2.2779922485351562, -0.6213551163673401, 1.5869508981704712, 0.22632938623428345, -7.520782947540283, -13.097900390625, 9.244686126708984, 3.3236663341522217, -0.22705471515655518, -6.952153205871582, -5.101907730102539, 5.423641681671143, -11.1527099609375, -1.2973246574401855, 3.474257469177246, 0.5729245543479919, 3.0191712379455566, -0.524997353553772, 9.573806762695312, -4.111983776092529, -5.44890832901001, 1.098129391670227, 5.71832275390625, -0.22356753051280975, 4.196019649505615, -2.123349189758301, -4.741097927093506, 5.599682331085205, 6.274852275848389, 6.132515907287598, -1.7530101537704468, -4.620829105377197, -4.677491664886475, -0.8985407948493958, 2.3297512531280518, 2.036686897277832, -5.7376227378845215, 0.7806912660598755, 0.07092096656560898, 0.3605109751224518, -0.09478902816772461, -0.8076809644699097, 2.373100519180298, 0.5074382424354553, 0.9016432762145996, -2.7630233764648438, 1.5831111669540405, -0.48460808396339417, -8.460644721984863, 1.9344403743743896, -1.0572062730789185, 3.705127477645874, -0.6483950614929199, -0.38297009468078613, 0.8467597365379333, -0.04742563143372536, -1.7877192497253418, -7.172801971435547, 7.370753288269043, -6.315145015716553, -1.366087794303894, 2.612553119659424, 1.5200680494308472, 2.269073963165283, 3.076759099960327, 5.05349063873291, 1.1891334056854248, 11.626570701599121, -4.053647041320801, 0.9095156192779541, -1.4434117078781128, -2.180549144744873, 1.1490607261657715, 5.832962512969971, -2.840834379196167, -2.843822956085205, 2.8979554176330566, 2.175976276397705, 7.831027507781982, 4.111519813537598, 0.9183651804924011, -1.317492127418518, 0.8610512018203735, -0.7008680701255798, -5.460363864898682, 0.032939061522483826, -0.7004887461662292, -0.9301631450653076, -1.7172350883483887, 6.76027774810791, 2.785064697265625, 3.747451066970825, -2.430684804916382, -4.0904860496521, -1.1315345764160156, 0.9762871265411377, -1.3813157081604004, -0.7529118061065674, 3.8456780910491943, 1.1968964338302612, -3.9360597133636475, -1.711997151374817, -2.630159854888916, 0.7237535715103149, -4.669006824493408, -0.714989423751831, 2.770566940307617, 0.11489585041999817, -3.655505895614624, -2.465656280517578, 0.49740397930145264, 1.9323261976242065, 3.3482813835144043, -0.5808457136154175, -5.268999099731445, 5.699970245361328, 1.1915452480316162, 1.624287486076355, 1.2220665216445923, -4.149961471557617, -1.2772181034088135, 1.02903413772583, 4.130512237548828, 0.24072739481925964, -12.479793548583984, -0.9654051065444946, 1.410616397857666, 5.175556659698486, 2.8452649116516113, 9.960835456848145, 5.69736909866333, 5.390655040740967, 3.7711493968963623, -5.529741287231445, 3.388201951980591, 2.3979365825653076, -2.1040821075439453, 2.346583843231201, 1.672264575958252, -1.8600783348083496, -0.4935651421546936, -1.0531792640686035, 0.9219163656234741, -1.6983015537261963, 2.179652690887451, -7.670351505279541, -1.1277128458023071, -6.7387495040893555, -1.9315975904464722, 2.067746162414551, 0.08644825220108032, -0.44254088401794434, 0.2751694917678833, 6.278003692626953, -4.4250078201293945, 3.5509326457977295, -6.050387382507324, -1.8697638511657715, 0.6543898582458496, -3.4067211151123047, 0.820549488067627, 2.9147696495056152, -4.774970531463623, -0.3094874918460846, -1.5062649250030518, -7.761535167694092, -0.7482966184616089, -3.191082715988159, 5.356339454650879, -3.502250909805298, 6.79788875579834, -2.133787155151367, 7.7525763511657715, 2.1575558185577393, 1.8153865337371826, 3.222856044769287, 3.6092307567596436, 5.613165855407715, 6.930832862854004, -1.386854887008667, -5.000539302825928, -10.68702507019043, -1.5490121841430664, 3.0409932136535645, 1.5134490728378296, 0.4607439339160919, -1.4007859230041504, -2.5599701404571533, -1.134697437286377, 1.5559040307998657, -7.05842924118042, 2.915687322616577, -5.833834171295166, -0.6330016851425171, -2.5380733013153076, -1.478119969367981, -4.191036701202393, -4.622884750366211, -0.9334955811500549, -4.352129936218262, 9.821073532104492, -1.6841233968734741, -2.646477699279785, -2.2218005657196045, -4.3110456466674805, -9.851105690002441, -1.0500710010528564, 1.2147127389907837, -6.042954921722412, -0.006033717188984156, -4.867465972900391, -0.17182894051074982, 1.4076628684997559, 1.912111759185791, 1.7104299068450928, -2.8475334644317627, 3.4892001152038574, -1.8239850997924805, 1.367774248123169, 2.7858810424804688, -1.249546766281128, -5.171266555786133, 2.420362710952759, -4.502793312072754, -3.5985946655273438, -4.445046901702881, 3.3355512619018555, -0.6880838871002197, -9.030155181884766, 2.903798818588257, 8.247572898864746, 7.163975238800049, -2.3084611892700195, 0.2721545994281769, 1.997552514076233, 3.5355007648468018, 0.10046236962080002, -4.417336463928223, 2.979029417037964, 2.4204108715057373, -5.363296985626221, -1.5103843212127686, 1.033699870109558, -0.05414815992116928, 2.0539989471435547, 3.1382668018341064, 2.122288703918457, -0.5045762658119202, -5.441715240478516, 2.4168195724487305, 5.558329105377197, 7.85563325881958, -0.18098513782024384, -0.1912354826927185, 5.73712158203125, 5.706171035766602, 1.3123699426651, 2.8000104427337646, -3.0973734855651855, 2.2839949131011963, -3.357879161834717, -0.6345653533935547, -0.5468657612800598, -1.9300804138183594, -8.502723693847656, -3.360996723175049, 1.2768899202346802, -0.2997593879699707, -1.6311991214752197, 1.658316731452942, 2.6983461380004883, 3.33351731300354, -3.287545919418335, 0.024936247617006302, -1.3989349603652954, -3.5768821239471436, -6.051164627075195, -0.4584156572818756, -3.55754017829895, 8.5476655960083, -0.46093297004699707, -0.9645275473594666, -1.657257318496704, 7.4356160163879395, -3.8341140747070312, -0.7107216715812683, -3.1453843116760254, -0.5654481649398804, -0.2099931538105011, 4.328803539276123, -5.731781482696533, -0.15053129196166992, 6.398736476898193, -2.106313467025757, -0.7770920991897583, -3.615110397338867, -10.540422439575195, -0.4857926368713379, 4.635277271270752, -1.9386721849441528, -7.453760147094727, 0.9981602430343628, 0.24851426482200623, 3.934083938598633, 3.5743770599365234, -1.148457646369934, 1.5317462682724, -7.265230178833008, 4.387449741363525, 0.6758832931518555, 4.27314567565918, 4.183072090148926, -1.2173820734024048, -0.6657364964485168, -3.5670828819274902, -8.333417892456055, -0.5958877205848694, 0.5465805530548096, -4.411227226257324, 2.7792258262634277, 0.32587921619415283, 1.7464687824249268, -3.14654541015625, -7.126054763793945, 3.0436813831329346, 2.376020908355713, -0.5072278380393982, -1.7524257898330688, -1.1760764122009277, -0.6157529950141907, -2.1032917499542236, -8.111851692199707, 3.069072723388672, -6.047026634216309, -1.3992259502410889, -1.2944445610046387, 0.9775394201278687, -6.520291805267334, 4.7632365226745605, 2.559400796890259, 3.018815517425537, 5.010692596435547, -3.2490437030792236, -12.594436645507812, 1.281026840209961, 5.203204154968262, 4.6064229011535645, -2.1819252967834473, -0.40291744470596313, 1.1091850996017456, -7.261781215667725, -2.6977994441986084, -2.5729827880859375, 6.308798789978027, -2.6287431716918945, 6.263002872467041, 0.03624657168984413, -6.7219767570495605, 8.85789966583252, -3.897620439529419, 2.4669361114501953, 6.822348117828369, -2.140587568283081, -4.922301769256592, 4.16368293762207, 0.2616692781448364, -4.90933084487915, -4.3248066902160645, -1.558645486831665, 3.856078863143921, -1.148232102394104, 2.6180803775787354, 6.610855579376221, 5.077700138092041, 5.750314712524414, 4.9276227951049805, -6.434108257293701, 4.8917741775512695, 1.314162254333496, -0.49759146571159363, -4.338596343994141, -6.3560309410095215, -4.444399356842041, -0.08545597642660141, -4.700351715087891, -3.265899181365967, -2.466834306716919, 2.666154623031616, -1.180719256401062, -0.5835770964622498, 5.304919242858887, -1.2849719524383545, 2.3969690799713135, -4.229190826416016, 1.3165854215621948, 0.4780591130256653, 0.7013695240020752, -1.8655014038085938, -4.122393608093262, -1.1981168985366821, -1.8718963861465454, 4.1335015296936035, 3.6313376426696777, -4.80848503112793, 2.5092923641204834, -1.9189262390136719, 1.8356813192367554, -1.9262299537658691, -5.887070178985596, 4.065070152282715, 1.809181571006775, -4.9027485847473145, -7.565648555755615, -2.1368954181671143, -4.72737455368042, -1.4556585550308228, 2.481106758117676, 5.876339435577393, -1.42720627784729, 5.5094122886657715, 4.657377243041992, -1.7376788854599, -6.668092727661133, 3.414362907409668, 2.441629409790039, 9.419685363769531, 5.7271013259887695, 3.1374268531799316, 6.762890338897705, -4.181085109710693, 0.7879970073699951, -1.0782111883163452, 6.986756801605225, 2.2792556285858154, -12.217638969421387, 5.5050201416015625, -3.6012020111083984, 3.1393396854400635, -1.881577491760254, 1.0168815851211548, 2.583477020263672, 0.8215916156768799, -3.7680013179779053, 5.647916316986084, 1.64167320728302, -0.011298677884042263, 2.8826444149017334, -2.0813844203948975, 7.606346607208252, 0.9640344381332397, 1.528629183769226, 1.4564237594604492, -3.581665277481079, -4.626989364624023, 0.9405511617660522, 1.4826937913894653, 5.45833158493042, 1.8802393674850464, -4.28122615814209, 2.1817305088043213, 3.3865506649017334, 4.089512825012207, 1.4120514392852783, -4.809140682220459, 4.722811222076416, -2.8505806922912598, 2.3722116947174072, -6.21502685546875, 3.1357181072235107, -3.063333749771118, 4.4062299728393555, 7.132449626922607, 4.753787040710449, -4.597642421722412, -3.703286647796631, -3.41042423248291, 2.721752166748047, 1.4528330564498901, 0.7035333514213562, 1.306452751159668, 2.3871994018554688, 2.672821521759033, 6.8249311447143555, -0.15982404351234436, 0.8767844438552856, -7.363534927368164, -5.237381458282471, -2.718435525894165, -4.692536354064941, 0.9326415657997131, 4.211826801300049, -1.6985324621200562, -1.9017302989959717, -2.0098257064819336, 0.5576077699661255, -2.148176908493042, 1.735764980316162, 1.2302265167236328, 1.4014525413513184, 3.0776307582855225, 0.14434531331062317, 0.5913007259368896, 7.052011489868164, -0.17139863967895508, 3.2014217376708984, 0.222809299826622, -1.8287384510040283, -1.8272348642349243, -3.281602144241333, 0.04540273919701576, -2.419222116470337, -1.6961969137191772, 4.873420238494873, -0.5387120246887207, 4.555781364440918, 1.0567065477371216, 3.441859006881714, -2.4501941204071045, 0.5813986659049988, 10.533742904663086, 3.2997689247131348, 4.00233268737793, 0.920248806476593, 3.4154961109161377, -4.664831161499023, -3.42182993888855, 1.6958627700805664, 3.259378433227539, -1.6630280017852783, -0.7746884226799011, -1.4737805128097534, -0.7802200317382812, -4.833414077758789, -5.765635967254639, -0.3644380569458008, -6.991142749786377, -3.957625389099121, -2.6876206398010254, -4.261819362640381, 0.8983714580535889, -1.843137264251709, -0.38696807622909546, -4.109273433685303, -3.041457176208496, 1.971644639968872, 1.3704352378845215, -1.9467462301254272, 1.3650398254394531, 0.6563611626625061, 0.5397096872329712, 6.277432918548584, -1.708325982093811, 1.5693347454071045, 1.5744527578353882, -1.050789475440979, -4.777047157287598, 2.6424150466918945, 0.9225688576698303, 0.06958657503128052, -0.04009176790714264, -0.052412860095500946, -3.718278169631958, 1.0214694738388062, -7.3640899658203125, -0.2618541121482849, -6.147766590118408, -1.5761845111846924, 2.6883864402770996, -1.214613437652588, -2.1006479263305664, 1.8679285049438477, -1.2677483558654785, -1.8259330987930298, -1.7709425687789917, 5.846439838409424, 2.452836275100708, 1.7159087657928467, -4.971155643463135, 2.2625670433044434, -5.389217376708984, 1.6967616081237793, -4.687288761138916, 7.5824151039123535, -6.965123176574707, 4.654284954071045, 2.969144821166992, -7.337677001953125, -1.201285481452942, 5.536880970001221, 4.302427768707275, -6.875899791717529, -2.323803663253784, -4.939262866973877, -4.344668865203857, -2.8158531188964844, 7.115502834320068, 0.2827501595020294, -1.1550514698028564, -1.5915900468826294, 2.410430431365967, 2.5092930793762207, -3.9316048622131348, 5.61279821395874, -4.5873260498046875, -1.3654099702835083, 1.544189691543579, 6.583793640136719, -1.6432092189788818, 0.8844926953315735, 4.950002193450928, 2.2332630157470703, 4.248351097106934, 3.892624855041504, 0.7793152332305908, 2.2627716064453125, 3.405918598175049, -0.37139594554901123, 6.1322550773620605, -8.203970909118652, -3.9424800872802734, 5.5147175788879395, 1.4479585886001587, -2.8354368209838867, 1.252922534942627, 4.087436676025391, -2.494023084640503, 2.8133742809295654, -0.7461297512054443, -0.7108235955238342, -0.5336623787879944, 4.189306259155273, -4.734073162078857, -1.7254129648208618, 2.3944785594940186, -5.130612850189209, -6.577666282653809, 2.021164655685425, -0.24804821610450745, -0.8425900936126709, 0.3154715895652771, -4.141961574554443, 11.396700859069824, -4.189574241638184, -1.9945423603057861, -5.9393229484558105, -1.9166698455810547, 5.851384162902832, -0.17765721678733826, 2.125821590423584, -2.1869330406188965, 6.998777389526367, 0.4021587073802948, -5.246488094329834, 2.8265645503997803, -1.731758952140808, 2.1744532585144043, 0.30325084924697876, 0.8004316091537476, 0.2928050458431244, -3.875452756881714, -6.262345790863037, 3.6489858627319336, -0.7964774370193481, 0.887712299823761, -2.1104443073272705, -8.347955703735352, -5.363210678100586, 5.3252058029174805, -3.397655725479126, -1.7877744436264038, -6.1267828941345215, -2.3692946434020996, -1.749894380569458, -1.1472039222717285, -0.5567883253097534, -2.7890660762786865, -2.5617544651031494, -0.3205512762069702, 1.0244699716567993, -0.8359032869338989, 6.701812744140625, 0.5521508455276489, 0.37419217824935913, -2.8953590393066406, 3.8356237411499023, -4.367999076843262, -6.747224807739258, -4.176980018615723, 2.1990342140197754, -4.855961322784424, 4.896243095397949, -0.3706822991371155, -6.34525203704834, -1.1978594064712524, 6.851430416107178, -3.72308611869812, -5.522660732269287, 1.765303373336792, 4.940267562866211, 3.779994487762451, -1.599710464477539, 0.46452775597572327, 1.0139915943145752, 4.409934997558594, 8.308342933654785, -0.7173579335212708, 4.910728931427002, -1.6622509956359863, 2.069596529006958, -4.019207954406738, -3.342538595199585, 1.3743376731872559, -3.8735084533691406, 3.261626720428467, -2.4890401363372803, 0.023355567827820778, -0.6206700205802917, -1.6201398372650146, 0.23727358877658844, 4.030416965484619, 12.134342193603516, -5.033897399902344, 0.9140440821647644, -0.9885891079902649, -3.9395298957824707, 0.8589469790458679, -3.801100015640259, -0.5989704132080078, 4.672580718994141, -8.370100021362305, 0.7042850852012634, 1.5006349086761475, -4.115747451782227, 1.8092460632324219, 1.1199707984924316, -1.6820242404937744, 4.561850070953369, -5.515651702880859, 2.2861697673797607, -1.2205194234848022, -4.511811256408691, -0.9965577125549316, -1.6834464073181152, 2.988839864730835, -6.2673234939575195, -1.0651310682296753, -2.109323024749756, 6.004426002502441, -4.75998592376709, 3.9460651874542236, -7.3157196044921875, 1.0832659006118774, -0.839836061000824, 1.2126597166061401, -3.113191843032837, -8.10822582244873, -6.488810062408447, 4.937518119812012, 0.5366576313972473, -3.7764434814453125, 3.276136636734009, -3.306797504425049, -3.2851054668426514, -5.184297561645508, 4.374414443969727, 7.511709213256836, -4.196939945220947, 0.8127490282058716, 3.563354015350342, 8.111174583435059, -9.945342063903809, 0.20795276761054993, 2.221837043762207, -6.280930042266846, 1.2666411399841309, 0.42184242606163025, 2.523804187774658, -2.7016754150390625, 1.2975234985351562, -2.7958924770355225, 2.0477027893066406, 0.2556963562965393, -1.8857390880584717, -6.041466236114502, 6.211853504180908, 4.055456161499023, 3.1383838653564453, -3.0390682220458984, -1.7830145359039307, 6.299149990081787, 2.5966928005218506, -3.7873330116271973, -0.19344712793827057, 0.18826673924922943, -1.174977421760559, -4.148453235626221, 4.647333145141602, 1.9879631996154785, -1.1554464101791382, -6.656325340270996, -2.6953489780426025, -0.3755629360675812, 3.117276668548584, -1.8298804759979248, 4.086354732513428, 0.21773748099803925, -3.3180220127105713, -11.507765769958496, -5.0314106941223145, -6.293139934539795, -0.9333055019378662, 1.0065714120864868, 5.240291595458984, -0.3672407865524292, -4.94704532623291, 9.099154472351074, 0.9378865361213684, -1.3593133687973022, -1.3522189855575562, -5.708490371704102, 2.246032953262329, 1.200125813484192, 6.063174724578857, 4.119967937469482, 3.1887025833129883, 4.751814365386963, -1.415253758430481, -2.0495758056640625, -0.46969473361968994, -5.25170373916626, -0.8359725475311279, -1.5777264833450317, 2.419858694076538, -0.31004372239112854, -7.881460666656494, -0.12213484942913055, 5.424449920654297, -7.578470230102539, -3.2707550525665283, -0.16192762553691864, -5.5316972732543945, -2.420431613922119, 0.0039199101738631725, -5.49420166015625, 0.9638825058937073, 1.427647352218628, -5.279074668884277, -0.09439129382371902, 4.120792388916016, -0.8981603980064392, 1.5515559911727905, 2.385336399078369, -1.1235053539276123, 3.7502639293670654, 1.3628191947937012, -4.137558460235596, 1.9201152324676514, 3.294705390930176, 3.0588269233703613, -2.7322580814361572, 1.686585783958435, 5.342585563659668, 0.7794777750968933, 4.598971366882324, 5.068200588226318, -0.2760784327983856, 2.5159401893615723, -1.2847884893417358, 4.153433799743652, -0.47898051142692566, -2.3125932216644287, -0.21279510855674744, -0.3980550169944763, 1.4232516288757324, 1.758568286895752, -4.759479999542236, -2.7038159370422363, 1.3443351984024048, 1.8320481777191162, 0.2544584572315216, 10.450751304626465, 4.397237777709961, -0.37736776471138, 2.84720778465271, 5.291072368621826, 1.4065933227539062, -7.0318498611450195, 0.3454133868217468, 0.16512686014175415, 8.499061584472656, -7.377424240112305, 6.164576530456543, -0.6396598815917969, 3.7154414653778076, -1.172896146774292, 0.70229572057724, 4.53221321105957, 0.347606360912323, 4.231559753417969, -0.30492791533470154, -1.6171700954437256, -0.8498160243034363, -1.983339786529541, -0.448931485414505, 5.483712196350098, -4.642163276672363, -6.600665092468262, -0.029350632801651955, 6.7864909172058105, 4.238287448883057, 5.345546722412109, 5.060308933258057, -2.8551578521728516, 0.5392130613327026, -10.574812889099121, -2.3662710189819336, 2.504194974899292, -6.150301456451416, -8.162566184997559, -5.112276077270508, 0.9639596343040466, 4.765350818634033, 10.449361801147461, 5.832085132598877, -5.947888374328613, -9.159987449645996, 0.72904372215271, 4.37490701675415, -6.328217506408691, -1.9377310276031494, -1.6385855674743652, 0.3063482940196991, -0.540037214756012, 2.2535812854766846, 4.4276275634765625, 0.49229899048805237, 3.2146341800689697, 8.193401336669922, -3.8606748580932617, -3.957108497619629, 2.2777442932128906, 7.6571736335754395, 0.9225665330886841, 4.633284091949463, -0.9649471044540405, 3.938319683074951, -2.2255818843841553, 2.9866068363189697, 1.8771746158599854, 1.8817782402038574, 5.818521499633789, 4.5918121337890625, -2.065809965133667, 3.8797895908355713, -6.448828220367432, 4.313638687133789, 11.365833282470703, 2.4993560314178467, 2.4186952114105225, 0.6510116457939148, 3.7365007400512695, -1.38540780544281, 3.1145448684692383, -0.645837664604187, -1.2466789484024048, 0.9561423659324646, 0.22775518894195557, -6.468266487121582, 8.786967277526855, 1.0461616516113281, -6.292773723602295, 4.8037943840026855, 0.38154613971710205, 1.3150972127914429, 3.32668137550354, -3.0812532901763916, -11.275229454040527, 0.10088135302066803, 1.469252347946167, 4.511943817138672, -2.526411771774292, -0.5656889081001282, 2.0986227989196777, -0.7639214992523193, -0.49640774726867676, 1.520967960357666, -3.4828264713287354, 6.837366104125977, -3.1889657974243164, 0.6439071893692017, -0.8195867538452148, -2.3345627784729004, 2.636820077896118, 4.53945779800415, 6.317218780517578, 2.820500135421753, -1.3021152019500732, 3.9403204917907715, -5.2903666496276855, 0.681316614151001, 2.572206974029541, 2.735581398010254, -4.1341681480407715, -5.524392604827881, 7.106631278991699, -1.5924049615859985, -6.141786098480225, -4.517266273498535, 0.48749464750289917, 5.14687442779541, 0.4088050127029419, 5.214889049530029, 1.3114566802978516, 2.2866764068603516, 4.3308210372924805, -1.117096185684204, 1.5656863451004028, -1.1761794090270996, -2.1785266399383545, 2.9895503520965576, -2.9259276390075684, 0.6735256910324097, -2.8167731761932373, -0.014917867258191109, -1.7742912769317627, 4.705543518066406, 0.6386579871177673, 0.6216616034507751, 6.897258758544922, -11.28841495513916, -4.189929485321045, -10.70259952545166, 0.21732375025749207, 1.9884083271026611, 2.8004226684570312, 1.2901909351348877, -3.2800068855285645, 7.211940765380859, 1.4123902320861816, -0.8587653636932373, -4.307513236999512, 1.8518459796905518, -1.8019574880599976, -1.6433943510055542, 0.9776052236557007], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [0.07305484265089035, 5.415465831756592, 4.86263370513916, -0.2346222698688507, -0.27761605381965637, -0.2281956523656845, 2.0672290325164795, -0.0996084213256836, 2.8441646099090576, -0.004562501795589924, 6.254315376281738, -0.1561221033334732, 2.553553342819214, -0.2003326117992401, -0.2729405164718628, 1.6012744903564453, 3.18051815032959, -0.24940745532512665, -0.21499955654144287, 2.576869487762451, -0.10390891879796982, -0.014142528176307678, 1.3100666999816895, -0.17174021899700165, -0.0035421319771558046, -0.12081928551197052, -0.2496906965970993, -0.040023207664489746, -0.015211798250675201, 0.07863538712263107, 3.4925127029418945, -0.2633572220802307, -0.14454501867294312, 1.2681097984313965, 2.7148513793945312, -0.2777710258960724, 1.9571388959884644, 4.108118057250977, -0.25174593925476074, -0.12098991870880127, 1.1134392023086548, 0.7219662070274353, 4.341692924499512, -0.02997334487736225, -0.17521639168262482, -0.11334669589996338, -0.00673594418913126, 0.3154829740524292, 1.8831813335418701, -0.1462603062391281, 5.520966053009033, 3.1693546772003174, 2.320805072784424, 1.3384406566619873, 3.74086856842041, -0.02573763206601143, 2.261664390563965, 2.212601661682129, -0.006001539994031191, 1.1109017133712769, -0.269062876701355, 2.9946579933166504, -0.2586265504360199, -0.035151392221450806, 1.0938434600830078, -0.10223861783742905, 2.194211721420288, -0.16649344563484192, 1.7288181781768799, -0.2303023785352707, 4.289158821105957, 2.2232706546783447, -0.23437704145908356, 2.3409457206726074, 5.006577491760254, 7.159164905548096, -0.01160284411162138, 3.4322006702423096, 1.6097488403320312, -0.1528903692960739, -0.07055241614580154, 4.996045112609863, -0.03223909065127373, 2.0550036430358887, 5.179113388061523, -0.010628157295286655, -0.27793940901756287, -0.02769656851887703, 0.7090815305709839, 6.043357849121094, 8.091753959655762, -0.005791149567812681, -0.20960308611392975, 2.6550028324127197, 6.3061113357543945, 1.421239972114563, -0.1436375379562378, 0.9109782576560974, -0.03213401883840561, -0.25005826354026794, -0.14781160652637482, 1.2776083946228027, 0.9532990455627441, -0.15177731215953827, -0.2587369680404663, -0.02994118072092533, -0.24334478378295898, 4.787403106689453, -0.24251329898834229, -0.01811143383383751, 5.3122382164001465, 1.893966794013977, 8.430513381958008, 0.14057448506355286, -0.25172701478004456, -0.017643319442868233, 5.268930435180664, 10.277692794799805, -0.11865216493606567, -0.180904820561409, 2.1836185455322266, -0.11573323607444763, -0.2738955318927765, -0.27695831656455994, 0.5244719982147217, 4.325061321258545, 0.04608183354139328, 3.584825277328491, -0.2639494240283966, 1.0328446626663208, -0.001650256454013288, -0.017509812489151955, 3.056990146636963, -0.20354747772216797, 2.112032175064087, 1.9469413757324219, 5.262559413909912, 3.125920057296753, 3.9995250701904297, -0.17427389323711395, -0.16115295886993408, -0.1488628387451172, -0.24268728494644165, -0.04871377721428871, -0.22045013308525085, -0.15124832093715668, 0.7370463013648987, -0.27783939242362976, -0.0035011963918805122, -0.12381099164485931, 0.6410940289497375, 4.281963348388672, 2.1030116081237793, -0.18229402601718903, -0.1516920030117035, 8.03980541229248, 2.8866019248962402, 7.320939064025879, 8.51607608795166, -0.11954677850008011, 1.2692639827728271, 1.5799051523208618, 2.022773265838623, 1.9450017213821411, 0.5590687990188599, 0.46486595273017883, -0.014712966047227383, -0.22519652545452118, -0.14311793446540833, 0.31445208191871643, -0.2497076690196991, 2.2818586826324463, 3.1570677757263184, 0.9565169811248779, 0.986228883266449, 0.22005708515644073, -0.2560091018676758, -0.22450335323810577, -0.23997919261455536, 1.2806601524353027, 0.029023366048932076, 2.1180851459503174, -0.026015987619757652, 3.4520668983459473, -0.13784542679786682, -0.1478796750307083, -0.24658073484897614, -0.2141868621110916, 0.8875237703323364, -0.20991472899913788, -0.058529648929834366, -0.1110515221953392, 5.220482349395752, 0.09633145481348038, -0.04335801303386688, 0.017035504803061485, -0.01230945996940136, 7.152698040008545, -0.013674654066562653, 1.6690870523452759, 1.95256507396698, -0.017506364732980728, 0.9233160614967346, 3.9161453247070312, 1.620603084564209, 0.7722256779670715, 3.7130227088928223, 0.42193734645843506, 1.0677140951156616, -0.19054028391838074, 2.3402798175811768, -0.08521845191717148, 3.0390260219573975, -0.23646967113018036, -0.05490192398428917, 1.185868263244629, 6.764072895050049, 3.766303062438965, -0.23263122141361237, 2.6310009956359863, -0.25401899218559265, -0.1877470314502716, 0.210780069231987, -0.0008741088095121086, 0.5233032703399658, -0.06593392789363861, -0.16148702800273895, -0.01615113392472267, -0.014040459878742695, -0.08476322144269943, 3.1851894855499268, 6.655674457550049, -0.04787116497755051, 4.312450885772705, -0.2725224494934082, -0.00364667852409184, 0.3871326744556427, 1.5259294509887695, -0.26160240173339844, 0.9309760332107544, 2.0796713829040527, -0.10589101165533066, -0.016020966693758965, -0.2585481107234955, -0.253814697265625, -0.276672899723053, 3.87431263923645, 6.895422458648682, -0.26478686928749084, 2.634620428085327, 3.678395986557007, 2.6970269680023193, 4.127426624298096, -0.2710399627685547, 2.813328742980957, 8.095465660095215, 3.090519428253174, -0.19167739152908325, 2.2343151569366455, 5.017347812652588, 1.2013593912124634, -0.21418049931526184, 6.159471035003662, -0.03348900005221367, 1.7765427827835083, -0.03130491450428963, -0.03996524587273598, 4.840940475463867, 1.1144087314605713, -0.04345671832561493, 2.207989454269409, -0.2607702612876892, -0.003764899680390954, -0.16470345854759216, 0.7817797660827637, 2.3574130535125732, 0.436937540769577, 0.09846740961074829, 1.6310656070709229, -0.25377312302589417, 4.665712833404541, 0.3831459581851959, 0.8984256982803345, 1.9401606321334839, -0.06073391065001488, -0.1570570021867752, 6.100827693939209, 1.0631873607635498, -0.275918573141098, 1.0882704257965088, -0.24493323266506195, 0.838956356048584, -0.1735599786043167, 0.8291859030723572, -0.011443184688687325, 5.297852993011475, -0.23363682627677917, -0.011307942681014538, -0.2596595585346222, -0.2742794156074524, 0.394924134016037, -0.04645473137497902, 1.320534348487854, 14.494263648986816, -0.030545201152563095, 1.0729502439498901, -0.2783243954181671, 0.5266793966293335, -0.005157372448593378, -0.04563566669821739, 1.58635413646698, -0.07094384729862213, 6.531526565551758, -0.15870901942253113, 0.46608391404151917, 11.652384757995605, -0.1368570327758789, -0.0947321280837059, -0.06533178687095642, -0.08120299875736237, 3.2651288509368896, 0.3248472213745117, -0.139914408326149, 0.5916940569877625, 10.55411148071289, 5.0558695793151855, -0.00016342345043085515, -0.205460324883461, -0.10256735235452652, 2.2983157634735107, -0.2310323268175125, 7.727390289306641, 3.6372320652008057, 3.4632787704467773, 0.5062557458877563, -0.07411372661590576, 1.0829650163650513, 5.15215539932251, 1.9120477437973022, -0.27683040499687195, 5.069441795349121, 0.5249866843223572, -0.13132542371749878, -0.1281542330980301, -0.21315564215183258, -0.1852625459432602, -0.0001574720663484186, 3.9989147186279297, 2.0529468059539795, -0.25536707043647766, 4.121135711669922, 1.4930105209350586, 1.2120795249938965, -0.0004971267771907151, -0.26331380009651184, 1.9023991823196411, 6.428472518920898, 7.356504440307617, 5.684396743774414, -0.04416215419769287, -0.0013372432440519333, 2.0704965591430664, 0.05827910825610161, 3.770005702972412, -0.18968744575977325, 0.6211321949958801, -0.014836279675364494, 1.5360944271087646, -0.2568902373313904, 1.122997522354126, 1.5852584838867188, -0.18046551942825317, 0.9887538552284241, -0.0264736395329237, 2.897740364074707, 1.769200086593628, -0.22053879499435425, -0.2044442594051361, 6.968440055847168, -0.16246338188648224, 3.893542766571045, 1.3033469915390015, -0.21389919519424438, -0.14263954758644104, 0.2968865931034088, 0.7558571100234985, 2.26357364654541, -0.031710196286439896, -0.20173759758472443, -0.15415985882282257, 6.392408847808838, -0.0007497568149119616, 1.49783456325531, -0.22332720458507538, 1.4646763801574707, -0.24129191040992737, 1.182979702949524, 6.246595859527588, 1.1402775049209595, 1.4733383655548096, 1.2174452543258667, -0.14216716587543488, 1.9498975276947021, -0.25245732069015503, -0.12156249582767487, -0.07116382569074631, -0.2610844373703003, 2.8728647232055664, -0.07835555821657181, -0.2274097502231598, 5.806720733642578, 3.879124402999878, 5.047488689422607, 2.234116792678833, -0.17202025651931763, -0.264156699180603, -0.27543070912361145, 3.7070629596710205, 0.27307605743408203, -0.01570512168109417, -0.07206815481185913, -0.019571632146835327, 0.4379109740257263, -0.038713887333869934, -0.03201432153582573, 0.2923719584941864, -0.1321564018726349, -0.008615683764219284, 2.8499906063079834, 8.650875091552734, -0.015781700611114502, 7.2743821144104, -0.16512805223464966, -0.24053741991519928, -0.23496757447719574, -0.01778154820203781, 5.167900085449219, 1.729737639427185, -0.14176568388938904, -0.2750670611858368, -0.014049758203327656, -0.27715349197387695, 3.658372163772583, -0.007180904038250446, -0.2594475746154785, 0.37738093733787537, 4.130578994750977, 5.372990131378174, -0.25305482745170593, -0.14571769535541534, 1.5463902950286865, -0.27540141344070435, -0.27830129861831665, 4.264064311981201, 6.75111722946167, 0.7905339598655701, -0.24351763725280762, 2.0739803314208984, -0.003969825338572264, -0.035166602581739426, -0.06011643260717392, 9.557036399841309, 2.7277979850769043, 0.36061719059944153, -0.2447967380285263, -0.00043017559801228344, 2.695488929748535, 6.507861137390137, -0.14985524117946625, 2.566816568374634, 2.7380294799804688, -0.012635909020900726, 3.6563644409179688, 5.3952178955078125, 0.6523579955101013, -0.11048058420419693, 3.0225703716278076, -0.012951375916600227, 0.2209683060646057, 4.1561174392700195, -0.03427652642130852, 3.8967604637145996, -0.26438042521476746, -0.0020203369203954935, -0.16742061078548431, -0.07807499170303345, 0.9985743165016174, -0.042741019278764725, 4.139103889465332, -0.14624588191509247, 1.1491647958755493, -0.20201843976974487, -0.2784458100795746, 4.726836204528809, 0.6052711009979248, -0.2698500454425812, -0.033304475247859955, 4.5954155921936035, 2.78283429145813, -0.27838215231895447, 0.7063470482826233, -0.06594013422727585, -0.06887499988079071, -0.11701648682355881, 1.1371172666549683, 0.9059459567070007, -0.14393863081932068, 2.804474353790283, 3.0084023475646973, 0.46085554361343384, 4.089317798614502, -0.2765456438064575, -0.17579998075962067, 1.6435036659240723, -0.06345919519662857, 3.0482728481292725, -0.03318203613162041, 5.6840996742248535, -0.1315169483423233, -0.22465521097183228, -0.022201120853424072, 2.2013638019561768, 5.064387321472168, 4.2259345054626465, 3.2186954021453857, -0.012260213494300842, -0.13121628761291504, -0.25134435296058655, -0.08684444427490234, 4.440989017486572, 0.4992474913597107, -0.046560872346162796, 1.7072765827178955, -0.08276457339525223, -0.09354466944932938, -0.10078869014978409, 1.7512162923812866, 1.0887298583984375, 2.1040167808532715, 4.256553649902344, -0.023711668327450752, -0.21532787382602692, 0.379704087972641, -0.024378981441259384, -0.22636260092258453, 3.987075090408325, 4.357638835906982, 4.24955940246582, 0.09341923892498016, -0.2542591392993927, 4.261315822601318, -0.27402013540267944, 1.075075626373291, 5.274815082550049, -0.2747953236103058, -0.19540151953697205, -0.018391303718090057, 2.1932549476623535, 1.6849759817123413, 0.405852735042572, 2.0087482929229736, -0.010630683973431587, -0.10708856582641602, 0.4293922185897827, -0.08835127204656601, -0.024943459779024124, -0.03823084384202957, -0.09100579470396042, -0.2346857488155365, -0.06731126457452774, 0.6755858659744263, 6.777233123779297, -0.21052628755569458, 5.0678606033325195, 0.8770527839660645, -0.050231847912073135, -0.27069732546806335, -0.07585880160331726, 2.9357001781463623, -0.04534223675727844, 5.263760089874268, 2.1044366359710693, -0.0024812021292746067, 3.006876230239868, 6.199634075164795, -0.1284358948469162, 3.8658998012542725, -0.005401404108852148, 0.739072322845459, -0.008626574650406837, 1.684897541999817, -0.005598463583737612, -0.0002810048754326999, 1.3147486448287964, 9.150588035583496, -0.20134380459785461, 3.20487380027771, 0.2696641683578491, -0.27843838930130005, 2.2680628299713135, 6.560421466827393, -0.03507843613624573, 2.0416035652160645, 1.1031605005264282, -0.019049303606152534, 6.448740482330322, 7.7942328453063965, -0.08249182999134064, 6.9449543952941895, 0.12484905123710632, -0.2235296368598938, 5.999129295349121, 0.6485887169837952, 1.0142611265182495, 6.056711196899414, 5.123253345489502, 3.4750096797943115, -0.02059466578066349, -0.01164912898093462, 4.455296039581299, -0.08883757889270782, 0.3826523721218109, -0.05772506818175316, -0.23817095160484314, -0.0305425263941288, 6.511107921600342, -0.06805370002985, -0.18561196327209473, -0.24105487763881683, 4.785953044891357, -0.019329549744725227, -0.062243878841400146, -0.25290587544441223, 1.89247727394104, 1.6398524045944214, -0.2509583532810211, 0.6926815509796143, -0.2736990451812744, 1.6229420900344849, -0.16365034878253937, -0.17757648229599, -0.05207955837249756, 1.179806113243103, 9.062479972839355, -0.24155792593955994, 0.9484671354293823, 5.8353776931762695, -0.0020393209997564554, -0.006531191524118185, -0.04059576243162155, -0.17716282606124878, 9.759344100952148, 1.2948557138442993, 4.794100761413574, -0.25888919830322266, -0.0005172527744434774, -0.0008853445178829134, 4.006363868713379, 0.5005056858062744, -0.028666451573371887, 4.418664455413818, -0.01302948035299778, 4.488911151885986, 5.787933826446533, -0.13711123168468475, -0.05206166207790375, 3.265815496444702, -0.05413493514060974, 3.3412160873413086, -0.03976898640394211, 3.1074306964874268, 0.22256961464881897, -0.2604377269744873, -0.09690431505441666, 0.6853322386741638, -0.003899605944752693, -0.27488014101982117, -0.26961085200309753, 2.5447044372558594, -0.03306084871292114, 6.846526145935059, -0.2783504128456116, -0.04748031869530678, 5.522723197937012, 6.643143177032471, -0.24078699946403503, -0.09815086424350739, 0.03313637152314186, -0.16413085162639618, -0.2783629894256592, 6.2215094566345215, 1.2971009016036987, 0.37303969264030457, 1.1067415475845337, -0.10274231433868408, 3.25395131111145, 3.4157681465148926, -0.27447566390037537, 0.25371187925338745, 9.65381145477295, 1.4228965044021606, -0.23957106471061707, 8.196512222290039, 0.019220957532525063, -0.05380475893616676, 2.5281338691711426, -0.09579549729824066, -0.23941130936145782, 1.0269144773483276, -0.19059720635414124, -0.13757233321666718, -0.21907664835453033, -0.20225955545902252, 4.621206760406494, -0.1527082771062851, -0.21068744361400604, 8.151071548461914, -0.2179759442806244, 5.942304611206055, 5.235806941986084, -0.214970201253891, -0.01936168782413006, 1.5576623678207397, 2.519775390625, 3.7922146320343018, 0.16432227194309235, 4.779605865478516, 5.540142059326172, -0.06124294549226761, 5.699223041534424, 0.4662248194217682, -0.21325388550758362, -0.27846285700798035, 7.5061798095703125, 3.860743284225464, -0.00557765644043684, 0.1945197433233261, -0.20344609022140503, -0.23268236219882965, -0.23356327414512634, -0.017146531492471695, 3.340177536010742, -0.0856303721666336, -0.27442631125450134, -0.1986619085073471, -0.15021704137325287, 2.8282737731933594, 1.2630019187927246, -0.1703512966632843, -0.08439035713672638, -0.24908660352230072, 3.7889156341552734, 0.3139597773551941, 0.6689956188201904, -0.2783391773700714, -0.27107274532318115, 0.6097431182861328, -0.0046244035474956036, 6.4380388259887695, -0.12938985228538513, -0.21223610639572144, 5.328786849975586, 0.6306756138801575, 0.09123453497886658, -0.20627151429653168, -0.01209592167288065, 0.5007936358451843, 4.741279125213623, -0.09584106504917145, 5.427724361419678, -0.04911736398935318, -0.14797736704349518, 1.787313461303711, 0.0001304737088503316, 2.599601984024048, -0.1452067494392395, -0.06551434099674225, -0.004261386580765247, -0.0387270487844944, -0.057371702045202255, -0.01888604462146759, 0.9556013941764832, -0.17433181405067444, -0.0027948974166065454, 9.933448791503906, -0.11435328423976898, -0.03288392722606659, 4.924587726593018, 4.618093967437744, -0.04558892175555229, -0.016172995790839195, -0.07193077355623245, 4.469178199768066, -0.27834105491638184, -0.12252499908208847, -0.0535283088684082, 4.778097152709961, -0.02134847268462181, -0.27208951115608215, -0.021775882691144943, 8.68715763092041, 7.268235683441162, 3.721775531768799, -0.19726155698299408, 2.460320472717285, -0.0018751112511381507, 0.8307486176490784, -0.02539190463721752, -0.026343343779444695, -0.1587357372045517, -0.2769213914871216, 2.0869836807250977, 2.6424622535705566, 0.7784642577171326, -0.08450453728437424, 2.53429913520813, -0.1844964176416397, 4.679236888885498, -0.09127049893140793, -0.00017508775636088103, -0.24662387371063232, 8.603477478027344, 1.6267204284667969, -0.24477584660053253, 3.4343578815460205, 1.7754838466644287, 1.593624234199524, -0.23341765999794006, 0.17537230253219604, 0.5560894012451172, -0.18264855444431305, -0.0504002645611763, 0.35342204570770264, 3.7949445247650146, 2.408397912979126, 1.7353323698043823, 1.7804282903671265, 10.335676193237305, 0.7679670453071594, 4.104609489440918, 7.825283527374268, 0.11734689027070999, 0.7854756116867065, 4.113022327423096, 6.742763042449951, 0.9740314483642578, 1.1766380071640015, 0.21432803571224213, -0.03849831223487854, 4.420106887817383, -0.26964929699897766, -0.007096524816006422, -0.20739978551864624, -0.006898706778883934, -0.0709613561630249, 0.04394505172967911, -0.09132305532693863, 4.243783473968506, -0.01670825481414795, 0.8029170632362366, 2.0432348251342773, 5.853361129760742, 3.6318626403808594, 0.6986104846000671, 0.20847344398498535, -0.09008439630270004, 4.543764591217041, -0.18062518537044525, 1.89384925365448, -0.14020872116088867, 0.7325378060340881, -0.26961812376976013, 8.579890251159668, -0.1810443103313446, 0.782750129699707, 0.8710641264915466, 2.285007953643799, 1.5954052209854126, 1.4846303462982178, 3.9006667137145996, -0.015171363949775696, 0.04535679519176483, -0.03659947216510773, 0.7110763788223267, -0.04294539615511894, -0.08366908878087997, -0.08721047639846802, 5.615339756011963, 2.1563234329223633, 0.9590103626251221, 5.645016670227051, -0.08745566755533218, -0.002033201279118657, -0.2662721872329712, -0.2727392017841339, 4.958477973937988, 8.608622550964355, -0.0270695723593235, 9.963156700134277, 6.300824165344238, 3.299973487854004, 1.2547755241394043, 9.758980751037598, 0.22395919263362885, -0.2522217333316803, 0.39675116539001465, -0.023863671347498894, 1.7005242109298706, -0.1553492248058319, 1.1009711027145386, 1.2231770753860474, -0.06302350014448166, 5.295410633087158, -0.07180362194776535, 3.5143282413482666, -0.011390521191060543, 1.271811842918396, -0.12353069335222244, 0.12754006683826447, -0.18499834835529327, -0.0014829039573669434, -0.05340266972780228, 0.19524052739143372, 2.372542142868042, -0.017280926927924156, -0.001824050908908248, -0.10779210925102234, -0.24577414989471436, -0.19250445067882538, -0.1272100955247879, 3.358970880508423, 3.849292039871216, 2.8890182971954346, -0.08989037573337555, 0.2166309654712677, -0.23451931774616241, -0.019834307953715324, 3.495293140411377, 0.24886101484298706, -0.2490532547235489, 5.713222980499268, 2.984616994857788, 3.381901979446411, 4.824244499206543, 2.78437876701355, -0.03346167132258415, -0.27385905385017395, 4.0049567222595215, -0.14091309905052185, 0.1144513413310051, 1.3840343952178955, 5.8075056076049805, 5.22593879699707, 3.5589354038238525, -0.23023422062397003, 3.627763032913208, 1.945755958557129, -0.21083775162696838, 5.462913513183594, 1.7969751358032227, -0.14487895369529724, 5.087183952331543, -0.26651647686958313, 2.2288131713867188, -0.014264366589486599, -0.013975033536553383, -0.08697841316461563, -0.08056087791919708, -0.003812206909060478, 8.616365432739258, -0.20270642638206482, 0.7269071340560913, 4.423530101776123, -0.17408299446105957, 1.8648992776870728, -0.024349164217710495, 0.5733360052108765, -0.26779696345329285, 1.805437684059143, 0.518900990486145, -0.022279515862464905, -0.004633833188563585, -0.07067365199327469, 2.839906930923462, 1.4744518995285034, 1.380144476890564, -0.0025798743590712547, -0.00023557957320008427, -0.1931145340204239, -0.27840331196784973, -0.20094862580299377, 1.1684024333953857, -0.2249431014060974, -0.04167625308036804, -0.10272006690502167, 0.3815157115459442, 0.1785636693239212, -0.12026023119688034, 9.08525276184082, -0.2712973952293396, 0.30763664841651917, -0.07350323349237442, -0.27630290389060974, 1.8794364929199219, 2.066225290298462, -0.21714703738689423, 1.3174654245376587, 0.12591655552387238, -0.004071864299476147, -2.684447645151522e-05, 9.243792533874512, 3.20810866355896, -0.10069398581981659, -0.00664388807490468, -0.030857982113957405, 5.39982271194458, -0.0001598872331669554, -0.2784261107444763, 3.3698432445526123, 0.36634954810142517, 2.8785767555236816, -0.19513332843780518, 9.573141098022461, -0.06624993681907654, -0.023335374891757965, 0.8234975934028625, 5.69959831237793, -0.09933993220329285, 4.13378381729126, -0.2268756777048111, -0.04102718457579613, 5.579045295715332, 6.263058662414551, 6.119230270385742, -0.2588634192943573, -0.045046884566545486, -0.04311048611998558, -0.25999319553375244, 2.1231284141540527, 1.801643967628479, -0.01842964068055153, 0.5354207754135132, 0.036717403680086136, 0.21240012347698212, -0.04514995962381363, -0.24907660484313965, 2.1708006858825684, 0.3167458772659302, 0.6413272023200989, -0.16400019824504852, 1.3134196996688843, -0.18471544981002808, -0.0017901945393532515, 1.6901980638504028, -0.27259397506713867, 3.616180181503296, -0.22262729704380035, -0.15526019036769867, 0.5926361083984375, -0.023150622844696045, -0.25627416372299194, -0.0054985410533845425, 7.366117000579834, -0.011401602998375893, -0.27766096591949463, 2.4340245723724365, 1.2472895383834839, 2.0564236640930176, 2.9411473274230957, 5.021419048309326, 0.911573052406311, 11.626466751098633, -0.069166399538517, 0.6483946442604065, -0.27571702003479004, -0.22134935855865479, 0.8725269436836243, 5.8159260749816895, -0.1566920131444931, -0.15641450881958008, 2.74652361869812, 1.9541819095611572, 7.827918529510498, 4.0452470779418945, 0.6563637256622314, -0.27830156683921814, 0.6052159070968628, -0.23242172598838806, -0.023119209334254265, 0.01674075238406658, -0.23235483467578888, -0.2631353735923767, -0.261410653591156, 6.752452373504639, 2.62315034866333, 3.6611297130584717, -0.1965513974428177, -0.06731212139129639, -0.2759564220905304, 0.7091465592384338, -0.277365118265152, -0.24107271432876587, 3.7652082443237305, 0.9191832542419434, -0.07537976652383804, -0.26177266240119934, -0.17680571973323822, 0.4873995780944824, -0.04339555278420448, -0.23487208783626556, 2.6072819232940674, 0.06074456498026848, -0.09210819005966187, -0.19306372106075287, 0.3093101978302002, 1.6878999471664429, 3.234602451324463, -0.20837144553661346, -0.026989854872226715, 5.6809611320495605, 0.9139358997344971, 1.3569064140319824, 0.9439571499824524, -0.06440883874893188, -0.2784643769264221, 0.7581188082695007, 4.065165996551514, 0.13478155434131622, -4.7456996981054544e-05, -0.2662561237812042, 1.1339426040649414, 5.146463394165039, 2.6889841556549072, 9.960365295410156, 5.678319454193115, 5.366190433502197, 3.6862707138061523, -0.02184988744556904, 3.277522087097168, 2.1981160640716553, -0.228715181350708, 2.14163875579834, 1.4078419208526611, -0.25054123997688293, -0.18708789348602295, -0.27236804366111755, 0.6595688462257385, -0.26270556449890137, 1.9582160711288452, -0.0035761967301368713, -0.27581986784935, -0.007970097474753857, -0.24448975920677185, 1.8355971574783325, 0.045091286301612854, -0.1730935424566269, 0.15639576315879822, 6.26624059677124, -0.05235861614346504, 3.451871395111084, -0.014226921834051609, -0.24974258244037628, 0.43058785796165466, -0.10930830985307693, 0.569750964641571, 2.7648680210113525, -0.03995563089847565, -0.13098742067813873, -0.2733765244483948, -0.0033034738153219223, -0.24034731090068817, -0.12605631351470947, 5.331185817718506, -0.10243472456932068, 6.790309906005859, -0.2258741706609726, 7.749247074127197, 1.933974027633667, 1.5612534284591675, 3.0993735790252686, 3.514094829559326, 5.592755317687988, 6.92406702041626, -0.2772466242313385, -0.033449940383434296, -0.0002440792741253972, -0.2714318335056305, 2.9022998809814453, 1.2403801679611206, 0.28252387046813965, -0.27692335844039917, -0.1837022453546524, -0.27606651186943054, 1.2848107814788818, -0.006065935827791691, 2.765869140625, -0.017024829983711243, -0.21954411268234253, -0.18586775660514832, -0.2745039463043213, -0.06246769055724144, -0.04497527703642845, -0.26344746351242065, -0.05533986911177635, 9.820540428161621, -0.26364994049072266, -0.17521384358406067, -0.21731387078762054, -0.057086050510406494, -0.0005190166411921382, -0.2721904516220093, 0.9367035627365112, -0.014315198175609112, -0.0030077570118010044, -0.037158749997615814, -0.0785512775182724, 1.130912184715271, 1.665939211845398, 1.448549509048462, -0.15607033669948578, 3.3858461380004883, -0.2534543573856354, 1.0901437997817993, 2.6240437030792236, -0.27837228775024414, -0.029193470254540443, 2.222781181335449, -0.04933544620871544, -0.09584281593561172, -0.051564350724220276, 3.2209033966064453, -0.23013615608215332, -0.0010811765678226948, 2.7528998851776123, 8.245413780212402, 7.1584343910217285, -0.20874173939228058, 0.15448087453842163, 1.7589246034622192, 3.4353795051574707, 0.052752237766981125, -0.052665550261735916, 2.83489727973938, 2.2228341102600098, -0.025012174621224403, -0.2732011079788208, 0.7624906301498413, -0.026341253891587257, 1.8205641508102417, 3.007852792739868, 1.8953115940093994, -0.1899557262659073, -0.023472081869840622, 2.218884229660034, 5.536982536315918, 7.852590084075928, -0.08232594281435013, -0.08650276064872742, 5.718684673309326, 5.687259197235107, 1.0340286493301392, 2.6395037174224854, -0.13385523855686188, 2.0728232860565186, -0.11295266449451447, -0.2198617160320282, -0.20047655701637268, -0.2446216344833374, -0.001724979723803699, -0.11271741986274719, 0.9984256625175476, -0.12758247554302216, -0.2669720947742462, 1.3930058479309082, 2.528158664703369, 3.2187139987945557, -0.1183546707034111, 0.012623569928109646, -0.276968389749527, -0.09729871898889542, -0.014217720367014408, -0.17757268249988556, -0.09861016273498535, 8.546008110046387, -0.17827251553535461, -0.26618319749832153, -0.2653774619102478, 7.431232929229736, -0.08114109188318253, -0.23413969576358795, -0.12982113659381866, -0.20485496520996094, -0.09401262551546097, 4.272478103637695, -0.018518386408686638, -0.06961140036582947, 6.388108730316162, -0.2285027801990509, -0.24474278092384338, -0.094746895134449, -0.000278739957138896, -0.1850312352180481, 4.590731620788574, -0.24387294054031372, -0.004315161146223545, 0.7293524146080017, 0.13961799442768097, 3.858596086502075, 3.4769091606140137, -0.2765152156352997, 1.2594962120056152, -0.005078042391687632, 4.333573341369629, 0.44798851013183594, 4.2144060134887695, 4.120231628417969, -0.27804699540138245, -0.22598567605018616, -0.09796142578125, -0.0020024485420435667, -0.21171003580093384, 0.3461723327636719, -0.05291113629937172, 2.616760015487671, 0.1892564445734024, 1.4871301651000977, -0.12972456216812134, -0.005723924841731787, 2.9052209854125977, 2.1740126609802246, -0.19063833355903625, -0.2589060068130493, -0.2772698998451233, -0.2159741073846817, -0.22879038751125336, -0.002432525157928467, 2.932802200317383, -0.014266773127019405, -0.2769613564014435, -0.27843692898750305, 0.7102994322776794, -0.009591817855834961, 4.722912788391113, 2.3756422996520996, 2.878190040588379, 4.977510929107666, -0.12138839066028595, -4.270541467121802e-05, 1.0025629997253418, 5.17474889755249, 4.560871124267578, -0.22121533751487732, -0.16141338646411896, 0.8340818285942078, -0.005093155894428492, -0.1702401340007782, -0.18241822719573975, 6.297336101531982, -0.17694415152072906, 6.251091003417969, 0.01845170371234417, -0.008084570057690144, 8.856639862060547, -0.07751055061817169, 2.2739999294281006, 6.814925193786621, -0.22521992027759552, -0.03558676317334175, 4.099928379058838, 0.14785532653331757, -0.03595297411084175, -0.05649583041667938, -0.2709566354751587, 3.7762088775634766, -0.27650824189186096, 2.4400932788848877, 6.601971626281738, 5.046240329742432, 5.732076168060303, 4.892185211181641, -0.01031558308750391, 4.855320453643799, 1.0358343124389648, -0.18814274668693542, -0.055909786373376846, -0.011016495525836945, -0.05158984661102295, -0.04090341553092003, -0.04235091060400009, -0.12005365639925003, -0.19294634461402893, 2.492853879928589, -0.27737799286842346, -0.20898474752902985, 5.2786993980407715, -0.27845993638038635, 2.197052001953125, -0.06071045994758606, 1.0382764339447021, 0.2951008379459381, 0.46885946393013, -0.2500949203968048, -0.06574080884456635, -0.27773594856262207, -0.24956580996513367, 4.068300247192383, 3.5376574993133545, -0.03892065957188606, 2.320570230484009, -0.24558676779270172, 1.5831596851348877, -0.24495567381381989, -0.016291968524456024, 3.996483564376831, 1.5545626878738403, -0.03614013269543648, -0.003916533663868904, -0.2255753129720688, -0.04146873950958252, -0.27531197667121887, 2.289581775665283, 5.859902381896973, -0.27621346712112427, 5.487197399139404, 4.613588333129883, -0.25997066497802734, -0.00846319179981947, 3.305616855621338, 2.2461700439453125, 9.41892147064209, 5.708510875701904, 3.0069427490234375, 6.755082130432129, -0.06293325126171112, 0.5416699647903442, -0.27369818091392517, 6.980307102203369, 2.0676138401031494, -6.038539868313819e-05, 5.482725143432617, -0.09566914290189743, 3.0090150833129883, -0.2487591803073883, 0.7467619776725769, 2.402092695236206, 0.5706562399864197, -0.08506903797388077, 5.628077983856201, 1.375332236289978, -0.005617423914372921, 2.729811668395996, -0.23086583614349365, 7.602565288543701, 0.6978923678398132, 1.25623619556427, 1.181138038635254, -0.09697650372982025, -0.044832609593868256, 0.6764547228813171, 1.208364486694336, 5.435173988342285, 1.6313683986663818, -0.0583837516605854, 1.9604960680007935, 3.275747776031494, 4.022152423858643, 1.1354154348373413, -0.03890066221356392, 4.68119478225708, -0.15578795969486237, 2.169823169708252, -0.012400059960782528, 3.005091428756714, -0.13676343858242035, 4.353116989135742, 7.126757621765137, 4.713164806365967, -0.04586175084114075, -0.08906315267086029, -0.10903557389974594, 2.553814649581909, 1.177425503730774, 0.47064319252967834, 1.0280723571777344, 2.186306953430176, 2.50016713142395, 6.8175249099731445, -0.07353964447975159, 0.6191461086273193, -0.004665181040763855, -0.027685128152370453, -0.1682555079460144, -0.042609211057424545, 0.6692737340927124, 4.150321960449219, -0.262690007686615, -0.24705903232097626, -0.23751142621040344, 0.35458213090896606, -0.22448815405368805, 1.4756577014923096, 0.9520209431648254, 1.1245454549789429, 2.9420933723449707, 0.07737252116203308, 0.3805987238883972, 7.045912265777588, -0.07837287336587906, 3.076207160949707, 0.12376455962657928, -0.25307658314704895, -0.2531962990760803, -0.11881948262453079, 0.02321663312613964, -0.19769546389579773, -0.2628471255302429, 4.836435317993164, -0.19850851595401764, 4.5084147453308105, 0.7841402888298035, 3.335118293762207, -0.19460509717464447, 0.3729027807712555, 10.533462524414062, 3.182365894317627, 3.9305102825164795, 0.6580633521080017, 3.306833267211914, -0.0435364730656147, -0.10819876939058304, 1.4329931735992432, 3.1388096809387207, -0.2650134265422821, -0.244387686252594, -0.27466651797294617, -0.24520167708396912, -0.03816681727766991, -0.018009623512625694, -0.14937792718410492, -0.006425902247428894, -0.07420606911182404, -0.17122238874435425, -0.059242188930511475, 0.6383960247039795, -0.25192123651504517, -0.15650823712348938, -0.06638304889202118, -0.13865311443805695, 1.7306854724884033, 1.0928542613983154, -0.2431652992963791, 1.087360143661499, 0.4321760833263397, 0.340958833694458, 6.265664100646973, -0.2620247006416321, 1.298920750617981, 1.3043041229248047, -0.27223172783851624, -0.039890777319669724, 2.4668052196502686, 0.6601582169532776, 0.03600337356328964, -0.019644098356366158, -0.02551981247961521, -0.08812462538480759, 0.7510443329811096, -0.0046629454009234905, -0.11388245224952698, -0.01311742514371872, -0.2700579762458801, 2.517238140106201, -0.27800774574279785, -0.22904175519943237, 1.6180340051651, -0.27845194935798645, -0.2532997727394104, -0.2575396001338959, 5.829591274261475, 2.2584946155548096, 1.4544061422348022, -0.03423816338181496, 2.0492727756500244, -0.024492893368005753, 1.4339524507522583, -0.042783450335264206, 7.578554630279541, -0.006570589262992144, 4.6103901863098145, 2.8241331577301025, -0.004770495928823948, -0.27779319882392883, 5.515157699584961, 4.244969367980957, -0.00709114084020257, -0.2072150856256485, -0.03511309623718262, -0.05565342307090759, -0.1590210646390915, 7.109726905822754, 0.16122987866401672, -0.27671292424201965, -0.2692338526248932, 2.211857318878174, 2.320570945739746, -0.07562415301799774, 5.592381477355957, -0.046228569000959396, -0.2776730954647064, 1.2725249528884888, 6.574703693389893, -0.26624733209609985, 0.62600177526474, 4.915185928344727, 2.0170764923095703, 4.188506603240967, 3.814833879470825, 0.5342459082603455, 2.049497604370117, 3.29655122756958, -0.15160520374774933, 6.118967056274414, -0.002243706490844488, -0.07502871006727219, 5.492599010467529, 1.172389030456543, -0.15719379484653473, 0.9745298027992249, 4.019972801208496, -0.19023971259593964, 2.654121160507202, -0.24000397324562073, -0.2341572791337967, -0.1972752809524536, 4.126758098602295, -0.04125267267227173, -0.2608397603034973, 2.194312810897827, -0.03015865758061409, -0.009137427434325218, 1.784691333770752, -0.10872051119804382, -0.2536108195781708, 0.1824120581150055, -0.0647929310798645, 11.396573066711426, -0.06253591924905777, -0.2389005720615387, -0.015601967461407185, -0.2457810640335083, 5.834604263305664, -0.0809587761759758, 1.899182677268982, -0.22072727978229523, 6.9923930168151855, 0.2409759908914566, -0.027483172714710236, 2.668544292449951, -0.2603920102119446, 1.9525108337402344, 0.1744411140680313, 0.5523512959480286, 0.1676843762397766, -0.07876213639974594, -0.011918113566935062, 3.556455612182617, -0.24752889573574066, 0.6288712024688721, -0.22810913622379303, -0.0019769975915551186, -0.02501392364501953, 5.299412250518799, -0.1099780723452568, -0.2562699615955353, -0.013349260203540325, -0.20267963409423828, -0.25909027457237244, -0.27647632360458374, -0.2028329223394394, -0.1615368276834488, -0.18352605402469635, -0.13480506837368011, 0.7538484334945679, -0.25277572870254517, 6.6935882568359375, 0.3504139482975006, 0.22169820964336395, -0.15166892111301422, 3.754570722579956, -0.054678160697221756, -0.007912851870059967, -0.06312617659568787, 1.979488730430603, -0.03749656304717064, 4.859916687011719, -0.1513778120279312, -0.01111678872257471, -0.27773118019104004, 6.844189643859863, -0.08782533556222916, -0.021976353600621223, 1.5073442459106445, 4.905182361602783, 3.6956491470336914, -0.2687866687774658, 0.28526079654693604, 0.7440676093101501, 4.356971740722656, 8.306296348571777, -0.2352754920721054, 4.874815464019775, -0.2650626301765442, 1.8376212120056152, -0.07093929499387741, -0.11411543190479279, 1.0968314409255981, -0.07887271046638489, 3.141235589981079, -0.1907353550195694, 0.011814148165285587, -0.21700428426265717, -0.26762375235557556, 0.13264583051204681, 3.9600589275360107, 12.13427734375, -0.032575491815805435, 0.6524693369865417, -0.26809632778167725, -0.07518985122442245, 0.6033592224121094, -0.08308345824480057, -0.2123824954032898, 4.629305362701416, -0.0019388390937820077, 0.4712632894515991, 1.2270230054855347, -0.0660654678940773, 1.5546320676803589, 0.8444393277168274, -0.2637878358364105, 4.51470422744751, -0.022102219983935356, 2.075213670730591, -0.278089314699173, -0.048995278775691986, -0.26869064569473267, -0.2636944353580475, 2.8455770015716553, -0.011868474073708057, -0.27302515506744385, -0.2282160222530365, 5.989645004272461, -0.04042647033929825, 3.8712317943573, -0.004861739464104176, 0.8093204498291016, -0.2532687783241272, 0.9346807599067688, -0.1325172781944275, -0.0024402670096606016, -0.009850331582129002, 4.9023566246032715, 0.3386495113372803, -0.08455894887447357, 3.156888484954834, -0.11685802042484283, -0.11854536086320877, -0.02889024093747139, 4.320001602172852, 7.507605075836182, -0.06219293549656868, 0.5629871487617493, 3.465139627456665, 8.10874080657959, -0.0004768609069287777, 0.11474868655204773, 2.0045268535614014, -0.011733798310160637, 0.9881919622421265, 0.2547608017921448, 2.3365225791931152, -0.1698666661977768, 1.019098162651062, -0.16089394688606262, 1.813681721687317, 0.14410485327243805, -0.2484104186296463, -0.014332941733300686, 6.199420928955078, 3.986381769180298, 3.007979393005371, -0.13886044919490814, -0.25663161277770996, 6.287594318389893, 2.4166080951690674, -0.08390466868877411, -0.08739718794822693, 0.10296837985515594, -0.27724355459213257, -0.06448109447956085, 4.603201389312744, 1.748468279838562, -0.2767244279384613, -0.008548129349946976, -0.17047639191150665, -0.15292829275131226, 2.9851036071777344, -0.25298556685447693, 4.018836498260498, 0.12067453563213348, -0.11599166691303253, -0.00011567179171834141, -0.03263993561267853, -0.011614197865128517, -0.26342976093292236, 0.7371611595153809, 5.212671279907227, -0.15027783811092377, -0.034897711127996445, 9.098136901855469, 0.6740333437919617, -0.277778297662735, -0.277891606092453, -0.018876301124691963, 2.0311052799224854, 0.9223533272743225, 6.049098968505859, 4.0541090965271, 3.0624520778656006, 4.711129188537598, -0.2765500843524933, -0.23384687304496765, -0.18068616092205048, -0.027368128299713135, -0.25278446078300476, -0.26997697353363037, 2.222226619720459, -0.1311807483434677, -0.002975553972646594, -0.05734282732009888, 5.400646209716797, -0.0038732162211090326, -0.1196710467338562, -0.07442296296358109, -0.02181507647037506, -0.19757471978664398, 0.001963796326890588, -0.022491665557026863, 0.6977530717849731, 1.15144681930542, -0.026771750301122665, -0.04496987164020538, 4.054973602294922, -0.2599533796310425, 1.2802482843399048, 2.184258222579956, -0.27566513419151306, 3.6641149520874023, 1.0851004123687744, -0.06500518321990967, 1.674630880355835, 3.1769087314605713, 2.921675682067871, -0.1669314056634903, 1.4230983257293701, 5.317151069641113, 0.5343846678733826, 4.55315637588501, 5.036501884460449, -0.11910450458526611, 2.327877998352051, -0.27846020460128784, 4.089191436767578, -0.18320664763450623, -0.20833076536655426, -0.09511964023113251, -0.15993043780326843, 1.1469240188598633, 1.5001118183135986, -0.04044245555996895, -0.1696605533361435, 1.0663293600082397, 1.5792357921600342, 0.14332972466945648, 10.450448989868164, 4.3437604904174805, -0.1534988284111023, 2.6911072731018066, 5.264557838439941, 1.1298152208328247, -0.006205725483596325, 0.2022412270307541, 0.08936470746994019, 8.497331619262695, -0.0046095517463982105, 6.151641845703125, -0.22088955342769623, 3.6271398067474365, -0.27719277143478394, 0.46962276101112366, 4.48397970199585, 0.2037101835012436, 4.170957088470459, -0.12939715385437012, -0.267796128988266, -0.25449562072753906, -0.2399112582206726, -0.17491036653518677, 5.4610276222229, -0.04430873319506645, -0.008961182087659836, -0.014459967613220215, 6.778838157653809, 4.177989482879639, 5.320173263549805, 5.028410911560059, -0.15536430478096008, 0.34058278799057007, -0.00027019588742405176, -0.2029813975095749, 2.3149666786193848, -0.013089682906866074, -0.002326727146282792, -0.030603662133216858, 0.697823703289032, 4.725093841552734, 10.44905948638916, 5.815035820007324, -0.01549155730754137, -0.0009632016881369054, 0.49180954694747925, 4.320514678955078, -0.011277084238827229, -0.24395516514778137, -0.2665284276008606, 0.17645467817783356, -0.19883032143115997, 2.039398670196533, 4.375373840332031, 0.30554431676864624, 3.090489149093628, 8.191136360168457, -0.0796058177947998, -0.0742340162396431, 2.0659525394439697, 7.6535563468933105, 0.6601561307907104, 4.588669300079346, -0.2662180960178375, 3.8630635738372803, -0.2169422209262848, 2.843146562576294, 1.6280477046966553, 1.6330358982086182, 5.8012800216674805, 4.545743465423584, -0.23233047127723694, 3.801273822784424, -0.010188344866037369, 4.256664752960205, 11.365701675415039, 2.3096466064453125, 2.220947265625, 0.4278698265552521, 3.6495063304901123, -0.2772781252861023, 2.9821417331695557, -0.22212174534797668, -0.2783529460430145, 0.6906682252883911, 0.12678992748260498, -0.01002263743430376, 8.785625457763672, 0.7741984724998474, -0.011617766693234444, 4.764730453491211, 0.2267322540283203, 1.0367764234542847, 3.211354970932007, -0.13522757589817047, -0.00014300458133220673, 0.05298278108239174, 1.194419503211975, 4.462953567504883, -0.18702326714992523, -0.2049107402563095, 1.8693886995315552, -0.24277077615261078, -0.18783335387706757, 1.2482295036315918, -0.10380508750677109, 6.830038070678711, -0.12622909247875214, 0.4221688508987427, -0.25067174434661865, -0.20614342391490936, 2.460664749145508, 4.491491794586182, 6.305837154388428, 2.6619136333465576, -0.27840426564216614, 3.8651740550994873, -0.026529276743531227, 0.4524163603782654, 2.389712333679199, 2.568967580795288, -0.06516901403665543, -0.02194535732269287, 7.100810527801514, -0.26918938755989075, -0.013183104805648327, -0.04879053682088852, 0.30201077461242676, 5.117105484008789, 0.24561060965061188, 5.186699390411377, 1.0331089496612549, 2.075770616531372, 4.274581432342529, -0.2754203975200653, 1.2950853109359741, -0.2772723436355591, -0.2215462327003479, 2.8463504314422607, -0.1488906592130661, 0.4460707902908325, -0.15893498063087463, -0.007403298746794462, -0.2572890520095825, 4.663363456726074, 0.4179697334766388, 0.40445077419281006, 6.89029598236084, -0.00014129643386695534, -0.06251934170722961, -0.00024065760953817517, 0.12042302638292313, 1.7489535808563232, 2.6399548053741455, 1.011741280555725, -0.1189444437623024, 7.206624507904053, 1.1357632875442505, -0.2555663287639618, -0.057238444685935974, 1.6006313562393188, -0.2551805078983307, -0.26623600721359253, 0.7103599905967712], "dtype": "float32"}, {"__nd__": [0.5681729316711426, 1.0190973281860352, 1.0285824537277222, -0.09271267801523209, 0.019970370456576347, -0.09533762186765671, 1.0991297960281372, 0.3888039290904999, 1.0885049104690552, -0.003942237701267004, 1.009961724281311, 0.3118136525154114, 1.0958788394927979, 0.2398669570684433, -0.04381999373435974, 1.0815247297286987, 1.0777091979980469, 0.13524140417575836, 0.21243499219417572, 1.0953924655914307, -0.07096032798290253, -0.01177480909973383, 1.0529612302780151, -0.09671591967344284, -0.0030793652404099703, -0.07923217862844467, -0.0828612670302391, -0.031301915645599365, -0.012624935247004032, 0.5730141997337341, 1.067007064819336, 0.09343007206916809, -0.08892399817705154, 1.0473318099975586, 1.0921052694320679, 0.01799166575074196, 1.097324013710022, 1.0472265481948853, 0.12891624867916107, -0.07931017130613327, 1.022609829902649, 0.9246873259544373, 1.040719985961914, -0.02396520972251892, -0.09740760922431946, -0.07570676505565643, -0.005758253857493401, 0.7424893975257874, 1.0953936576843262, -0.08952823281288147, 1.017636775970459, 1.0780872106552124, 1.0993170738220215, 1.0565296411514282, 1.0586577653884888, -0.02078258991241455, 1.0996984243392944, 1.0998331308364868, -0.005147676914930344, 1.0221478939056396, 0.07182031869888306, 1.0838630199432373, -0.0732802227139473, 0.4635506272315979, 1.0189932584762573, -0.07008744776248932, 1.099838137626648, 0.2960818409919739, 1.0892047882080078, -0.09456013888120651, 1.042127013206482, 1.099818468093872, 0.17189960181713104, 1.0991363525390625, 1.025795817375183, 1.0047600269317627, -0.009738067165017128, 1.069076418876648, 1.0821155309677124, 0.3165970742702484, -0.05179491639137268, 1.0259919166564941, -0.025645319372415543, 1.098987340927124, 1.0227583646774292, -0.008949510753154755, 0.015588521957397461, -0.022261319682002068, 0.9203921556472778, 1.0117747783660889, 1.0021642446517944, -0.004972207825630903, -0.09936003386974335, 1.0936113595962524, 1.0095573663711548, 1.065905213356018, -0.08859872072935104, 0.9792113900184631, 0.46678608655929565, 0.1335020512342453, -0.09006260335445404, 1.0486440658569336, 0.9894495010375977, 0.31823211908340454, 0.10853806138038635, -0.023941246792674065, -0.08776240050792694, 1.030133843421936, 0.15280675888061523, -0.014909466728568077, 1.0206276178359985, 1.0957139730453491, 1.0016169548034668, 0.623733639717102, -0.08099444210529327, -0.014542660675942898, 1.0213013887405396, 1.0003185272216797, -0.07823163270950317, -0.09836909174919128, 1.0998295545578003, -0.0768556147813797, 0.04850652441382408, -0.024223828688263893, 0.8492826223373413, 1.041162133216858, 0.5440779328346252, 1.0638638734817505, -0.06531430780887604, 1.0069725513458252, -0.0014569911872968078, -0.014437908306717873, 1.0818384885787964, -0.09976907074451447, 1.0995354652404785, 1.0970940589904785, 1.0214020013809204, 1.07955002784729, 1.0504647493362427, 0.28386804461479187, -0.09415910392999649, -0.09041812270879745, -0.08820097893476486, -0.03740967810153961, -0.0975843146443367, 0.3190070390701294, 0.9296137690544128, -0.01593044027686119, -0.0030445910524576902, 0.357423335313797, 0.8963600397109985, 1.0423225164413452, 1.0994681119918823, -0.0985703319311142, -0.09134795516729355, 1.0022624731063843, 1.0872377157211304, 1.0041590929031372, 1.0015021562576294, -0.07864686101675034, 1.047492504119873, 1.0799809694290161, 1.0985445976257324, 1.0970489978790283, 0.8640740513801575, 0.8220224976539612, -0.012228867039084435, 0.1918349266052246, -0.08841075748205185, 0.741878092288971, -0.08284632116556168, 1.099594235420227, 1.0785025358200073, 0.9902012348175049, 0.9969680309295654, 0.6816980242729187, -0.07649186253547668, 0.193282350897789, -0.08988761901855469, 1.0490608215332031, 0.5282118320465088, 1.0995769500732422, -0.020993433892726898, 1.0683937072753906, -0.08643347769975662, -0.09008578211069107, 0.14262376725673676, -0.09878484159708023, 0.9732474684715271, -0.09932847321033478, -0.04404702037572861, -0.074582040309906, 1.0220767259597778, 0.5880564451217651, -0.033671341836452484, 0.5167517066001892, -0.010307304561138153, 1.0047858953475952, -0.011401474475860596, 1.0859235525131226, 1.0972222089767456, -0.014435202814638615, 0.9822646379470825, 1.0530363321304321, 1.0828547477722168, 0.9406952261924744, 1.0595747232437134, 0.8008983731269836, 1.0139880180358887, 0.25702327489852905, 1.0991427898406982, -0.06065870076417923, 1.082426905632019, -0.0918097272515297, -0.04162636026740074, 1.0350017547607422, 1.0065958499908447, 1.0578255653381348, -0.09360834211111069, 1.094182014465332, -0.07868970185518265, 0.2617711126804352, 0.6753020882606506, -0.0007797206053510308, 0.848770022392273, -0.04886891320347786, -0.09424963593482971, -0.013368290849030018, -0.011693435721099377, -0.06039359048008919, 1.0775508880615234, 1.0072063207626343, 0.449675053358078, 1.041499137878418, 0.05590212345123291, -0.0031681181862950325, 0.782787561416626, 1.0757250785827637, -0.06910037249326706, 0.9841316342353821, 1.0992604494094849, -0.07198352366685867, -0.013265470042824745, 0.10912320762872696, 0.12313473969697952, -0.026277627795934677, 1.0543525218963623, 1.0059213638305664, -0.06383800506591797, 1.094097375869751, 1.0607227087020874, 1.092565655708313, 1.0466645956039429, -0.04977601021528244, 1.0894012451171875, 1.0021578073501587, 1.0807310342788696, -0.09955772757530212, 1.0997947454452515, 1.0255967378616333, 1.0374609231948853, -0.09878581017255783, 1.0107423067092896, -0.026565609499812126, 1.0914549827575684, 0.4676714241504669, -0.031260449439287186, 1.0290231704711914, 1.0227856636047363, -0.03374098613858223, 1.0998365879058838, -0.07033289968967438, -0.0032683818135410547, -0.0950891301035881, 0.9436078667640686, 1.0989701747894287, 0.8084279894828796, 0.5898410677909851, 1.0835490226745605, 0.12325286865234375, 1.032785415649414, 0.78065425157547, 0.9760458469390869, 1.0969347953796387, -0.04549923166632652, -0.09299995005130768, 1.0112528800964355, 1.013099193572998, 0.03550180420279503, 1.0179433822631836, -0.08665279299020767, 0.9602099061012268, -0.0970875695347786, 0.9574712514877319, -0.009609167464077473, 1.020849585533142, -0.09316574037075043, -0.009499898180365562, 0.10564619302749634, -0.038759082555770874, 0.7869214415550232, -0.03584291413426399, 1.0542994737625122, 1.0000064373016357, -0.024390706792473793, 1.0150079727172852, 0.007938282564282417, 0.8502482771873474, -0.004442069213837385, -0.03527122735977173, 1.0804550647735596, 0.4234652817249298, 1.0079712867736816, -0.09347836673259735, 0.8226030468940735, 1.0000927448272705, -0.08604893833398819, -0.06604700535535812, 0.429970383644104, 0.41134440898895264, 1.0748231410980225, 0.7479998469352722, -0.08722437918186188, 0.8773699998855591, 1.0002493858337402, 1.0248953104019165, -0.0001487361587351188, 0.23053397238254547, -0.07026000320911407, 1.0994890928268433, 0.179337278008461, 1.0029524564743042, 1.0620981454849243, 1.0680088996887207, 0.8411962985992432, 0.4197523295879364, 1.016935110092163, 1.0232115983963013, 1.09622061252594, -0.025169411674141884, 1.0246517658233643, 0.8495080471038818, 0.34722793102264404, -0.08248138427734375, -0.09893545508384705, -0.09895411878824234, -0.0001433713623555377, 1.0504834651947021, 1.0989621877670288, 0.11866865307092667, 1.0468472242355347, 1.072867512702942, 1.039124846458435, -0.00044688215712085366, 0.0935802087187767, 1.0959548950195312, 1.0086627006530762, 1.0040370225906372, 1.0155693292617798, -0.03423789516091347, -0.0011849193833768368, 1.0991653203964233, 0.555119514465332, 1.0577046871185303, 0.25847935676574707, 0.8888474106788635, -0.01232686173170805, 1.0765665769577026, -0.0754539966583252, 1.024332046508789, 1.0803749561309814, 0.2738712430000305, 0.9975288510322571, 0.4727993905544281, 1.0868991613388062, 1.0911294221878052, -0.09756358712911606, -0.0997321829199791, 1.0055748224258423, 0.3022662103176117, 1.0537452697753906, 1.0520882606506348, -0.09882815182209015, -0.08823657035827637, 0.731316864490509, 0.9356096982955933, 1.0996898412704468, -0.02525450475513935, -0.0998196229338646, 0.31472453474998474, 1.0089179277420044, -0.0006702799000777304, 1.0732991695404053, -0.09685542434453964, 1.0702418088912964, -0.08909345418214798, 1.0345360040664673, 1.0100229978561401, 1.0273773670196533, 1.0710612535476685, 1.03994619846344, -0.0880635604262352, 1.097161889076233, 0.1269490122795105, -0.07957110553979874, 0.4232085347175598, 0.10106497257947922, 1.0876519680023193, 0.4147389531135559, -0.09560828655958176, 1.0141676664352417, 1.0542004108428955, 1.0250465869903564, 1.0997952222824097, 0.2874443531036377, -0.06495559960603714, 0.03896464407444, 1.0597714185714722, 0.7165517210960388, -0.01301573310047388, 0.422151654958725, -0.016048820689320564, 0.8089110255241394, -0.03036283329129219, -0.025479333475232124, 0.7285575270652771, 0.3460862636566162, -0.007308349944651127, 1.088333010673523, 1.0013365745544434, -0.013076316565275192, 1.004323959350586, -0.09519556164741516, -0.08955516666173935, 0.17056354880332947, -0.01465105451643467, 1.0229458808898926, 1.0892510414123535, -0.08791570365428925, 0.0413871593773365, -0.011700850911438465, -0.022689305245876312, 1.0613902807235718, -0.006126786582171917, -0.07218804955482483, 0.7775474786758423, 1.046573281288147, 1.0197142362594604, -0.07968712598085403, -0.08933860063552856, 1.077399730682373, -0.03366520628333092, 0.008578281849622726, 1.0428110361099243, 1.0066663026809692, 0.9462411999702454, 0.15033668279647827, 1.0992023944854736, -0.0034419377334415913, 0.46353426575660706, -0.045093849301338196, 1.0006049871444702, 1.0917649269104004, 0.7683640122413635, 0.14714983105659485, -0.0003873954701703042, 1.0926049947738647, 1.0081249475479126, -0.09074877947568893, 1.0956047773361206, 1.0914926528930664, -0.010569610632956028, 1.0614575147628784, 1.019389271736145, 0.9005054831504822, 0.37496837973594666, 1.0829623937606812, -0.010822691954672337, 0.6823212504386902, 1.0458372831344604, -0.027143076062202454, 1.0536441802978516, -0.06456352770328522, -0.0017771142302080989, 0.2946456968784332, -0.0564238503575325, 0.9996891617774963, -0.03323538228869438, 1.0463266372680664, -0.08952320367097855, 1.0289099216461182, -0.09981384873390198, -0.002840326400473714, 1.0314311981201172, 0.882723331451416, -0.05298846960067749, -0.026430033147335052, 1.0343979597091675, 1.0902658700942993, -0.005916605703532696, 0.9194701910018921, -0.04887287691235542, -0.050739530473947525, -0.0774645283818245, 1.0268269777297974, 0.9779496192932129, -0.0887070819735527, 1.089654564857483, 1.083420753479004, 0.820103645324707, 1.047777771949768, 0.030577929690480232, 0.281427800655365, 1.0843510627746582, 0.4321218430995941, 1.0821243524551392, -0.026340022683143616, 1.0155729055404663, -0.08389788866043091, 0.19296588003635406, -0.018082553520798683, 1.099839210510254, 1.0247421264648438, 1.0438640117645264, 1.0764110088348389, -0.010267697274684906, -0.0837731584906578, -0.08135758340358734, -0.06160026416182518, 1.0381520986557007, 0.8380289077758789, 0.451122522354126, 1.0880826711654663, -0.059221915900707245, -0.06539054214954376, -0.06932191550731659, 1.090300440788269, 1.0180302858352661, 1.0994759798049927, 1.043016791343689, -0.01924067735671997, -0.0986032634973526, 0.7788025140762329, -0.01974998414516449, -0.09595319628715515, 1.0508440732955933, 1.0402995347976685, 1.0432096719741821, 0.5856126546859741, 0.1218678429722786, 1.0428861379623413, -0.03981183096766472, 1.015419602394104, 1.021208643913269, -0.036536600440740585, -0.09975725412368774, 0.48126181960105896, 1.0998377799987793, 1.0868486166000366, 0.7926417589187622, 1.0983206033706665, -0.008951560594141483, -0.07259501516819, 0.8046607971191406, -0.06246529892086983, -0.02017970196902752, -0.030015122145414352, -0.06397144496440887, 0.17120207846164703, -0.04974812641739845, 0.9088454842567444, 1.0065251588821411, 0.2210492342710495, 1.0246801376342773, 0.9705161452293396, -0.03845430165529251, 0.06467855721712112, -0.05507797375321388, 1.085728406906128, -0.03506595268845558, 1.0213828086853027, 1.0994793176651, -0.002173712244257331, 1.0834699869155884, 1.0104050636291504, -0.08260183036327362, 1.054619312286377, -0.004646478220820427, 0.9302673935890198, -0.007317280396819115, 1.0868442058563232, -0.004811281803995371, -0.0002543150039855391, 1.0535629987716675, 1.0008647441864014, -0.09982647746801376, 1.076882004737854, 0.714392364025116, -0.003352515632286668, 1.0996683835983276, 1.0077868700027466, -0.027729211375117302, 1.0988154411315918, 1.0207271575927734, -0.015642093494534492, 1.0085229873657227, 1.0027897357940674, -0.05906106159090996, 1.005684733390808, 0.6113553047180176, 0.1953033059835434, 1.0121917724609375, 0.8991256952285767, 1.0030717849731445, 1.0116519927978516, 1.0237064361572266, 1.067606806755066, -0.016842780634760857, -0.009775415994226933, 1.0377923250198364, -0.0627429261803627, 0.7803892493247986, -0.04351343959569931, 0.16318723559379578, -0.024388719350099564, 1.008103847503662, -0.05021973326802254, 0.2653595805168152, 0.15634572505950928, 1.0301644802093506, -0.015860429033637047, 0.4335130751132965, 0.1256975680589676, 1.0956705808639526, 1.084118127822876, -0.08171803504228592, 0.9148083329200745, -0.04106344282627106, 1.083011507987976, 0.30045443773269653, -0.0978328213095665, 0.4449973404407501, 1.0340214967727661, 1.0009342432022095, -0.08892728388309479, 0.9883137941360474, 1.013856053352356, -0.0017934940988197923, -0.005588319152593613, -0.03171101212501526, 0.2792355716228485, 1.0005054473876953, 1.0509697198867798, 1.0299930572509766, 0.1080646887421608, -0.0004647386958822608, -0.0007895940216258168, 1.0502572059631348, 0.8385999202728271, -0.02298908680677414, 1.0387189388275146, -0.010885289870202541, 1.0369559526443481, 1.0143747329711914, -0.08614824712276459, -0.03970464691519737, 1.0747997760772705, -0.041109710931777954, 1.0722081661224365, -0.031119972467422485, 1.0801682472229004, 0.6834144592285156, -0.07081092149019241, -0.06723573803901672, 0.9122628569602966, -0.0033825014252215624, 0.04258790239691734, -0.05359591171145439, 1.0960575342178345, -0.02625088579952717, 1.006164312362671, 0.007152890786528587, -0.036556024104356766, 1.017613172531128, 1.0072801113128662, 0.15698979794979095, -0.06791077554225922, 0.5320838093757629, -0.09494396299123764, 0.00674173841252923, 1.0102256536483765, 1.0512672662734985, 0.7751907110214233, 1.0213866233825684, 0.3848574161529541, 1.0752061605453491, 1.069641351699829, -0.03793473541736603, 0.7041473984718323, 1.0005550384521484, 1.066077709197998, 0.1598908007144928, 1.0019786357879639, 0.5188606381416321, -0.0408867746591568, 1.0963830947875977, -0.06663089990615845, -0.09021800756454468, 1.0057401657104492, 0.25692588090896606, -0.08632766455411911, -0.0978914126753807, -0.09980829060077667, 1.033799171447754, -0.09167219698429108, -0.099245585501194, 1.002057433128357, -0.09811925888061523, 1.0127472877502441, 1.0218288898468018, 0.21249234676361084, -0.015885449945926666, 1.0782901048660278, 1.0965427160263062, 1.0569831132888794, 0.6418308615684509, 1.0302983522415161, 1.0173823833465576, -0.04583258181810379, 1.0153932571411133, 0.8226701021194458, -0.09892165660858154, 0.0008634626283310354, 1.0035595893859863, 1.0547831058502197, -0.004793891683220863, 0.6638657450675964, 0.2342308759689331, 0.17569497227668762, -0.09319878369569778, -0.014152554795145988, 1.0722439289093018, -0.06089802458882332, -0.038144368678331375, -0.0998346135020256, -0.09086813032627106, 1.0889694690704346, 1.0466169118881226, -0.09641841053962708, -0.060175955295562744, -0.08338508754968643, 1.0570900440216064, 0.7415857911109924, 0.9065074324607849, -0.0072763534262776375, 0.06295143067836761, 0.8844640851020813, -0.0039943549782037735, 1.008596658706665, -0.0830073207616806, -0.09905918687582016, 1.0203757286071777, 0.8924658298492432, 0.5837713479995728, -0.09963204711675644, -0.010135495103895664, 0.8387303948402405, 1.0311179161071777, -0.06665582954883575, 1.0189220905303955, -0.03768803924322128, -0.09011902660131454, 1.0919194221496582, 0.5001304149627686, 1.0948976278305054, -0.08915875107049942, -0.04859994724392891, -0.003688354743644595, -0.030372295528650284, -0.04327850043773651, -0.015514777041971684, 0.9899876713752747, -0.09723890572786331, -0.0024424963630735874, 1.0004334449768066, -0.0761939138174057, -0.026120679453015327, 1.02735435962677, 1.0338712930679321, -0.035238537937402725, -0.013385552912950516, -0.05265586078166962, 1.037445306777954, 0.0074440836906433105, -0.08000683784484863, 0.4433768093585968, 1.0303301811218262, -0.017425579950213432, -0.04663315787911415, -0.01775520108640194, 1.0012948513031006, 1.0043458938598633, 1.0592856407165527, 0.24533750116825104, 1.097586750984192, -0.0016516793984919786, 0.9579119086265564, -0.020520377904176712, -0.021241089329123497, -0.09348598122596741, -0.024501387029886246, 1.0993306636810303, 1.093912124633789, 0.94260174036026, -0.06024264544248581, 1.0962631702423096, 0.2672208249568939, 1.0324820280075073, -0.06412038207054138, -0.00015924408216960728, -0.08538973331451416, 1.0013927221298218, 1.0832629203796387, 0.14720217883586884, 1.0690022706985474, 1.091408371925354, 1.0809810161590576, -0.09326387196779251, 0.6500174403190613, 0.8628286719322205, -0.0986194908618927, -0.03856978565454483, 0.7643530964851379, 1.0568945407867432, 1.0983562469482422, 1.0895308256149292, 1.0916242599487305, 1.0003026723861694, 0.9393837451934814, 1.0473294258117676, 1.002717137336731, 0.6053346991539001, 0.9447237253189087, 1.0470837354660034, 1.0067120790481567, 0.9942277073860168, 1.0335053205490112, 0.6777591109275818, -0.030207738280296326, 1.0386823415756226, 0.06932271271944046, -0.006056979298591614, 0.22693519294261932, -0.005893180146813393, -0.05205094441771507, 0.5421180129051208, -0.06414993107318878, 1.0433686971664429, -0.013807673007249832, 0.9499087929725647, 1.0988372564315796, 1.013663649559021, 1.062278151512146, 0.9168422222137451, 0.6736974716186523, 0.40060004591941833, 1.0356214046478271, -0.09832696616649628, 1.0957105159759521, 0.3348683714866638, 0.9281520843505859, 0.0694572776556015, 1.0014208555221558, -0.09838991612195969, 0.9439014196395874, 0.9689345955848694, 1.0995755195617676, 1.081108570098877, 1.072107195854187, 1.0535212755203247, -0.012592864222824574, 0.5434137582778931, -0.028835788369178772, 0.9210623502731323, -0.03337991237640381, -0.05975373461842537, 0.4041031002998352, 1.0164152383804321, 1.099766492843628, 0.9907810688018799, 1.0160467624664307, -0.061952028423547745, -0.0017882141983136535, -0.06102949008345604, 0.05478771775960922, 1.026700496673584, 1.0013864040374756, -0.021789314225316048, 1.0004221200942993, 1.0095977783203125, 1.0736271142959595, 1.0454516410827637, 1.0005054473876953, 0.6843608617782593, -0.080516017973423, 0.787883996963501, -0.01935681514441967, 1.087716817855835, 0.31296253204345703, 1.0203220844268799, 1.0408151149749756, -0.04699266701936722, 1.0208873748779297, -0.05257667973637581, 1.0662611722946167, -0.009566628374159336, 1.0478458404541016, -0.08045823127031326, 0.6134966015815735, -0.09892255812883377, -0.001311688800342381, -0.04061486944556236, 0.6643790006637573, 1.0988034009933472, -0.014258173294365406, -0.001607521204277873, -0.07295189052820206, 0.14468221366405487, -0.09961196035146713, -0.08207529038190842, 1.0715968608856201, 1.055147647857666, 1.0871645212173462, -0.06334134191274643, 0.6793466806411743, 0.1715783327817917, -0.01625301130115986, 1.066912055015564, 0.7009825110435486, -0.08341360092163086, 1.015228033065796, 1.0841844081878662, 1.0708073377609253, 1.0293656587600708, 1.0902223587036133, -0.026545535773038864, -0.040446631610393524, 1.050299882888794, -0.0875992625951767, 0.6029905676841736, 1.061877965927124, 1.0141583681106567, 1.0219882726669312, 1.0647417306900024, 0.1810811161994934, 1.0624157190322876, 1.0970664024353027, 0.22045700252056122, 1.018427848815918, 1.092322826385498, -0.08904272317886353, 1.0243366956710815, 0.08198743313550949, 1.099807620048523, -0.011871892027556896, 0.48582595586776733, -0.061677467077970505, -0.05791555345058441, -0.0033084743190556765, 1.0013768672943115, 0.2355782687664032, 0.9263133406639099, 1.0385949611663818, 0.28417226672172546, 1.094819188117981, -0.019727256149053574, 0.869964599609375, -0.057852957397699356, 1.0926660299301147, 0.8468317985534668, -0.018142839893698692, -0.004002292174845934, -0.05187087506055832, 1.088629961013794, 1.0711654424667358, 1.0614397525787354, -0.0022583550307899714, -0.00021361123071983457, -0.0996483713388443, -0.005108566954731941, -0.09983185678720474, 1.0321500301361084, 0.19236500561237335, -0.03248044103384018, -0.07034004479646683, 0.7797783017158508, 0.6523549556732178, 0.36216244101524353, 1.0009156465530396, 0.06190025433897972, 0.7378129959106445, -0.053631071001291275, -0.0286782905459404, 1.0952792167663574, 1.0991185903549194, 0.20821338891983032, 1.0539098978042603, 0.612205982208252, -0.0035282447934150696, -2.4794897399260662e-05, 1.000796914100647, 1.076771855354309, 0.38744062185287476, -0.005681879818439484, -0.024623019620776176, 1.0193228721618652, -0.00014554875087924302, -0.004055935889482498, 1.0712225437164307, 0.7715295553207397, 1.087480068206787, 0.24907909333705902, 1.0005959272384644, -0.04907112941145897, -0.018952861428260803, 0.9558584094047546, 1.0153886079788208, 0.3891404867172241, 1.0464807748794556, -0.0957864299416542, -0.03201863542199135, 1.0168756246566772, 1.0098919868469238, 1.0110905170440674, -0.07296963036060333, -0.034859079867601395, -0.03349657356739044, 0.10458644479513168, 1.0996088981628418, 1.0925133228302002, -0.015158373862504959, 0.8540427684783936, 0.5354307889938354, 0.6764256954193115, 0.4526764154434204, 0.1361197531223297, 1.0998073816299438, 0.7432370781898499, 0.8964465856552124, -0.09491058439016342, 1.053392767906189, 0.2668561339378357, -0.0015782250557094812, 1.087144374847412, 0.05553637817502022, 1.062805414199829, 0.197163388133049, 0.31309470534324646, 0.8777448534965515, 0.476296067237854, -0.07618403434753418, -0.004727744031697512, 1.0040043592453003, -0.00957558024674654, -0.01797298714518547, 1.0979939699172974, 1.0443764925003052, 1.0990046262741089, 1.0855584144592285, 1.0255218744277954, 0.9793598651885986, 1.000095009803772, -0.05092347413301468, 0.8990545272827148, -0.03203263506293297, -0.09736916422843933, 0.9693222641944885, 1.0140661001205444, -0.09289230406284332, -0.09280998259782791, 1.0912641286849976, 1.0972585678100586, 1.0027108192443848, 1.0490856170654297, 0.9019636511802673, -0.008278445340692997, 0.8827018141746521, 0.17627370357513428, -0.018787316977977753, 0.5164665579795837, 0.17642202973365784, 0.09419506043195724, -0.06938908994197845, 1.0066590309143066, 1.0943645238876343, 1.0612982511520386, -0.09979517012834549, -0.04974866658449173, 0.03522130846977234, 0.9204140305519104, -0.02087302878499031, 0.15630286931991577, 1.057861328125, 0.9812482595443726, -0.05478508770465851, -0.06884149461984634, -0.09769798815250397, 0.8326016664505005, -0.03369783237576485, 0.17078004777431488, 1.0947260856628418, 0.5573217868804932, -0.064590223133564, -0.0996454581618309, 0.7388150095939636, 1.0870146751403809, 1.075868010520935, 0.22511735558509827, -0.02172921411693096, 1.0156108140945435, 0.9799485802650452, 1.0587519407272339, 0.9872462153434753, -0.04788884520530701, 0.00027171772671863437, 0.9363195896148682, 1.0484918355941772, 0.6192111968994141, -4.3654108594637364e-05, 0.082973912358284, 1.0262709856033325, 1.0233083963394165, 1.0927700996398926, 1.0004229545593262, 1.0156424045562744, 1.0198153257369995, 1.0604609251022339, -0.0178122129291296, 1.074398159980774, 1.099839210510254, -0.09515298902988434, 1.099708080291748, 1.0644890069961548, -0.08210094273090363, 0.26288262009620667, 0.056685514748096466, 0.9031242728233337, -0.06738113611936569, 1.0973477363586426, -0.0031082930509001017, 0.03622452914714813, -0.006777944043278694, -0.08696988970041275, 1.0938137769699097, 0.5431702733039856, 0.285745233297348, 0.6358680129051208, 1.0098673105239868, -0.03990665078163147, 1.068400502204895, -0.011842061765491962, -0.08281564712524414, 0.805260419845581, -0.0737149566411972, 0.8684956431388855, 1.090764045715332, -0.031253568828105927, 0.34769147634506226, -0.04226762056350708, -0.002876446582376957, 0.1580430269241333, -0.08157406747341156, 1.0203392505645752, -0.0701904371380806, 1.006455659866333, -0.09610803425312042, 1.0028985738754272, 1.096785306930542, 1.078568935394287, 1.0804365873336792, 1.0662692785263062, 1.0167001485824585, 1.005783200263977, -0.021911831572651863, -0.02653691917657852, -0.00022123486269265413, -0.04864012822508812, 1.0867599248886108, 1.043371319770813, 0.7224735021591187, -0.02448655478656292, -0.09876033663749695, 0.03439441695809364, 1.049623966217041, -0.00520133413374424, 1.0907365083694458, -0.014056854881346226, 0.2034306526184082, -0.09902448952198029, -0.03781384602189064, -0.04663153737783432, -0.03480888903141022, 0.09311772137880325, -0.041920606046915054, 1.0004785060882568, -0.06582522392272949, -0.09740713238716125, -0.09824865311384201, -0.043088316917419434, -0.0004663031722884625, 0.05757591128349304, 0.98551344871521, -0.01191238034516573, 0.496983140707016, -0.029240969568490982, 0.4145064055919647, 1.0257377624511719, 1.085735559463501, 1.068676471710205, -0.0927073210477829, 1.0706714391708374, -0.07927889376878738, 1.0182973146438599, 1.094343900680542, 0.006421709433197975, -0.02338334172964096, 1.099819302558899, -0.03783826157450676, -0.06665679812431335, -0.039365775883197784, 1.0763357877731323, 0.1812945008277893, -0.0009613175643607974, 1.0910913944244385, 1.0018972158432007, 1.004763126373291, -0.09944170713424683, 0.6344159245491028, 1.0906611680984497, 1.0689671039581299, 0.5501468777656555, -0.040115177631378174, 1.0887764692306519, 1.0998191833496094, -0.0202319473028183, -0.0429021380841732, 0.9376852512359619, 0.4729391932487488, 1.0932562351226807, 1.083438515663147, 1.0957528352737427, 0.2580219507217407, -0.01905747875571251, 1.0998255014419556, 1.0174239873886108, 1.0026545524597168, 0.4099990129470825, 0.4049619138240814, 1.0151641368865967, 1.0155351161956787, 1.0072171688079834, 1.093982219696045, -0.0848548635840416, 1.0991902351379395, -0.07551505416631699, 0.20279119908809662, 0.23960843682289124, -0.08687621355056763, -0.0015217559412121773, -0.07540033012628555, 0.9996566772460938, 0.35233500599861145, -0.059611644595861435, 1.0628762245178223, 1.0963826179504395, 1.0764105319976807, -0.07809288054704666, 0.5124668478965759, -0.024147577583789825, -0.06744988262653351, -0.011834731325507164, 0.2785739302635193, -0.06815818697214127, 1.0014631748199463, 0.27744144201278687, 0.08324870467185974, -0.06275193393230438, 1.003791093826294, -0.058260977268218994, 0.17243452370166779, -0.08318942785263062, 0.2316492795944214, 0.39577001333236694, 1.0425808429718018, -0.015227731317281723, 0.42501765489578247, 1.0089493989944458, -0.09522902965545654, 0.14728516340255737, -0.06605514138936996, -0.0002522877184674144, 0.26632964611053467, 1.0345076322555542, -0.08740142732858658, -0.0037337387911975384, 0.9271137118339539, 0.6229898929595947, 1.0548511743545532, 1.067541480064392, 0.030832545831799507, 1.046122431755066, -0.004375542048364878, 1.0409352779388428, 0.8138723969459534, 1.0441861152648926, 1.046873688697815, 0.013855721801519394, 0.1901777982711792, -0.06780853122472763, -0.0017616760451346636, 0.21879249811172485, 0.7602687478065491, -0.040281832218170166, 1.0945112705230713, 0.6601009964942932, 1.0723354816436768, -0.08314871042966843, -0.004916088189929724, 1.0866706371307373, 1.0998140573501587, 0.25685545802116394, -0.07291334867477417, 0.02385721355676651, 0.21052633225917816, -0.09512589126825333, -0.002131922636181116, 1.0858186483383179, -0.01187380962073803, -0.024200934916734695, -0.0034432716201990843, 0.92080157995224, -0.008106635883450508, 1.031516671180725, 1.098767638206482, 1.087491750717163, 1.0263395309448242, -0.07949189841747284, -3.931445462512784e-05, 1.0005571842193604, 1.022830843925476, 1.0352123975753784, -0.09740196168422699, 0.30386224389076233, 0.9588484764099121, -0.004388219211250544, -0.09639408439397812, -0.09858763962984085, 1.00962495803833, -0.09772250056266785, 1.0099871158599854, 0.5181193351745605, -0.006872139405459166, 1.00111722946167, -0.05608249455690384, 1.0996381044387817, 1.0063267946243286, -0.09630955010652542, -0.02809978276491165, 1.0474660396575928, 0.6293567419052124, -0.02836627885699272, -0.042694613337516785, -0.05001214146614075, 1.0575029850006104, 0.030890444293618202, 1.0979030132293701, 1.0075281858444214, 1.0250691175460815, 1.0150089263916016, 1.0279912948608398, -0.008695779368281364, 1.0287301540374756, 1.007589340209961, 0.26110216975212097, -0.0423026941716671, -0.009264166466891766, -0.039383165538311005, 0.45732393860816956, -0.032959163188934326, -0.07888076454401016, -0.09963863343000412, 1.0970360040664673, 0.022707343101501465, 0.22396470606327057, 1.021147608757019, -0.0014111933996900916, 1.0998389720916748, -0.04548385366797447, 1.0080910921096802, 0.7302275896072388, 0.823922336101532, -0.0825033113360405, -0.048745185136795044, 0.01845652237534523, -0.08297063410282135, 1.0483988523483276, 1.0654656887054443, -0.030511464923620224, 1.0993191003799438, -0.08617489784955978, 1.080221176147461, -0.08663659542798996, -0.013479466550052166, 1.050557255744934, 1.078047513961792, -0.0285023245960474, -0.0033968330826610327, -0.09620092064142227, -0.03233293071389198, -0.03410935774445534, 1.0995471477508545, 1.0135936737060547, -0.029222285374999046, 1.0180929899215698, 1.03397536277771, -0.07146897166967392, -0.007183242589235306, 1.0734328031539917, 1.0997594594955444, 1.0006823539733887, 1.0152838230133057, 1.0834678411483765, 1.0066444873809814, -0.046934086829423904, 0.8567265272140503, 0.04962332546710968, 1.0055207014083862, 1.0991339683532715, -5.544262603507377e-05, 1.0181546211242676, -0.06656170636415482, 1.0834009647369385, -0.08366350829601288, 0.9327310919761658, 1.0984400510787964, 0.8688672780990601, -0.060571759939193726, 1.0162560939788818, 1.060892939567566, 0.49435073137283325, 1.0917115211486816, -0.09433898329734802, 1.003282070159912, 0.9165968298912048, 1.0456597805023193, 1.0342377424240112, -0.06727497279644012, -0.034708838909864426, 0.909152090549469, 1.0385518074035645, 1.018816590309143, 1.083568811416626, -0.0439504012465477, 1.0973974466323853, 1.0744589567184448, 1.0497794151306152, 1.026529312133789, -0.030497100204229355, 1.032438039779663, -0.09262264519929886, 1.0998051166534424, -0.010380146093666553, 1.0835274457931519, -0.08601229637861252, 1.0404186248779297, 1.0048893690109253, 1.031730055809021, -0.0354292169213295, -0.06287144124507904, -0.07357829809188843, 1.0958733558654785, 1.033633828163147, 0.8247673511505127, 1.005981683731079, 1.099832534790039, 1.09690523147583, 1.0063129663467407, 0.4204268455505371, 0.8880881071090698, -0.004028673749417067, -0.022252719849348068, -0.09594721347093582, -0.033142101019620895, 0.9066064953804016, 1.0460035800933838, -0.06740611046552658, -0.08505015820264816, -0.09126834571361542, 0.7650025486946106, -0.09652705490589142, 1.0712780952453613, 0.9891499876976013, 1.0246083736419678, 1.0855287313461304, 0.571922779083252, 0.7792847752571106, 1.0052292346954346, 0.41471847891807556, 1.0812050104141235, 0.6104897260665894, -0.07966503500938416, -0.07954337447881699, -0.07830952852964401, 0.5226935744285583, -0.09982145577669144, -0.06715301424264908, 1.029115080833435, 0.24312660098075867, 1.036476969718933, 0.94432133436203, 1.0724178552627563, -0.09972435235977173, 0.7751160860061646, 1.000253438949585, 1.0776466131210327, 1.0525883436203003, 0.9025797843933105, 1.073391079902649, -0.033797238022089005, -0.07315736263990402, 1.0671167373657227, 1.0791172981262207, -0.06342600286006927, 0.14817404747009277, -0.03710860386490822, 0.14613091945648193, -0.0299689881503582, -0.014829753898084164, 0.32173559069633484, -0.005500846542418003, -0.054064542055130005, -0.09660638868808746, -0.04451800137758255, 0.8953571319580078, -0.08080781251192093, 0.31123852729797363, -0.04915621504187584, -0.0867445170879364, 1.0892987251281738, 1.0188075304031372, -0.08788329362869263, 1.0177710056304932, 0.80605548620224, 0.7573046684265137, 1.0098717212677002, -0.06845411658287048, 1.0515073537826538, 1.0522133111953735, 0.057369817048311234, -0.031207159161567688, 1.0974810123443604, 0.9033371806144714, 0.5347652435302734, 0.4799594581127167, 0.47380560636520386, -0.06233565881848335, 0.9340911507606506, -0.004026792477816343, 0.37055397033691406, -0.0109557481482625, -0.052450645714998245, 1.096590518951416, 0.014510056003928185, -0.09503457695245743, 1.082681655883789, 0.0023509750608354807, -0.07943767309188843, -0.07466170191764832, 1.013918399810791, 1.0997123718261719, 1.0692510604858398, -0.0271149855107069, 1.0989158153533936, -0.019836783409118652, 1.0672142505645752, -0.03326539695262909, 1.0033494234085083, -0.005621034651994705, 1.0340495109558105, 1.0890897512435913, -0.004117257427424192, 0.017692144960165024, 1.0177146196365356, 1.0433361530303955, -0.006052523851394653, -0.09956692904233932, -0.02775450237095356, -0.042130935937166214, -0.09356708824634552, 1.0049595832824707, 0.6395136117935181, 0.02914615161716938, -0.05452987924218178, 1.0998337268829346, 1.0993192195892334, -0.05493459478020668, 1.016704797744751, -0.03568524494767189, -0.017842378467321396, 1.0479445457458496, 1.0076969861984253, -0.061078716069459915, 0.8906999230384827, 1.0275378227233887, 1.0984560251235962, 1.0449150800704956, 1.0562522411346436, 0.8535354733467102, 1.0989187955856323, 1.073744535446167, 0.3184843063354492, 1.0110924243927002, -0.001969602657482028, -0.05457000806927681, 1.0180190801620483, 1.0328084230422974, -0.09304013103246689, 0.9943406581878662, 1.0498452186584473, -0.09945031255483627, 1.093632698059082, 0.1588621437549591, 0.17239494621753693, 0.24531321227550507, 1.04668390750885, -0.03217920660972595, -0.07023198157548904, 1.0998382568359375, -0.024103201925754547, -0.007735575083643198, 1.0918077230453491, 0.3772360682487488, 0.12371295690536499, 0.6551579236984253, -0.04813631251454353, 1.0001164674758911, -0.0466759167611599, -0.0905086100101471, -0.01293408963829279, -0.08603034168481827, 1.0138641595840454, 0.41163647174835205, 1.0958640575408936, -0.09751921147108078, 1.0054662227630615, 0.6957879066467285, -0.02210080996155739, 1.0932806730270386, -0.07087603211402893, 1.09722101688385, 0.649333119392395, 0.8612587451934814, 0.6443371176719666, -0.05683809146285057, -0.009992293082177639, 1.0648261308670044, 0.14017756283283234, 0.8917854428291321, -0.095367930829525, -0.0017397053306922317, -0.0202332753688097, 1.0208251476287842, -0.07404940575361252, -0.07618892192840576, -0.011141337454319, -0.09979725629091263, -0.07266872376203537, 0.03115466795861721, 0.2353481650352478, -0.09426306188106537, -0.09873734414577484, 0.34242740273475647, 0.9349772334098816, 0.12606151401996613, 1.00698721408844, 0.7626636624336243, 0.6828198432922363, -0.09134051203727722, 1.0582085847854614, -0.041475810110569, -0.0067308153957128525, -0.04705928638577461, 1.0977894067764282, -0.029485266655683517, 1.0286375284194946, 0.3188174366950989, -0.009345327503979206, 0.01851877011358738, 1.0061763525009155, -0.062164198607206345, -0.017909595742821693, 1.0741368532180786, 1.0277341604232788, 1.0601495504379272, -0.055602505803108215, 0.724173367023468, 0.9318709373474121, 1.040317416191101, 1.0017998218536377, 0.16986402869224548, 1.0283374786376953, -0.06333566457033157, 1.0938866138458252, -0.05203714221715927, -0.07607914507389069, 1.019552230834961, -0.05690461024641991, 1.079035997390747, -0.09948920458555222, 0.5116767287254333, 0.20849578082561493, -0.05823057144880295, 0.6175329685211182, 1.0516730546951294, 1.000059723854065, -0.02589345909655094, 0.900546133518219, 0.07579980045557022, -0.05466878414154053, 0.8819758296012878, -0.05940970033407211, 0.21750320494174957, 1.0336129665374756, -0.0017067512962967157, 0.8250606060028076, 1.0413932800292969, -0.04895311966538429, 1.0780528783798218, 0.9617294669151306, -0.06559101492166519, 1.0363236665725708, -0.018006471917033195, 1.0992152690887451, 0.013117021881043911, -0.03760388866066933, 0.07337214052677155, -0.06574980169534683, 1.0884631872177124, -0.00995229184627533, 0.05328952893614769, -0.09533045440912247, 1.0122828483581543, -0.03159014508128166, 1.0544501543045044, -0.004193948116153479, 0.9517793655395508, 0.12467864155769348, 0.9850268363952637, -0.0843101218342781, -0.002138570649549365, -0.008317328989505768, 1.0277897119522095, 0.7559844851493835, -0.06027441471815109, 1.0785086154937744, -0.0773896798491478, -0.07818186283111572, -0.023156601935625076, 1.0412969589233398, 1.0035557746887207, -0.046452682465314865, 0.8657037019729614, 1.067944884300232, 1.0021330118179321, -0.0004288898780941963, 0.6032317876815796, 1.0982493162155151, -0.009843714535236359, 0.9974042177200317, 0.704828679561615, 1.0991781949996948, -0.09631181508302689, 1.0040981769561768, -0.09408850967884064, 1.0929913520812988, 0.6264685392379761, -0.08395599573850632, -0.011926509439945221, 1.010406494140625, 1.0508654117584229, 1.0834344625473022, -0.08682388067245483, -0.07576299458742142, 1.009699821472168, 1.0982439517974854, -0.05989181622862816, 0.40387633442878723, 0.5935801863670349, 0.024130357429385185, -0.04793542996048927, 1.0342165231704712, 1.090169906616211, 0.02904556877911091, -0.007252940442413092, -0.09644568711519241, 0.31654122471809387, 1.0841686725616455, -0.07975716888904572, 1.0498796701431274, 0.6080145835876465, -0.07697874307632446, -0.00010561899398453534, -0.02594096027314663, -0.009747229516506195, 0.09317908436059952, 0.9296508431434631, 1.022204041481018, 0.32042497396469116, -0.027597278356552124, 1.0009055137634277, 0.9082966446876526, -0.016661854460835457, -0.015274853445589542, -0.015507177449762821, 1.0986685752868652, 0.9820284843444824, 1.0117216110229492, 1.0488208532333374, 1.0816588401794434, 1.0317747592926025, -0.027103612199425697, -0.09307079762220383, 0.273510217666626, -0.02201422117650509, 0.1260371208190918, -0.05266139656305313, 1.0998202562332153, 0.3474263846874237, -0.0025968921836465597, 0.43908408284187317, 1.0193109512329102, -0.003360155038535595, -0.07870429754257202, 0.41938865184783936, -0.017785396426916122, -0.09981922060251236, 0.5019599199295044, -0.018305882811546326, 0.9165492057800293, 1.0292997360229492, -0.02156468667089939, 0.4528743326663971, 1.0487951040267944, 0.10471326857805252, 1.0490046739578247, 1.099830150604248, 0.03733418136835098, 1.0611985921859741, 1.0173419713974, -0.04827288165688515, 1.0862507820129395, 1.077831745147705, 1.086163878440857, -0.09563598036766052, 1.066098690032959, 1.0205528736114502, 0.8535954356193542, 1.0353963375091553, 1.025246024131775, 0.3636944890022278, 1.0992566347122192, -0.0013715813402086496, 1.0477815866470337, 0.2693614959716797, -0.09947787970304489, 0.3944000005722046, 0.3061061203479767, 1.0285255908966064, 1.0735013484954834, -0.03160156309604645, -0.09626603871583939, 1.0137168169021606, 1.0799312591552734, 0.6258693933486938, 1.0002738237380981, 1.0406655073165894, 0.3157005310058594, 1.0927163362503052, 1.0213702917099, 1.0255440473556519, -0.005317732226103544, 0.6693328619003296, 0.5821897387504578, 1.001526117324829, -0.003981852903962135, 1.0108091831207275, 0.20071223378181458, 1.062436819076538, 0.024648619815707207, 0.8242841958999634, 1.0370776653289795, 0.6703655123710632, 1.0454132556915283, 0.34986618161201477, -0.05785475671291351, 0.12118998169898987, -0.089927539229393, 0.2828521430492401, 1.01845383644104, -0.03434096649289131, -0.00759139796718955, 0.4853267967700958, 1.0065165758132935, 1.0452131032943726, 1.020506501197815, 1.0253936052322388, -0.09249480068683624, 0.7570900917053223, -0.0002446380676701665, -0.09978830814361572, 1.0993648767471313, -0.010933523997664452, -0.0020410153083503246, -0.024434149265289307, 0.9165733456611633, 1.0314691066741943, 1.0002737045288086, 1.0140761137008667, -0.012846661731600761, -0.0008579472196288407, 0.8346324563026428, 1.041283369064331, -0.009474956430494785, -0.08734451979398727, -0.06051793694496155, 0.6508115530014038, 0.2425537407398224, 1.0987855195999146, 1.0398353338241577, 0.7365567684173584, 1.0807321071624756, 1.0019874572753906, -0.0573447085916996, -0.0540817528963089, 1.0991154909133911, 1.003143310546875, 0.9033364057540894, 1.0345560312271118, 0.0831172987818718, 1.0547094345092773, -0.09831880778074265, 1.0885347127914429, 1.083350658416748, 1.0836776494979858, 1.014227032661438, 1.0355737209320068, -0.0937369242310524, 1.0566898584365845, -0.008592372760176659, 1.0430136919021606, 1.0001198053359985, 1.0994065999984741, 1.0998224020004272, 0.8038957118988037, 1.0616867542266846, -0.021641267463564873, 1.0842633247375488, 0.1982003003358841, 0.007072023116052151, 0.9141136407852173, 0.6129006743431091, -0.008457598276436329, 1.00118887424469, 0.9412999749183655, -0.009750111028552055, 1.0306144952774048, 0.68624347448349, 1.0077831745147705, 1.076661229133606, -0.0854056105017662, -0.00013031969137955457, 0.5503551959991455, 1.036367416381836, 1.0376005172729492, -0.09915121644735336, 0.23154674470424652, 1.0949640274047852, 0.1521761268377304, 0.261625200510025, 1.0445122718811035, -0.07090635597705841, 1.0062483549118042, -0.08164947479963303, 0.8010158538818359, 0.13184794783592224, -0.09964016824960709, 1.097581148147583, 1.0368925333023071, 1.0095592737197876, 1.0934433937072754, -0.005069623701274395, 1.0546422004699707, -0.021381601691246033, 0.8160296678543091, 1.0985982418060303, 1.09555983543396, -0.04837821424007416, -0.017885733395814896, 1.0049967765808105, -0.05463824048638344, -0.01100834645330906, -0.03746265545487404, 0.7344263792037964, 1.0238127708435059, 0.6988487839698792, 1.0226316452026367, 1.0070271492004395, 1.0992209911346436, 1.0425236225128174, 0.0390348918735981, 1.0510002374649048, 0.023831631988286972, -0.0973205491900444, 1.0884405374526978, -0.090427465736866, 0.8129338026046753, -0.0935426652431488, 0.4925413131713867, -0.07497022300958633, 1.0328381061553955, 0.7988793849945068, 0.7919129133224487, 1.005946397781372, -0.00012877772678621113, -0.04666513577103615, -0.00021816631488036364, 0.6078125238418579, 1.0901930332183838, 1.093971610069275, 1.002534031867981, -0.07836762815713882, 1.0045751333236694, 1.0265902280807495, 0.11808693408966064, -0.04318981245160103, 1.0814794301986694, -0.07743070274591446, -0.061101045459508896, 0.9208219647407532], "dtype": "float32"}]}}, {"a": [{"__nd__": [[[0.25383704900741577, -0.7863626480102539, 1.286618947982788, -0.9701332449913025, -1.0861010551452637, 0.33748188614845276, 0.15581680834293365, 0.09723509103059769, -0.38538095355033875], [0.06759358942508698, -0.5387552976608276, -1.6251484155654907, 0.7581910490989685, -0.9910189509391785, -0.20723098516464233, 0.7522143721580505, 0.04475650191307068, 2.7624144554138184], [-1.379819631576538, -0.7292047142982483, 0.10678300261497498, 1.3744405508041382, -0.2313249260187149, -1.41489839553833, 1.1594443321228027, -0.9062055945396423, -0.1834375411272049], [0.8957656621932983, -0.6558253169059753, -0.43880072236061096, 0.5000855326652527, -0.1919773668050766, 0.297719269990921, -0.9359434247016907, -0.1090845912694931, -1.0825095176696777], [-1.548913598060608, 1.7637443542480469, 0.4617643654346466, 0.9235436320304871, -0.9614047408103943, -0.022643115371465683, -0.1997973918914795, -1.4154510498046875, 0.551532506942749], [-0.7170168161392212, -0.19801665842533112, -1.5124908685684204, 0.9974308013916016, 0.7635931968688965, 0.39671701192855835, -0.9050182700157166, -0.5346614718437195, 1.2260791063308716], [-0.27257323265075684, 0.2710590362548828, 0.14485576748847961, -1.1144986152648926, 1.528316617012024, -1.9798102378845215, 0.22801780700683594, -0.08354317396879196, -1.6138184070587158]], [[0.33607324957847595, 0.5242025256156921, 0.05571998655796051, -0.8236974477767944, 0.4540460705757141, 1.1187982559204102, 0.22526311874389648, 1.7280607223510742, 0.8633707165718079], [0.7738565802574158, 1.394853115081787, -0.6567584276199341, -0.5165150761604309, 0.5204017758369446, 0.5243023037910461, 1.5143526792526245, 1.1509201526641846, 0.18134093284606934], [0.22525347769260406, -0.2678124010562897, -0.3561781048774719, -0.44199833273887634, 0.9148128628730774, -0.45901742577552795, -0.3419206738471985, 0.17331182956695557, -0.4972454607486725], [-0.21193507313728333, -2.164274215698242, 0.39416566491127014, -0.2830442488193512, 0.21655137836933136, 0.395698606967926, -1.824933409690857, 2.4375529289245605, 0.20156878232955933], [0.2297794222831726, -1.7245489358901978, 0.6920809149742126, 0.4913942813873291, -0.27156245708465576, 0.34509071707725525, 1.389080286026001, -0.21988987922668457, -0.781312882900238], [1.460516333580017, 0.40035223960876465, -0.037999268621206284, 2.4435250759124756, 0.30446484684944153, -0.02715979889035225, 0.4132464528083801, -0.2543710172176361, 0.36088570952415466], [0.29522940516471863, -0.5112581253051758, -0.11616485565900803, 0.45170044898986816, 0.5226628184318542, 0.7210120558738708, 0.9219391942024231, 0.38448405265808105, 0.6508653163909912]], [[-0.24818766117095947, -0.12627585232257843, 1.1667931079864502, -1.8491159677505493, 0.49371641874313354, 3.2818069458007812, -1.734318733215332, 1.6134370565414429, -1.6482142210006714], [0.4650425612926483, 1.7209542989730835, 1.5779813528060913, -0.8211223483085632, -1.219365119934082, -1.3614988327026367, 0.7885647416114807, -0.32300296425819397, 0.07014422118663788], [0.3197571635246277, 0.4688089191913605, 0.5914093255996704, 0.1775089055299759, -1.5537521839141846, 1.5059634447097778, -2.697430372238159, 1.2834581136703491, -0.5565227270126343], [-0.15229374170303345, -0.4782603085041046, -0.32656779885292053, 0.020421765744686127, -0.23861096799373627, -1.0573861598968506, 0.8892454504966736, -0.5405218005180359, 1.3817715644836426], [-0.11449361592531204, -0.667657732963562, 0.47478678822517395, 0.925800621509552, -0.886284589767456, -0.24611565470695496, 1.464553713798523, -0.9611939787864685, -0.8127199411392212], [0.20388488471508026, -1.1775338649749756, -2.427327871322632, -1.6307036876678467, 0.411334753036499, 0.691758930683136, 0.6292776465415955, 0.42758119106292725, -1.1959296464920044], [0.7144575119018555, 1.189182162284851, -0.660447895526886, 0.9798712134361267, -0.3294444978237152, 0.2280895859003067, 0.4057886302471161, -1.447401523590088, 1.5044094324111938]], [[-0.8167674541473389, -0.06182324141263962, -0.4543454945087433, 0.4977503716945648, 0.4717484712600708, 0.06107362359762192, 1.4539319276809692, 0.32284218072891235, -0.3588906228542328], [-2.839308977127075, -0.24120305478572845, -0.24952559173107147, -0.8302331566810608, -0.8196539878845215, 0.6341829299926758, 1.0859260559082031, 1.935760498046875, 1.1834930181503296], [-0.6665120720863342, 0.2973428964614868, -1.3222918510437012, 0.7718214392662048, 0.39883729815483093, -0.5694808959960938, -0.0642252266407013, -0.20915745198726654, -0.5952231884002686], [1.3621968030929565, 3.1466548442840576, 0.8654130697250366, 1.0998727083206177, 0.20223990082740784, -0.3074783980846405, 2.237739086151123, -0.3233644962310791, 0.4386433959007263], [-0.07648865878582001, 0.6335176825523376, -0.0064241401851177216, 0.6360477805137634, -1.2237902879714966, -0.058222025632858276, 1.4131340980529785, -0.990725576877594, -0.6247061491012573], [1.4133023023605347, 0.447041392326355, 1.1113066673278809, -2.1952526569366455, 0.18756863474845886, -1.2339162826538086, -1.350220799446106, 0.4528181850910187, 1.225705623626709], [-1.456261396408081, -1.0676881074905396, -0.5444940328598022, -0.08709894865751266, -1.2927582263946533, 2.301841974258423, 0.6270906329154968, -0.3229014575481415, -0.48251843452453613]], [[2.3657214641571045, 2.281644821166992, -0.45462456345558167, 0.38317325711250305, 1.2971643209457397, -1.1542186737060547, -0.5561326146125793, 1.3869233131408691, -0.6646700501441956], [-0.4027581214904785, -1.3222121000289917, -0.7345042824745178, 0.7160645723342896, -0.799659788608551, -1.285730242729187, -0.6070308685302734, 1.8023414611816406, 0.19950492680072784], [-0.5427983999252319, 0.17368720471858978, 0.46392592787742615, 0.1400620937347412, -1.79904043674469, 0.3165435791015625, -0.19950278103351593, 0.8622168302536011, -1.34687340259552], [1.0669448375701904, -0.48700228333473206, -1.7070789337158203, 1.348166584968567, -0.5101401805877686, 0.38389095664024353, -0.15568023920059204, 1.1064070463180542, 1.1178319454193115], [1.1274888515472412, 0.7235288023948669, -1.6225941181182861, -1.690760612487793, 0.20374658703804016, -0.9588313102722168, 0.4703589975833893, 0.16768096387386322, -1.424636721611023], [-1.0448960065841675, -1.1435818672180176, -0.8955109119415283, -0.20996807515621185, 0.6851730942726135, 1.3483935594558716, -0.9804847240447998, -0.5700996518135071, 1.5655848979949951], [0.9001184701919556, 0.23382458090782166, -2.3701975345611572, -1.018689513206482, 0.1574651151895523, -0.5980643033981323, 2.00570011138916, 0.08852750808000565, 0.2830216884613037]]], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [[[0.14294090867042542, -0.24609236419200897, 1.0081616640090942, -0.2666449546813965, -0.2740810811519623, 0.19694724678993225, 0.08396587520837784, 0.05097935348749161, -0.15601366758346558], [0.03493858501315117, -0.1985190212726593, -0.26733049750328064, 0.5162984132766724, -0.2682795822620392, -0.09291758388280869, 0.5112507939338684, 0.02287895418703556, 2.598356246948242], [-0.2773961126804352, -0.23726080358028412, 0.05623944476246834, 1.0969364643096924, -0.10234399884939194, -0.2765597105026245, 0.8826058506965637, -0.2607850432395935, -0.0833299458026886], [0.6360639929771423, -0.2240810990333557, -0.17202164232730865, 0.31129297614097595, -0.08680304139852524, 0.17085659503936768, -0.26367440819740295, -0.05157037824392319, -0.27390891313552856], [-0.2714366316795349, 1.5056697130203247, 0.28326132893562317, 0.6610389351844788, -0.2659216821193695, -0.011193385347723961, -0.0899520143866539, -0.2765447199344635, 0.3499424457550049], [-0.23521752655506134, -0.08923758566379547, -0.27311038970947266, 0.7286761999130249, 0.5208724141120911, 0.23719654977321625, -0.2606636583805084, -0.1975201666355133, 0.9479205012321472], [-0.11782672256231308, 0.15378612279891968, 0.07766453176736832, -0.27531811594963074, 1.2559093236923218, -0.2402283102273941, 0.12695090472698212, -0.040027741342782974, -0.26798930764198303]], [[0.19601012766361237, 0.32926732301712036, 0.028635971248149872, -0.2512107193470001, 0.2776949405670166, 0.8433118462562561, 0.12526406347751617, 1.4674073457717896, 0.6072642207145691], [0.5295922756195068, 1.1177875995635986, -0.2242620885372162, -0.19300489127635956, 0.3264177143573761, 0.32934218645095825, 1.2413229942321777, 0.8743298053741455, 0.09886915236711502], [0.1252581626176834, -0.11608172953128815, -0.14670445024967194, -0.17293843626976013, 0.6531617641448975, -0.17774023115634918, -0.14201436936855316, 0.09414641559123993, -0.18805237114429474], [-0.09478025883436203, -0.22293077409267426, 0.23542927205562592, -0.12162625789642334, 0.1199537068605423, 0.23649075627326965, -0.25337913632392883, 2.2416868209838867, 0.11090762168169022], [0.12803159654140472, -0.2609003782272339, 0.4612232744693756, 0.3048781156539917, -0.11745716631412506, 0.20202527940273285, 1.1118828058242798, -0.09790552407503128, -0.24536114931106567], [1.185371994972229, 0.23971982300281525, -0.018638690933585167, 2.2482547760009766, 0.1752297580242157, -0.013395496644079685, 0.2487190216779709, -0.11109601706266403, 0.21265363693237305], [0.16924791038036346, -0.19166997075080872, -0.054712653160095215, 0.27600860595703125, 0.3281121253967285, 0.48511838912963867, 0.6595894694328308, 0.22875037789344788, 0.42775219678878784]], [[-0.1087731197476387, -0.059156812727451324, 0.8897554874420166, -0.2514367401599884, 0.3065887689590454, 3.163003444671631, -0.260210245847702, 1.3454258441925049, -0.2659400999546051], [0.2856336534023285, 1.4598023891448975, 1.3080177307128906, -0.25087377429008484, -0.2780740261077881, -0.277741402387619, 0.542156457901001, -0.13564318418502808, 0.036301661282777786], [0.18522414565086365, 0.2883651852607727, 0.3806833326816559, 0.09661119431257248, -0.27119967341423035, 1.232574224472046, -0.1702757179737091, 1.0049962997436523, -0.20277039706707], [-0.07035970687866211, -0.18301251530647278, -0.13685670495033264, 0.010315141640603542, -0.10513884574174881, -0.27260395884513855, 0.6302388906478882, -0.19894781708717346, 1.104416012763977], [-0.05397318676114082, -0.2263503223657608, 0.2927137017250061, 0.6630793213844299, -0.25868621468544006, -0.1079905778169632, 1.1895517110824585, -0.26590391993522644, -0.24975794553756714], [0.11229885369539261, -0.27730438113212585, -0.19688642024993896, -0.2670016288757324, 0.24737995862960815, 0.46095916628837585, 0.41049396991729736, 0.25881317257881165, -0.2776950001716614], [0.4796763062477112, 0.9116207957267761, -0.22497430443763733, 0.7124474048614502, -0.13783159852027893, 0.12699490785598755, 0.24350467324256897, -0.27558785676956177, 1.2309547662734985]], [[-0.2502986192703247, -0.0299563966691494, -0.17643511295318604, 0.3095661401748657, 0.29050153493881226, 0.03146902099251747, 1.1785613298416138, 0.18725386261940002, -0.1475859135389328], [-0.1568337380886078, -0.10612690448760986, -0.10927731543779373, -0.252055287361145, -0.2506806254386902, 0.41439878940582275, 0.811853289604187, 1.6916333436965942, 0.9060530066490173], [-0.2261330634355545, 0.17061321437358856, -0.2782594561576843, 0.5278601050376892, 0.2386675477027893, -0.20578570663928986, -0.031081747263669968, -0.0936817079782486, -0.21156452596187592], [1.0844674110412598, 3.016939401626587, 0.6090694069862366, 0.8251643776893616, 0.11131048202514648, -0.13028793036937714, 2.021993398666382, -0.13576652109622955, 0.266666978597641], [-0.036782413721084595, 0.41386863589286804, -0.003201752668246627, 0.41588589549064636, -0.2781309485435486, -0.028263799846172333, 1.1365269422531128, -0.2682575583457947, -0.2178424596786499], [1.1366996765136719, 0.27266645431518555, 0.8361167311668396, -0.21991518139839172, 0.10255411267280579, -0.2782440781593323, -0.27792173624038696, 0.2768118679523468, 0.9475514888763428], [-0.2752913534641266, -0.27316054701805115, -0.19990704953670502, -0.04165411740541458, -0.2784424424171448, 2.092442274093628, 0.40875616669654846, -0.13560853898525238, -0.18415683507919312]], [[2.1626851558685303, 2.070240020751953, -0.17651334404945374, 0.22784946858882904, 1.018737554550171, -0.2766885757446289, -0.20267850160598755, 1.1096782684326172, -0.22578266263008118], [-0.16136497259140015, -0.278260201215744, -0.23813150823116302, 0.4810091257095337, -0.24797312915325165, -0.2784588038921356, -0.21412213146686554, 1.547190546989441, 0.10967014729976654], [-0.19949841499328613, 0.09436651319265366, 0.28482502698898315, 0.07492739707231522, -0.255405992269516, 0.1831146627664566, -0.08983392268419266, 0.6062450408935547, -0.27797043323516846], [0.7938235402107239, -0.18535305559635162, -0.26210999488830566, 1.0702146291732788, -0.19138452410697937, 0.22834263741970062, -0.07179324328899384, 0.8314191102981567, 0.8423829078674316], [0.8516771197319031, 0.48721247911453247, -0.2674804627895355, -0.26321059465408325, 0.11221570521593094, -0.26570388674736023, 0.2894912660121918, 0.09085328131914139, -0.2762879431247711], [-0.2718886435031891, -0.2763618230819702, -0.2596747875213623, -0.09400270134210587, 0.45556628704071045, 1.070444941520691, -0.26747193932533264, -0.20592789351940155, 1.2949786186218262], [0.6399606466293335, 0.1305188536643982, -0.20258954167366028, -0.27024075388908386, 0.08491859585046768, -0.21218523383140564, 1.7678126096725464, 0.04622175544500351, 0.16140355169773102]]], "dtype": "float32"}, {"__nd__": [[[0.6255686283111572, 0.14387252926826477, 1.0017664432525635, 0.08149741590023041, 0.04743735119700432, 0.6655917167663574, 0.5775942802429199, 0.5485410690307617, 0.31197506189346313], [0.5337710976600647, 0.2431078553199768, -0.058859627693891525, 0.8456801772117615, 0.07505745440721512, 0.397121399641037, 0.8434345722198486, 0.5223707556724548, 1.0949255228042603], [-0.020591020584106445, 0.16530582308769226, 0.5532901883125305, 1.0195717811584473, 0.38536086678504944, -0.027040038257837296, 0.9719699621200562, 0.10203979909420013, 0.40879303216934204], [0.8944871425628662, 0.19416050612926483, 0.28744223713874817, 0.7399989366531372, 0.404597669839859, 0.6466894745826721, 0.09232853353023529, 0.44556567072868347, 0.04843004420399666], [-0.04862595349550247, 1.0739905834197998, 0.7229322195053101, 0.903654932975769, 0.08422848582267761, 0.4886793792247772, 0.40076199173927307, -0.027138926088809967, 0.7623981833457947], [0.16999587416648865, 0.40163490176200867, -0.043224919587373734, 0.9268926978111267, 0.847701370716095, 0.6932756900787354, 0.10243311524391174, 0.2448800951242447, 0.9881848692893982], [0.3653823435306549, 0.6338880062103271, 0.5721753835678101, 0.03972775861620903, 1.0456132888793945, -0.08974016457796097, 0.6130285859107971, 0.45827698707580566, -0.05742809176445007]], [[0.664926290512085, 0.7505745887756348, 0.527845561504364, 0.1303827166557312, 0.7194572687149048, 0.9614180326461792, 0.6116862297058105, 1.0705018043518066, 0.8835005760192871], [0.8515179753303528, 1.0233967304229736, 0.1937844455242157, 0.25278225541114807, 0.7489166259765625, 0.750618040561676, 1.0435092449188232, 0.9697988033294678, 0.590175986289978], [0.6116815209388733, 0.3676774203777313, 0.32560598850250244, 0.28599122166633606, 0.9007986783981323, 0.2783030867576599, 0.33231332898139954, 0.5862240791320801, 0.26125505566596985], [0.3948204219341278, -0.09696295112371445, 0.692095935344696, 0.3603450655937195, 0.6074353456497192, 0.6928048729896545, -0.07935629040002823, 1.0997740030288696, 0.6001060009002686], [0.6138867139816284, -0.07014361023902893, 0.8202800750732422, 0.7361559271812439, 0.365869402885437, 0.6691808700561523, 1.0223267078399658, 0.39093467593193054, 0.14572831988334656], [1.0349218845367432, 0.694954514503479, 0.48100489377975464, 1.0997521877288818, 0.6499127149581909, 0.48642173409461975, 0.700889527797699, 0.3741728663444519, 0.6766012907028198], [0.6454980373382568, 0.2550854980945587, 0.44204795360565186, 0.7183988094329834, 0.7499032616615295, 0.8315460681915283, 0.9031317830085754, 0.687608540058136, 0.8038365244865417]], [[0.3771684765815735, 0.4370294511318207, 0.9738240838050842, -0.08127047121524811, 0.7371842861175537, 1.0783021450042725, -0.07113336026668549, 1.0573794841766357, -0.061680104583501816], [0.7244043946266174, 1.0697742700576782, 1.0526962280273438, 0.13129998743534088, 0.013388460502028465, -0.017086241394281387, 0.8569344878196716, 0.34126344323158264, 0.5350434184074402], [0.6571953296661377, 0.726093053817749, 0.7793303728103638, 0.5882905721664429, -0.04931811988353729, 1.042220950126648, -0.0964018777012825, 1.0010839700698853, 0.23546187579631805], [0.42414647340774536, 0.2696825861930847, 0.33957257866859436, 0.5102102160453796, 0.3818170428276062, 0.05548515543341637, 0.8923012614250183, 0.24234436452388763, 1.0209589004516602], [0.44287803769111633, 0.1894087940454483, 0.7287670969963074, 0.904389500617981, 0.10869553685188293, 0.3781733214855194, 1.0355925559997559, 0.08429472148418427, 0.1343066692352295], [0.6012405157089233, 0.02349555492401123, -0.09980408847332001, -0.059550389647483826, 0.7000115513801575, 0.8201534152030945, 0.7950438857078552, 0.7074505686759949, 0.018985938280820847], [0.8290140628814697, 0.9793717861175537, 0.1922997236251831, 0.9215219020843506, 0.3382095992565155, 0.613063633441925, 0.6974605917930603, -0.0327136404812336, 1.0419803857803345]], [[0.13285572826862335, 0.46910804510116577, 0.28040769696235657, 0.738968014717102, 0.7274089455604553, 0.5305178165435791, 1.0338190793991089, 0.6586602330207825, 0.32433366775512695], [-0.09293419122695923, 0.38055774569511414, 0.37651991844177246, 0.12806342542171478, 0.13182394206523895, 0.7970525026321411, 0.9525143504142761, 1.0872249603271484, 0.9779765605926514], [0.18986724317073822, 0.6465094089508057, -0.009266069158911705, 0.8507635593414307, 0.6942551732063293, 0.229932963848114, 0.46790945529937744, 0.3961787819862366, 0.21907071769237518], [1.017221450805664, 1.0831449031829834, 0.8842025995254517, 0.9563323855400085, 0.6004348397254944, 0.34864935278892517, 1.098532795906067, 0.3410918712615967, 0.7124863266944885], [0.461792916059494, 0.7967804670333862, 0.4967879354953766, 0.7978144288063049, 0.012350007891654968, 0.4709054231643677, 1.0267237424850464, 0.07514699548482895, 0.20683367550373077], [1.0267540216445923, 0.7162930369377136, 0.9594180583953857, -0.09770697355270386, 0.5932372808456421, 0.009995772503316402, -0.014881637878715992, 0.7189033031463623, 0.988097608089447], [-0.03421052172780037, 0.052568696439266205, 0.2406301498413086, 0.45650553703308105, -0.0030834111385047436, 1.0993798971176147, 0.7941462993621826, 0.34131163358688354, 0.2677856385707855]], [[1.099786639213562, 1.0991625785827637, 0.28028184175491333, 0.6869996786117554, 1.0040218830108643, 0.029358407482504845, 0.23562897741794586, 1.021924614906311, 0.19060510396957397], [0.303935706615448, -0.009249692782759666, 0.16327950358390808, 0.8296359777450562, 0.13902120292186737, -0.001574815483763814, 0.21414344012737274, 1.0774637460708618, 0.5990946888923645], [0.24136145412921906, 0.5864089131355286, 0.7239031791687012, 0.5698027610778809, -0.07717864215373993, 0.6556678414344788, 0.40090638399124146, 0.8831034302711487, -0.014220332726836205], [0.9472220540046692, 0.26579228043556213, -0.06832185387611389, 1.0144761800765991, 0.25557610392570496, 0.6873331069946289, 0.4224731922149658, 0.9581007361412048, 0.9611609578132629], [0.9637165069580078, 0.8325151801109314, -0.05853956565260887, -0.06655921787023544, 0.6011728644371033, 0.08503815531730652, 0.7267871499061584, 0.5834491848945618, -0.02877010963857174], [0.05906492844223976, 0.03208804875612259, 0.10559799522161484, 0.3957822620868683, 0.8175560832023621, 1.0145211219787598, 0.07828884571790695, 0.229669988155365, 1.0509861707687378], [0.8959391713142395, 0.6158556342124939, -0.09979976713657379, 0.06673221290111542, 0.5784083604812622, 0.2178819328546524, 1.0910669565200806, 0.5442059636116028, 0.6396440863609314]]], "dtype": "float32"}]}}, {"a": [{"__nd__": [-20.0, -19.959959030151367, -19.919919967651367, -19.879878997802734, -19.839839935302734, -19.7997989654541, -19.7597599029541, -19.71971893310547, -19.67967987060547, -19.639638900756836, -19.599599838256836, -19.559558868408203, -19.519519805908203, -19.47947883605957, -19.43943977355957, -19.399398803710938, -19.359359741210938, -19.319318771362305, -19.279279708862305, -19.239238739013672, -19.199199676513672, -19.15915870666504, -19.11911964416504, -19.079078674316406, -19.039039611816406, -18.998998641967773, -18.958959579467773, -18.91891860961914, -18.87887954711914, -18.838838577270508, -18.798799514770508, -18.758758544921875, -18.718719482421875, -18.678678512573242, -18.638639450073242, -18.59859848022461, -18.55855941772461, -18.518518447875977, -18.478479385375977, -18.438438415527344, -18.39839744567871, -18.35835838317871, -18.318317413330078, -18.278278350830078, -18.238237380981445, -18.198198318481445, -18.158157348632812, -18.118118286132812, -18.07807731628418, -18.03803825378418, -17.997997283935547, -17.957958221435547, -17.917917251586914, -17.877878189086914, -17.83783721923828, -17.79779815673828, -17.75775718688965, -17.71771812438965, -17.677677154541016, -17.637638092041016, -17.597597122192383, -17.557558059692383, -17.51751708984375, -17.47747802734375, -17.437437057495117, -17.397397994995117, -17.357357025146484, -17.317317962646484, -17.27727699279785, -17.23723793029785, -17.19719696044922, -17.15715789794922, -17.117116928100586, -17.077077865600586, -17.037036895751953, -16.996997833251953, -16.95695686340332, -16.91691780090332, -16.876876831054688, -16.836837768554688, -16.796796798706055, -16.756755828857422, -16.716716766357422, -16.67667579650879, -16.63663673400879, -16.596595764160156, -16.556556701660156, -16.516515731811523, -16.476476669311523, -16.43643569946289, -16.39639663696289, -16.356355667114258, -16.316316604614258, -16.276275634765625, -16.236236572265625, -16.196195602416992, -16.156156539916992, -16.11611557006836, -16.07607650756836, -16.036035537719727, -15.99599552154541, -15.955955505371094, -15.915915489196777, -15.875875473022461, -15.835835456848145, -15.795795440673828, -15.755755424499512, -15.715715408325195, -15.675675392150879, -15.635635375976562, -15.595595359802246, -15.55555534362793, -15.515515327453613, -15.475475311279297, -15.43543529510498, -15.395395278930664, -15.355355262756348, -15.315315246582031, -15.275275230407715, -15.235235214233398, -15.195195198059082, -15.155155181884766, -15.11511516571045, -15.075075149536133, -15.035035133361816, -14.9949951171875, -14.954955101013184, -14.914915084838867, -14.87487506866455, -14.834835052490234, -14.794795036315918, -14.754755020141602, -14.714715003967285, -14.674674987792969, -14.634634971618652, -14.594594955444336, -14.55455493927002, -14.514514923095703, -14.474474906921387, -14.43443489074707, -14.394393920898438, -14.354353904724121, -14.314313888549805, -14.274273872375488, -14.234233856201172, -14.194193840026855, -14.154153823852539, -14.114113807678223, -14.074073791503906, -14.03403377532959, -13.993993759155273, -13.953953742980957, -13.91391372680664, -13.873873710632324, -13.833833694458008, -13.793793678283691, -13.753753662109375, -13.713713645935059, -13.673673629760742, -13.633633613586426, -13.59359359741211, -13.553553581237793, -13.513513565063477, -13.47347354888916, -13.433433532714844, -13.393393516540527, -13.353353500366211, -13.313313484191895, -13.273273468017578, -13.233233451843262, -13.193193435668945, -13.153153419494629, -13.113113403320312, -13.073073387145996, -13.03303337097168, -12.992993354797363, -12.952953338623047, -12.91291332244873, -12.872873306274414, -12.832833290100098, -12.792792320251465, -12.752752304077148, -12.712712287902832, -12.672672271728516, -12.6326322555542, -12.592592239379883, -12.552552223205566, -12.51251220703125, -12.472472190856934, -12.432432174682617, -12.3923921585083, -12.352352142333984, -12.312312126159668, -12.272272109985352, -12.232232093811035, -12.192192077636719, -12.152152061462402, -12.112112045288086, -12.07207202911377, -12.032032012939453, -11.991991996765137, -11.95195198059082, -11.911911964416504, -11.871871948242188, -11.831831932067871, -11.791791915893555, -11.751751899719238, -11.711711883544922, -11.671671867370605, -11.631631851196289, -11.591591835021973, -11.551551818847656, -11.51151180267334, -11.471471786499023, -11.431431770324707, -11.39139175415039, -11.351351737976074, -11.311311721801758, -11.271271705627441, -11.231231689453125, -11.191190719604492, -11.151150703430176, -11.11111068725586, -11.071070671081543, -11.031030654907227, -10.99099063873291, -10.950950622558594, -10.910910606384277, -10.870870590209961, -10.830830574035645, -10.790790557861328, -10.750750541687012, -10.710710525512695, -10.670670509338379, -10.630630493164062, -10.590590476989746, -10.55055046081543, -10.510510444641113, -10.470470428466797, -10.43043041229248, -10.390390396118164, -10.350350379943848, -10.310310363769531, -10.270270347595215, -10.230230331420898, -10.190190315246582, -10.150150299072266, -10.11011028289795, -10.070070266723633, -10.030030250549316, -9.989990234375, -9.949950218200684, -9.909910202026367, -9.86987018585205, -9.829830169677734, -9.789790153503418, -9.749750137329102, -9.709710121154785, -9.669670104980469, -9.629630088806152, -9.58958911895752, -9.549549102783203, -9.509509086608887, -9.46946907043457, -9.429429054260254, -9.389389038085938, -9.349349021911621, -9.309309005737305, -9.269268989562988, -9.229228973388672, -9.189188957214355, -9.149148941040039, -9.109108924865723, -9.069068908691406, -9.02902889251709, -8.988988876342773, -8.948948860168457, -8.90890884399414, -8.868868827819824, -8.828828811645508, -8.788788795471191, -8.748748779296875, -8.708708763122559, -8.668668746948242, -8.628628730773926, -8.58858871459961, -8.548548698425293, -8.508508682250977, -8.46846866607666, -8.428428649902344, -8.388388633728027, -8.348348617553711, -8.308308601379395, -8.268268585205078, -8.228228569030762, -8.188188552856445, -8.148148536682129, -8.108108520507812, -8.068068504333496, -8.02802848815918, -7.987987995147705, -7.947947978973389, -7.907907962799072, -7.867867946624756, -7.8278279304504395, -7.787787914276123, -7.747747898101807, -7.70770788192749, -7.667667865753174, -7.627627849578857, -7.587587356567383, -7.547547340393066, -7.50750732421875, -7.467467308044434, -7.427427291870117, -7.387387275695801, -7.347347259521484, -7.307307243347168, -7.267267227172852, -7.227227210998535, -7.187187194824219, -7.147147178649902, -7.107107162475586, -7.0670671463012695, -7.027027130126953, -6.986987113952637, -6.94694709777832, -6.906907081604004, -6.8668670654296875, -6.826827049255371, -6.7867865562438965, -6.74674654006958, -6.706706523895264, -6.666666507720947, -6.626626491546631, -6.5865864753723145, -6.546546459197998, -6.506506443023682, -6.466466426849365, -6.426426410675049, -6.386386394500732, -6.346346378326416, -6.3063063621521, -6.266266345977783, -6.226226329803467, -6.18618631362915, -6.146146297454834, -6.106106281280518, -6.066066265106201, -6.026026248931885, -5.98598575592041, -5.945945739746094, -5.905905723571777, -5.865865707397461, -5.8258256912231445, -5.785785675048828, -5.745745658874512, -5.705705642700195, -5.665665626525879, -5.6256256103515625, -5.585585594177246, -5.54554557800293, -5.505505561828613, -5.465465545654297, -5.4254255294799805, -5.385385513305664, -5.345345497131348, -5.305305480957031, -5.265265464782715, -5.225225448608398, -5.185184955596924, -5.145144939422607, -5.105104923248291, -5.065064907073975, -5.025024890899658, -4.984984874725342, -4.944944858551025, -4.904904842376709, -4.864864826202393, -4.824824810028076, -4.78478479385376, -4.744744777679443, -4.704704761505127, -4.6646647453308105, -4.624624729156494, -4.584584712982178, -4.544544696807861, -4.504504680633545, -4.4644646644592285, -4.424424648284912, -4.3843841552734375, -4.344344139099121, -4.304304122924805, -4.264264106750488, -4.224224090576172, -4.1841840744018555, -4.144144058227539, -4.104104042053223, -4.064064025878906, -4.02402400970459, -3.9839839935302734, -3.943943977355957, -3.9039039611816406, -3.863863945007324, -3.823823928833008, -3.7837836742401123, -3.743743658065796, -3.7037036418914795, -3.663663625717163, -3.6236236095428467, -3.5835835933685303, -3.543543577194214, -3.5035035610198975, -3.463463544845581, -3.4234235286712646, -3.383383274078369, -3.3433432579040527, -3.3033032417297363, -3.26326322555542, -3.2232232093811035, -3.183183193206787, -3.1431431770324707, -3.1031031608581543, -3.063063144683838, -3.0230231285095215, -2.982982873916626, -2.9429428577423096, -2.902902841567993, -2.8628628253936768, -2.8228228092193604, -2.782782793045044, -2.7427427768707275, -2.702702760696411, -2.6626627445220947, -2.6226227283477783, -2.582582473754883, -2.5425424575805664, -2.50250244140625, -2.4624624252319336, -2.422422409057617, -2.382382392883301, -2.3423423767089844, -2.302302360534668, -2.2622623443603516, -2.222222328186035, -2.1821820735931396, -2.1421420574188232, -2.102102041244507, -2.0620620250701904, -2.022022008895874, -1.9819819927215576, -1.9419419765472412, -1.9019019603729248, -1.8618618249893188, -1.8218218088150024, -1.781781792640686, -1.7417417764663696, -1.7017017602920532, -1.6616616249084473, -1.6216216087341309, -1.5815815925598145, -1.541541576385498, -1.5015015602111816, -1.4614614248275757, -1.4214214086532593, -1.3813813924789429, -1.3413413763046265, -1.30130136013031, -1.261261224746704, -1.2212212085723877, -1.1811811923980713, -1.1411411762237549, -1.1011011600494385, -1.0610610246658325, -1.0210210084915161, -0.9809809923171997, -0.9409409165382385, -0.9009009003639221, -0.8608608841896057, -0.8208208084106445, -0.7807807922363281, -0.7407407164573669, -0.7007007002830505, -0.6606606841087341, -0.620620608329773, -0.5805805921554565, -0.5405405163764954, -0.500500500202179, -0.46046045422554016, -0.42042040824890137, -0.38038039207458496, -0.34034034609794617, -0.3003003001213074, -0.2602602541446686, -0.22022022306919098, -0.18018017709255219, -0.14014014601707458, -0.10010010004043579, -0.060060061514377594, -0.0200200192630291, 0.0200200192630291, 0.060060061514377594, 0.10010010004043579, 0.14014014601707458, 0.18018017709255219, 0.22022022306919098, 0.2602602541446686, 0.3003003001213074, 0.34034034609794617, 0.38038039207458496, 0.42042040824890137, 0.46046045422554016, 0.500500500202179, 0.5405405163764954, 0.5805805921554565, 0.620620608329773, 0.6606606841087341, 0.7007007002830505, 0.7407407164573669, 0.7807807922363281, 0.8208208084106445, 0.8608608841896057, 0.9009009003639221, 0.9409409165382385, 0.9809809923171997, 1.0210210084915161, 1.0610610246658325, 1.1011011600494385, 1.1411411762237549, 1.1811811923980713, 1.2212212085723877, 1.261261224746704, 1.30130136013031, 1.3413413763046265, 1.3813813924789429, 1.4214214086532593, 1.4614614248275757, 1.5015015602111816, 1.541541576385498, 1.5815815925598145, 1.6216216087341309, 1.6616616249084473, 1.7017017602920532, 1.7417417764663696, 1.781781792640686, 1.8218218088150024, 1.8618618249893188, 1.9019019603729248, 1.9419419765472412, 1.9819819927215576, 2.022022008895874, 2.0620620250701904, 2.102102041244507, 2.1421420574188232, 2.1821820735931396, 2.222222328186035, 2.2622623443603516, 2.302302360534668, 2.3423423767089844, 2.382382392883301, 2.422422409057617, 2.4624624252319336, 2.50250244140625, 2.5425424575805664, 2.582582473754883, 2.6226227283477783, 2.6626627445220947, 2.702702760696411, 2.7427427768707275, 2.782782793045044, 2.8228228092193604, 2.8628628253936768, 2.902902841567993, 2.9429428577423096, 2.982982873916626, 3.0230231285095215, 3.063063144683838, 3.1031031608581543, 3.1431431770324707, 3.183183193206787, 3.2232232093811035, 3.26326322555542, 3.3033032417297363, 3.3433432579040527, 3.383383274078369, 3.4234235286712646, 3.463463544845581, 3.5035035610198975, 3.543543577194214, 3.5835835933685303, 3.6236236095428467, 3.663663625717163, 3.7037036418914795, 3.743743658065796, 3.7837836742401123, 3.823823928833008, 3.863863945007324, 3.9039039611816406, 3.943943977355957, 3.9839839935302734, 4.02402400970459, 4.064064025878906, 4.104104042053223, 4.144144058227539, 4.1841840744018555, 4.224224090576172, 4.264264106750488, 4.304304122924805, 4.344344139099121, 4.3843841552734375, 4.424424648284912, 4.4644646644592285, 4.504504680633545, 4.544544696807861, 4.584584712982178, 4.624624729156494, 4.6646647453308105, 4.704704761505127, 4.744744777679443, 4.78478479385376, 4.824824810028076, 4.864864826202393, 4.904904842376709, 4.944944858551025, 4.984984874725342, 5.025024890899658, 5.065064907073975, 5.105104923248291, 5.145144939422607, 5.185184955596924, 5.225225448608398, 5.265265464782715, 5.305305480957031, 5.345345497131348, 5.385385513305664, 5.4254255294799805, 5.465465545654297, 5.505505561828613, 5.54554557800293, 5.585585594177246, 5.6256256103515625, 5.665665626525879, 5.705705642700195, 5.745745658874512, 5.785785675048828, 5.8258256912231445, 5.865865707397461, 5.905905723571777, 5.945945739746094, 5.98598575592041, 6.026026248931885, 6.066066265106201, 6.106106281280518, 6.146146297454834, 6.18618631362915, 6.226226329803467, 6.266266345977783, 6.3063063621521, 6.346346378326416, 6.386386394500732, 6.426426410675049, 6.466466426849365, 6.506506443023682, 6.546546459197998, 6.5865864753723145, 6.626626491546631, 6.666666507720947, 6.706706523895264, 6.74674654006958, 6.7867865562438965, 6.826827049255371, 6.8668670654296875, 6.906907081604004, 6.94694709777832, 6.986987113952637, 7.027027130126953, 7.0670671463012695, 7.107107162475586, 7.147147178649902, 7.187187194824219, 7.227227210998535, 7.267267227172852, 7.307307243347168, 7.347347259521484, 7.387387275695801, 7.427427291870117, 7.467467308044434, 7.50750732421875, 7.547547340393066, 7.587587356567383, 7.627627849578857, 7.667667865753174, 7.70770788192749, 7.747747898101807, 7.787787914276123, 7.8278279304504395, 7.867867946624756, 7.907907962799072, 7.947947978973389, 7.987987995147705, 8.02802848815918, 8.068068504333496, 8.108108520507812, 8.148148536682129, 8.188188552856445, 8.228228569030762, 8.268268585205078, 8.308308601379395, 8.348348617553711, 8.388388633728027, 8.428428649902344, 8.46846866607666, 8.508508682250977, 8.548548698425293, 8.58858871459961, 8.628628730773926, 8.668668746948242, 8.708708763122559, 8.748748779296875, 8.788788795471191, 8.828828811645508, 8.868868827819824, 8.90890884399414, 8.948948860168457, 8.988988876342773, 9.02902889251709, 9.069068908691406, 9.109108924865723, 9.149148941040039, 9.189188957214355, 9.229228973388672, 9.269268989562988, 9.309309005737305, 9.349349021911621, 9.389389038085938, 9.429429054260254, 9.46946907043457, 9.509509086608887, 9.549549102783203, 9.58958911895752, 9.629630088806152, 9.669670104980469, 9.709710121154785, 9.749750137329102, 9.789790153503418, 9.829830169677734, 9.86987018585205, 9.909910202026367, 9.949950218200684, 9.989990234375, 10.030030250549316, 10.070070266723633, 10.11011028289795, 10.150150299072266, 10.190190315246582, 10.230230331420898, 10.270270347595215, 10.310310363769531, 10.350350379943848, 10.390390396118164, 10.43043041229248, 10.470470428466797, 10.510510444641113, 10.55055046081543, 10.590590476989746, 10.630630493164062, 10.670670509338379, 10.710710525512695, 10.750750541687012, 10.790790557861328, 10.830830574035645, 10.870870590209961, 10.910910606384277, 10.950950622558594, 10.99099063873291, 11.031030654907227, 11.071070671081543, 11.11111068725586, 11.151150703430176, 11.191190719604492, 11.231231689453125, 11.271271705627441, 11.311311721801758, 11.351351737976074, 11.39139175415039, 11.431431770324707, 11.471471786499023, 11.51151180267334, 11.551551818847656, 11.591591835021973, 11.631631851196289, 11.671671867370605, 11.711711883544922, 11.751751899719238, 11.791791915893555, 11.831831932067871, 11.871871948242188, 11.911911964416504, 11.95195198059082, 11.991991996765137, 12.032032012939453, 12.07207202911377, 12.112112045288086, 12.152152061462402, 12.192192077636719, 12.232232093811035, 12.272272109985352, 12.312312126159668, 12.352352142333984, 12.3923921585083, 12.432432174682617, 12.472472190856934, 12.51251220703125, 12.552552223205566, 12.592592239379883, 12.6326322555542, 12.672672271728516, 12.712712287902832, 12.752752304077148, 12.792792320251465, 12.832833290100098, 12.872873306274414, 12.91291332244873, 12.952953338623047, 12.992993354797363, 13.03303337097168, 13.073073387145996, 13.113113403320312, 13.153153419494629, 13.193193435668945, 13.233233451843262, 13.273273468017578, 13.313313484191895, 13.353353500366211, 13.393393516540527, 13.433433532714844, 13.47347354888916, 13.513513565063477, 13.553553581237793, 13.59359359741211, 13.633633613586426, 13.673673629760742, 13.713713645935059, 13.753753662109375, 13.793793678283691, 13.833833694458008, 13.873873710632324, 13.91391372680664, 13.953953742980957, 13.993993759155273, 14.03403377532959, 14.074073791503906, 14.114113807678223, 14.154153823852539, 14.194193840026855, 14.234233856201172, 14.274273872375488, 14.314313888549805, 14.354353904724121, 14.394393920898438, 14.43443489074707, 14.474474906921387, 14.514514923095703, 14.55455493927002, 14.594594955444336, 14.634634971618652, 14.674674987792969, 14.714715003967285, 14.754755020141602, 14.794795036315918, 14.834835052490234, 14.87487506866455, 14.914915084838867, 14.954955101013184, 14.9949951171875, 15.035035133361816, 15.075075149536133, 15.11511516571045, 15.155155181884766, 15.195195198059082, 15.235235214233398, 15.275275230407715, 15.315315246582031, 15.355355262756348, 15.395395278930664, 15.43543529510498, 15.475475311279297, 15.515515327453613, 15.55555534362793, 15.595595359802246, 15.635635375976562, 15.675675392150879, 15.715715408325195, 15.755755424499512, 15.795795440673828, 15.835835456848145, 15.875875473022461, 15.915915489196777, 15.955955505371094, 15.99599552154541, 16.036035537719727, 16.07607650756836, 16.11611557006836, 16.156156539916992, 16.196195602416992, 16.236236572265625, 16.276275634765625, 16.316316604614258, 16.356355667114258, 16.39639663696289, 16.43643569946289, 16.476476669311523, 16.516515731811523, 16.556556701660156, 16.596595764160156, 16.63663673400879, 16.67667579650879, 16.716716766357422, 16.756755828857422, 16.796796798706055, 16.836837768554688, 16.876876831054688, 16.91691780090332, 16.95695686340332, 16.996997833251953, 17.037036895751953, 17.077077865600586, 17.117116928100586, 17.15715789794922, 17.19719696044922, 17.23723793029785, 17.27727699279785, 17.317317962646484, 17.357357025146484, 17.397397994995117, 17.437437057495117, 17.47747802734375, 17.51751708984375, 17.557558059692383, 17.597597122192383, 17.637638092041016, 17.677677154541016, 17.71771812438965, 17.75775718688965, 17.79779815673828, 17.83783721923828, 17.877878189086914, 17.917917251586914, 17.957958221435547, 17.997997283935547, 18.03803825378418, 18.07807731628418, 18.118118286132812, 18.158157348632812, 18.198198318481445, 18.238237380981445, 18.278278350830078, 18.318317413330078, 18.35835838317871, 18.39839744567871, 18.438438415527344, 18.478479385375977, 18.518518447875977, 18.55855941772461, 18.59859848022461, 18.638639450073242, 18.678678512573242, 18.718719482421875, 18.758758544921875, 18.798799514770508, 18.838838577270508, 18.87887954711914, 18.91891860961914, 18.958959579467773, 18.998998641967773, 19.039039611816406, 19.079078674316406, 19.11911964416504, 19.15915870666504, 19.199199676513672, 19.239238739013672, 19.279279708862305, 19.319318771362305, 19.359359741210938, 19.399398803710938, 19.43943977355957, 19.47947883605957, 19.519519805908203, 19.559558868408203, 19.599599838256836, 19.639638900756836, 19.67967987060547, 19.71971893310547, 19.7597599029541, 19.7997989654541, 19.839839935302734, 19.879878997802734, 19.919919967651367, 19.959959030151367, 20.0], "dtype": "float32"}], "e": {"__tup__": [{"__nd__": [-4.1223074731533416e-08, -4.282127363808286e-08, -4.4481176075805706e-08, -4.620531512955495e-08, -4.7996010721362836e-08, -4.9855987782621014e-08, -5.178774387104568e-08, -5.3794217080849194e-08, -5.587810392171377e-08, -5.8042576966954584e-08, -6.029053167821985e-08, -6.262541063506433e-08, -6.505031535652961e-08, -6.7568947770269e-08, -7.018468295427738e-08, -7.290149284244762e-08, -7.572300830815948e-08, -7.865353524039165e-08, -8.169697451876345e-08, -8.485798019819413e-08, -8.814073026996994e-08, -9.15502482712327e-08, -9.509106746463658e-08, -9.876858086954599e-08, -1.0258767701998295e-07, -1.0655416815552599e-07, -1.1067332650327444e-07, -1.1495141905015771e-07, -1.1939413013806188e-07, -1.240082241338314e-07, -1.2879979749413906e-07, -1.337761688091632e-07, -1.389439319154917e-07, -1.4431088857236318e-07, -1.4988424368311826e-07, -1.5567236744118418e-07, -1.616829763406713e-07, -1.679251937503068e-07, -1.7440729038753489e-07, -1.8113904332039965e-07, -1.8812974644788483e-07, -1.9538899209692318e-07, -2.0292773683650012e-07, -2.1075595668662572e-07, -2.188855177109872e-07, -2.273271491048945e-07, -2.3609361221588188e-07, -2.451965315231064e-07, -2.546496205013682e-07, -2.6446537049196195e-07, -2.7465864604891976e-07, -2.852428622190928e-07, -2.9623402042489033e-07, -3.076465873164125e-07, -3.194978148712835e-07, -3.3180333502969006e-07, -3.4458167874618084e-07, -3.5784958640761033e-07, -3.716272374276741e-07, -3.859326795918605e-07, -4.0078737129078945e-07, -4.162110087690962e-07, -4.322266420331289e-07, -4.488554452564131e-07, -4.661223726998287e-07, -4.840501901526295e-07, -5.026656708650989e-07, -5.219933996158943e-07, -5.420623097052157e-07, -5.628988333228335e-07, -5.845341775057022e-07, -6.069967639632523e-07, -6.303201303126116e-07, -6.545350288433838e-07, -6.796775551265455e-07, -7.057807920318737e-07, -7.32883734144707e-07, -7.610219086018333e-07, -7.902374932200473e-07, -8.205683457163104e-07, -8.520602250428055e-07, -8.847556500768405e-07, -9.186987881548703e-07, -9.539402299196809e-07, -9.90526245914225e-07, -1.0285111784469336e-06, -1.067944708665891e-06, -1.1088856126661994e-06, -1.1513872095747502e-06, -1.1955130503338296e-06, -1.2413200920491363e-06, -1.2888771152574918e-06, -1.3382455108512659e-06, -1.3894992889618152e-06, -1.4427043879550183e-06, -1.4979406159909558e-06, -1.5552792547168792e-06, -1.6148056829479174e-06, -1.6765970940468833e-06, -1.7407452332918183e-06, -1.8073350247505005e-06, -1.87646026006405e-06, -1.9482167772366665e-06, -2.0227048480592202e-06, -2.1000273591198493e-06, -2.1802918581670383e-06, -2.263609530928079e-06, -2.350096110603772e-06, -2.4398711957474006e-06, -2.533058705012081e-06, -2.6297889235138427e-06, -2.730194637479144e-06, -2.8344149995973567e-06, -2.9425941647787113e-06, -3.0548817449016497e-06, -3.1714330361864995e-06, -3.2924083370744484e-06, -3.4179754493379733e-06, -3.5483069495967356e-06, -3.683583145175362e-06, -3.823990027740365e-06, -3.969721547036897e-06, -4.1209777918993495e-06, -4.277967036614427e-06, -4.440905740921153e-06, -4.610017185768811e-06, -4.785535111295758e-06, -4.967699624103261e-06, -5.1567612899816595e-06, -5.352979769668309e-06, -5.556622909352882e-06, -5.767972197645577e-06, -5.987315489619505e-06, -6.214953828020953e-06, -6.45119962427998e-06, -6.696375749015715e-06, -6.950816896278411e-06, -7.214870947791496e-06, -7.488899882446276e-06, -7.773276593070477e-06, -8.068396709859371e-06, -8.37465086078737e-06, -8.692461051396094e-06, -9.022261110658292e-06, -9.364500328956638e-06, -9.71964436757844e-06, -1.008817707770504e-05, -1.0470599590917118e-05, -1.0867432138184085e-05, -1.1279212230874691e-05, -1.1706499208230525e-05, -1.2149875146860722e-05, -1.260993849427905e-05, -1.3087314982840326e-05, -1.3582649444288108e-05, -1.4096613995207008e-05, -1.4629903489549179e-05, -1.5183239156613126e-05, -1.5757370420033112e-05, -1.6353067621821538e-05, -1.697114021226298e-05, -1.761242310749367e-05, -1.8277774870512076e-05, -1.896809772006236e-05, -1.9684315702761523e-05, -2.0427398339961655e-05, -2.1198344256845303e-05, -2.199818663939368e-05, -2.2827998691354878e-05, -2.3688900910201482e-05, -2.458204653521534e-05, -2.550862882344518e-05, -2.646989014465362e-05, -2.7467121981317177e-05, -2.8501654014689848e-05, -2.9574874133686535e-05, -3.068821388296783e-05, -3.184315937687643e-05, -3.304125493741594e-05, -3.4284097637282684e-05, -3.557337186066434e-05, -3.691073652589694e-05, -3.8297999708447605e-05, -3.9737013139529154e-05, -4.122967948205769e-05, -4.277798507246189e-05, -4.438398900674656e-05, -4.604981586453505e-05, -4.7777677536942065e-05, -4.956986231263727e-05, -5.142872760188766e-05, -5.335675450623967e-05, -5.535648233490065e-05, -5.743053407059051e-05, -5.958167093922384e-05, -6.181271601235494e-05, -6.412661605281755e-05, -6.65264087729156e-05, -6.901525193825364e-05, -7.159641245380044e-05, -7.427329546771944e-05, -7.704940071562305e-05, -7.992836617631838e-05, -8.291396807180718e-05, -8.601011359132826e-05, -8.922085544327274e-05, -9.255037730326876e-05, -9.600303746992722e-05, -9.95833397610113e-05, -0.00010329594078939408, -0.00010714569361880422, -0.00011113758228020743, -0.00011527680908329785, -0.00011956875823670998, -0.00012401898857206106, -0.00012863326992373914, -0.00013341757585294545, -0.00013837810547556728, -0.00014352127618622035, -0.00014885372365824878, -0.00015438244736287743, -0.00016011434490792453, -0.00016605689597781748, -0.00017221778398379683, -0.00017860488151200116, -0.00018522641039453447, -0.00019209091260563582, -0.000199207104742527, -0.00020658413995988667, -0.0002142314478987828, -0.0002221587928943336, -0.00023037631763145328, -0.000238894484937191, -0.0002477241214364767, -0.0002568764903116971, -0.0002663632039912045, -0.0002761963405646384, -0.0002863883273676038, -0.00029695205739699304, -0.0003079009766224772, -0.0003192488511558622, -0.00033101000008173287, -0.00034319929545745254, -0.00035583204589784145, -0.0003689242003019899, -0.00038249220233410597, -0.0003965530195273459, -0.0004111242888029665, -0.00042622428736649454, -0.0004418718453962356, -0.00045808652066625655, -0.00047488854033872485, -0.0004922988591715693, -0.0005103391013108194, -0.0005290316184982657, -0.0005483996937982738, -0.0005684673087671399, -0.0005892592598684132, -0.0006108011584728956, -0.0006331197218969464, -0.000656242948025465, -0.0006801981362514198, -0.000705015379935503, -0.0007307249470613897, -0.0007573585025966167, -0.0007849485846236348, -0.0008135286625474691, -0.0008431339520029724, -0.0008738003089092672, -0.0009055648697540164, -0.0009384665172547102, -0.0009725448326207697, -0.0010078412014991045, -0.0010443981736898422, -0.0010822599288076162, -0.001121471868827939, -0.0011620812583714724, -0.0012041368754580617, -0.0012476890115067363, -0.001292789587751031, -0.0013394927373155951, -0.0013878537574782968, -0.0014379305066540837, -0.001489782240241766, -0.0015434703091159463, -0.0015990580432116985, -0.0016566116828471422, -0.0017161981668323278, -0.001777888392098248, -0.001841754070483148, -0.0019078702898696065, -0.001976314466446638, -0.002047165995463729, -0.0021205071825534105, -0.0021964237093925476, -0.0022750033531337976, -0.0023563369177281857, -0.0024405180010944605, -0.002527643693611026, -0.002617813879624009, -0.0027111335657536983, -0.002807705895975232, -0.0029076433274894953, -0.0030110585503280163, -0.003118069376796484, -0.0032287968788295984, -0.0033433656208217144, -0.00346190482378006, -0.0035845476668328047, -0.0037114315200597048, -0.0038426993414759636, -0.003978494089096785, -0.004118967801332474, -0.0042642769403755665, -0.004414581228047609, -0.004570046439766884, -0.00473084207624197, -0.004897145554423332, -0.005069136619567871, -0.005247003864496946, -0.005430939141660929, -0.005621140357106924, -0.005817812867462635, -0.006021168548613787, -0.0062314230017364025, -0.006448801141232252, -0.0066735330037772655, -0.006905856542289257, -0.007146016228944063, -0.00739426352083683, -0.007650861982256174, -0.007916072383522987, -0.008190171793103218, -0.008473443798720837, -0.008766177110373974, -0.009068676270544529, -0.009381243959069252, -0.009704201482236385, -0.01003787200897932, -0.010382591746747494, -0.010738705284893513, -0.011106567457318306, -0.011486540548503399, -0.011878998950123787, -0.012284329161047935, -0.012702924199402332, -0.013135189190506935, -0.013581541366875172, -0.014042406342923641, -0.014518224634230137, -0.015009448863565922, -0.015516533516347408, -0.016039954498410225, -0.01658019796013832, -0.017137760296463966, -0.01771315559744835, -0.018306901678442955, -0.018919533118605614, -0.019551599398255348, -0.020203664898872375, -0.02087629772722721, -0.02157008647918701, -0.02228563278913498, -0.02302354946732521, -0.023784464225172997, -0.024569012224674225, -0.02537785843014717, -0.026211656630039215, -0.027071097865700722, -0.027956871315836906, -0.02886969968676567, -0.02981027588248253, -0.03077935054898262, -0.03177766501903534, -0.03280598297715187, -0.0338650718331337, -0.034955721348524094, -0.03607872128486633, -0.03723488375544548, -0.0384250283241272, -0.039649978280067444, -0.040910571813583374, -0.04220765829086304, -0.04354209452867508, -0.04491474851965904, -0.04632648825645447, -0.0477781817317009, -0.04927073046565056, -0.05080500245094299, -0.05238189548254013, -0.0540023073554039, -0.05566711351275444, -0.05737718567252159, -0.059133436530828476, -0.060936734080314636, -0.06278795003890991, -0.06468794494867325, -0.06663757562637329, -0.0686376765370369, -0.07068905979394913, -0.07279253005981445, -0.07494886964559555, -0.07715882360935211, -0.07942311465740204, -0.08174244314432144, -0.08411747217178345, -0.08654876798391342, -0.08903693407773972, -0.09158248454332352, -0.09418587386608124, -0.0968475192785263, -0.09956774860620499, -0.10234682261943817, -0.10518494993448257, -0.10808223485946655, -0.1110386997461319, -0.11405422538518906, -0.11712867021560669, -0.12026170641183853, -0.12345294654369354, -0.12670181691646576, -0.13000765442848206, -0.13336962461471558, -0.13678671419620514, -0.1402577906847, -0.14378155767917633, -0.14735642075538635, -0.15098071098327637, -0.15465247631072998, -0.15836955606937408, -0.16212961077690125, -0.16592994332313538, -0.16976775228977203, -0.17363975942134857, -0.1775425672531128, -0.18147245049476624, -0.1854253113269806, -0.1893967241048813, -0.19338199496269226, -0.19737602770328522, -0.20137332379817963, -0.2053680717945099, -0.20935401320457458, -0.21332450211048126, -0.21727243065834045, -0.22119031846523285, -0.22507017850875854, -0.22890353202819824, -0.2326814979314804, -0.2363947331905365, -0.24003329873085022, -0.24358683824539185, -0.24704444408416748, -0.25039470195770264, -0.25362566113471985, -0.25672492384910583, -0.25967949628829956, -0.2624758183956146, -0.2650999426841736, -0.26753732562065125, -0.2697729170322418, -0.2717911899089813, -0.2735760807991028, -0.27511122822761536, -0.2763795256614685, -0.27736374735832214, -0.2780460715293884, -0.27840831875801086, -0.27843207120895386, -0.278098464012146, -0.27738845348358154, -0.2762826979160309, -0.27476176619529724, -0.27280595898628235, -0.27039551734924316, -0.2675107419490814, -0.2641318142414093, -0.2602391242980957, -0.25581300258636475, -0.250834196805954, -0.24528354406356812, -0.23914234340190887, -0.23239222168922424, -0.22501522302627563, -0.21699398756027222, -0.2083117663860321, -0.1989523321390152, -0.188900426030159, -0.17814137041568756, -0.16666144132614136, -0.1544477939605713, -0.14148861169815063, -0.1277730017900467, -0.11329121887683868, -0.09803463518619537, -0.08199575543403625, -0.06516827642917633, -0.04754713177680969, -0.02912849932909012, -0.009909813292324543, 0.010110205970704556, 0.030931560322642326, 0.0525529682636261, 0.07497186958789825, 0.09818442910909653, 0.1221855878829956, 0.1469690501689911, 0.17252731323242188, 0.19885174930095673, 0.22593256831169128, 0.25375896692276, 0.2823190987110138, 0.31160008907318115, 0.34158816933631897, 0.37226882576942444, 0.40362659096717834, 0.4356454312801361, 0.4683084487915039, 0.5015983581542969, 0.53549724817276, 0.5699866414070129, 0.605047881603241, 0.6406618356704712, 0.6768090724945068, 0.7134702205657959, 0.750625491142273, 0.7882550954818726, 0.8263393640518188, 0.8648585081100464, 0.9037927985191345, 0.9431227445602417, 0.982829213142395, 1.0228930711746216, 1.0632953643798828, 1.104017734527588, 1.1450419425964355, 1.1863502264022827, 1.227925419807434, 1.2697503566741943, 1.311808705329895, 1.3540842533111572, 1.3965617418289185, 1.4392259120941162, 1.4820622205734253, 1.5250568389892578, 1.5681960582733154, 1.6114671230316162, 1.6548576354980469, 1.6983551979064941, 1.7419487237930298, 1.7856272459030151, 1.8293805122375488, 1.8731985092163086, 1.917072057723999, 1.9609917402267456, 2.0049498081207275, 2.048938035964966, 2.0929481983184814, 2.136974334716797, 2.18100905418396, 2.225046396255493, 2.269080400466919, 2.313105821609497, 2.3571174144744873, 2.4011099338531494, 2.445080280303955, 2.489022970199585, 2.53293514251709, 2.576812744140625, 2.620652914047241, 2.6644532680511475, 2.7082102298736572, 2.751922130584717, 2.795586347579956, 2.8392012119293213, 2.882765293121338, 2.926276206970215, 2.969733476638794, 3.0131356716156006, 3.05648136138916, 3.0997700691223145, 3.1430015563964844, 3.1861746311187744, 3.2292888164520264, 3.2723445892333984, 3.3153412342071533, 3.358278751373291, 3.4011566638946533, 3.4439756870269775, 3.4867360591888428, 3.52943754196167, 3.5720810890197754, 3.6146669387817383, 3.6571950912475586, 3.6996660232543945, 3.7420811653137207, 3.784440755844116, 3.82674503326416, 3.868995189666748, 3.911191463470459, 3.9533348083496094, 3.995426654815674, 4.037466526031494, 4.079456329345703, 4.121396064758301, 4.16328763961792, 4.2051310539245605, 4.246927261352539, 4.288677215576172, 4.330381870269775, 4.372042655944824, 4.413659572601318, 4.455234050750732, 4.496766567230225, 4.538258075714111, 4.579710006713867, 4.62112283706665, 4.662497043609619, 4.703834056854248, 4.7451348304748535, 4.786400318145752, 4.827630043029785, 4.868825912475586, 4.909988880157471, 4.951119899749756, 4.992218971252441, 5.033287525177002, 5.0743255615234375, 5.115334510803223, 5.156315326690674, 5.197268486022949, 5.238194942474365, 5.2790937423706055, 5.319967746734619, 5.360816478729248, 5.401640892028809, 5.442441940307617, 5.483220100402832, 5.523975372314453, 5.564709186553955, 5.605422019958496, 5.646114349365234, 5.68678617477417, 5.727438926696777, 5.768072128295898, 5.808688163757324, 5.84928560256958, 5.889865398406982, 5.930429458618164, 5.97097635269165, 6.011508464813232, 6.052023887634277, 6.092524528503418, 6.133011341094971, 6.173483371734619, 6.213942050933838, 6.254386901855469, 6.2948198318481445, 6.335239887237549, 6.375648021697998, 6.416043758392334, 6.456428050994873, 6.496801853179932, 6.537165641784668, 6.577517509460449, 6.617859840393066, 6.6581926345825195, 6.698515892028809, 6.73883056640625, 6.779135704040527, 6.819432735443115, 6.859720706939697, 6.900001049041748, 6.940273761749268, 6.9805378913879395, 7.020795822143555, 7.061046123504639, 7.101289749145508, 7.141525745391846, 7.181756019592285, 7.221980094909668, 7.262198448181152, 7.302410125732422, 7.342616081237793, 7.382816791534424, 7.423012733459473, 7.463202953338623, 7.503388404846191, 7.543569087982178, 7.583744049072266, 7.623916149139404, 7.664083957672119, 7.7042460441589355, 7.7444047927856445, 7.78455924987793, 7.824709892272949, 7.864856243133545, 7.904999732971191, 7.945140838623047, 7.98527717590332, 8.025410652160645, 8.06554126739502, 8.105668067932129, 8.145792007446289, 8.185914039611816, 8.226032257080078, 8.26614761352539, 8.306262016296387, 8.346372604370117, 8.386481285095215, 8.426587104797363, 8.466691017150879, 8.506793022155762, 8.546892166137695, 8.586990356445312, 8.62708568572998, 8.6671781539917, 8.707270622253418, 8.747360229492188, 8.78744888305664, 8.827535629272461, 8.867621421813965, 8.90770435333252, 8.947787284851074, 8.98786735534668, 9.027946472167969, 9.068024635314941, 9.108100891113281, 9.148176193237305, 9.188250541687012, 9.228323936462402, 9.26839542388916, 9.308465957641602, 9.348535537719727, 9.388604164123535, 9.428671836853027, 9.468738555908203, 9.508804321289062, 9.548868179321289, 9.588932991027832, 9.628996849060059, 9.669058799743652, 9.709120750427246, 9.749181747436523, 9.789241790771484, 9.829301834106445, 9.869359970092773, 9.909417152404785, 9.94947624206543, 9.989531517028809, 10.029587745666504, 10.0696439743042, 10.109699249267578, 10.14975357055664, 10.189807891845703, 10.22986125946045, 10.269913673400879, 10.309967041015625, 10.350019454956055, 10.390070915222168, 10.430122375488281, 10.470173835754395, 10.510223388671875, 10.550273895263672, 10.590324401855469, 10.630373001098633, 10.670422554016113, 10.710472106933594, 10.750519752502441, 10.790568351745605, 10.830615997314453, 10.870664596557617, 10.910711288452148, 10.950758934020996, 10.990805625915527, 11.030851364135742, 11.07089900970459, 11.110944747924805, 11.150991439819336, 11.191036224365234, 11.231082916259766, 11.271127700805664, 11.311172485351562, 11.351218223571777, 11.391263008117676, 11.431307792663574, 11.471352577209473, 11.511396408081055, 11.551440238952637, 11.591484069824219, 11.631528854370117, 11.671571731567383, 11.711615562438965, 11.751659393310547, 11.791703224182129, 11.831746101379395, 11.871788024902344, 11.911832809448242, 11.951874732971191, 11.991917610168457, 12.031960487365723, 12.072003364562988, 12.112045288085938, 12.152088165283203, 12.192129135131836, 12.232172012329102, 12.272214889526367, 12.312256813049316, 12.352298736572266, 12.392340660095215, 12.43238353729248, 12.472424507141113, 12.512466430664062, 12.552507400512695, 12.592550277709961, 12.632591247558594, 12.672633171081543, 12.712674140930176, 12.752716064453125, 12.792757034301758, 12.832799911499023, 12.87283992767334, 12.912880897521973, 12.952922821044922, 12.992963790893555, 13.033005714416504, 13.07304573059082, 13.113086700439453, 13.153128623962402, 13.193168640136719, 13.233209609985352, 13.2732515335083, 13.313291549682617, 13.35333251953125, 13.393372535705566, 13.433414459228516, 13.473454475402832, 13.513495445251465, 13.553535461425781, 13.59357738494873, 13.633617401123047, 13.673657417297363, 13.713699340820312, 13.753739356994629, 13.793778419494629, 13.833820343017578, 13.873860359191895, 13.913900375366211, 13.95394229888916, 13.993982315063477, 14.034022331237793, 14.074063301086426, 14.114103317260742, 14.154143333435059, 14.194183349609375, 14.234223365783691, 14.27426528930664, 14.314305305480957, 14.354345321655273, 14.39438533782959, 14.434426307678223, 14.474468231201172, 14.514508247375488, 14.554548263549805, 14.594588279724121, 14.634628295898438, 14.674668312072754, 14.714709281921387, 14.754749298095703, 14.79478931427002, 14.834829330444336, 14.874869346618652, 14.914909362792969, 14.954949378967285, 14.994989395141602, 15.03503131866455, 15.075071334838867, 15.115111351013184, 15.1551513671875, 15.195191383361816, 15.235231399536133, 15.27527141571045, 15.315311431884766, 15.355351448059082, 15.395391464233398, 15.435431480407715, 15.475471496582031, 15.515511512756348, 15.555553436279297, 15.595593452453613, 15.63563346862793, 15.675673484802246, 15.715713500976562, 15.755753517150879, 15.795793533325195, 15.835833549499512, 15.875873565673828, 15.915913581848145, 15.955953598022461, 15.995993614196777, 16.036033630371094, 16.076074600219727, 16.116113662719727, 16.15615463256836, 16.19619369506836, 16.236234664916992, 16.276273727416992, 16.316314697265625, 16.356353759765625, 16.396394729614258, 16.436433792114258, 16.47647476196289, 16.51651382446289, 16.556554794311523, 16.596593856811523, 16.63663673400879, 16.67667579650879, 16.716716766357422, 16.756755828857422, 16.796796798706055, 16.836837768554688, 16.876876831054688, 16.91691780090332, 16.95695686340332, 16.996997833251953, 17.037036895751953, 17.077077865600586, 17.117116928100586, 17.15715789794922, 17.19719696044922, 17.23723793029785, 17.27727699279785, 17.317317962646484, 17.357357025146484, 17.397397994995117, 17.437437057495117, 17.47747802734375, 17.51751708984375, 17.557558059692383, 17.597597122192383, 17.637638092041016, 17.677677154541016, 17.71771812438965, 17.75775718688965, 17.79779815673828, 17.83783721923828, 17.877878189086914, 17.917917251586914, 17.957958221435547, 17.997997283935547, 18.03803825378418, 18.07807731628418, 18.118118286132812, 18.158157348632812, 18.198198318481445, 18.238237380981445, 18.278278350830078, 18.318317413330078, 18.35835838317871, 18.39839744567871, 18.438438415527344, 18.478479385375977, 18.518518447875977, 18.55855941772461, 18.59859848022461, 18.638639450073242, 18.678678512573242, 18.718719482421875, 18.758758544921875, 18.798799514770508, 18.838838577270508, 18.87887954711914, 18.91891860961914, 18.958959579467773, 18.998998641967773, 19.039039611816406, 19.079078674316406, 19.11911964416504, 19.15915870666504, 19.199199676513672, 19.239238739013672, 19.279279708862305, 19.319318771362305, 19.359359741210938, 19.399398803710938, 19.43943977355957, 19.47947883605957, 19.519519805908203, 19.559558868408203, 19.599599838256836, 19.639638900756836, 19.67967987060547, 19.71971893310547, 19.7597599029541, 19.7997989654541, 19.839839935302734, 19.879878997802734, 19.919919967651367, 19.959959030151367, 20.0], "dtype": "float32"}, {"__nd__": [-3.916192170549948e-08, -4.067591419243399e-08, -4.224817828912819e-08, -4.388109076103319e-08, -4.557683652706146e-08, -4.7337980646489086e-08, -4.916687501577144e-08, -5.1066276540723265e-08, -5.303872185891123e-08, -5.5087198802539206e-08, -5.721442519757147e-08, -5.942363046074206e-08, -6.171773492269494e-08, -6.410022024283535e-08, -6.657425188905108e-08, -6.914356731613225e-08, -7.181157002378313e-08, -7.458229589474286e-08, -7.745942554038265e-08, -8.044730748224538e-08, -8.354987812708714e-08, -8.677184126781867e-08, -9.011746016085453e-08, -9.359177965961862e-08, -9.719939697561131e-08, -1.0094576197161587e-07, -1.0483580581421847e-07, -1.0887541890269858e-07, -1.1306991609671968e-07, -1.1742563543748474e-07, -1.2194831811029871e-07, -1.2664477821999753e-07, -1.3152120459380967e-07, -1.3658493003276817e-07, -1.418426478494439e-07, -1.4730224506820377e-07, -1.5297094080324314e-07, -1.5885723314568168e-07, -1.6496888122219389e-07, -1.7131505103407108e-07, -1.7790441120268952e-07, -1.8474594298822922e-07, -1.9184987820608512e-07, -1.9922555338780512e-07, -2.0688405300006707e-07, -2.148354099062999e-07, -2.2309154701360967e-07, -2.3166330720414408e-07, -2.4056350866885623e-07, -2.4980383273032203e-07, -2.593981207610341e-07, -2.693589351565606e-07, -2.797011688926432e-07, -2.9043837912468007e-07, -3.0158659569679003e-07, -3.1316039894591086e-07, -3.2517709769308567e-07, -3.376523238785012e-07, -3.506048358303815e-07, -3.6405145920070936e-07, -3.7801223129463324e-07, -3.9250548411473574e-07, -4.075526760516368e-07, -4.231735033499717e-07, -4.39391243389764e-07, -4.562270419228298e-07, -4.7370588163175853e-07, -4.91850471462385e-07, -5.106879825689248e-07, -5.302428007780691e-07, -5.505440299202746e-07, -5.716180453418929e-07, -5.934961109232972e-07, -6.16206705217337e-07, -6.3978336584114e-07, -6.642569019277289e-07, -6.896634090480802e-07, -7.16035970071971e-07, -7.434136932715774e-07, -7.718317647231743e-07, -8.013325896172319e-07, -8.319556172864395e-07, -8.637418318357959e-07, -8.967380153990234e-07, -9.309872552876186e-07, -9.665398010838544e-07, -1.003441752800427e-06, -1.041747509589186e-06, -1.0815064115377027e-06, -1.1227773484279169e-06, -1.1656130709525314e-06, -1.2100771300538327e-06, -1.2562264828375191e-06, -1.3041294550930616e-06, -1.3538472103391541e-06, -1.4054533039598027e-06, -1.459013674320886e-06, -1.5146072200877825e-06, -1.5723054502814193e-06, -1.6321929479090613e-06, -1.6943481568887364e-06, -1.758857479217113e-06, -1.8258097043144517e-06, -1.8952970322061446e-06, -1.9674148461490404e-06, -2.0422617126314435e-06, -2.119940518241492e-06, -2.2005576738592936e-06, -2.2842236830911133e-06, -2.3710529148956994e-06, -2.4611647404526593e-06, -2.5546814867993817e-06, -2.6517318474361673e-06, -2.7524479264684487e-06, -2.85696773971722e-06, -2.9654336231033085e-06, -3.077993142142077e-06, -3.194800910932827e-06, -3.3160151815536665e-06, -3.441801709413994e-06, -3.5723312521440675e-06, -3.7077816159580834e-06, -3.848336746159475e-06, -3.994188318756642e-06, -4.145533694099868e-06, -4.3025788727391046e-06, -4.465536676434567e-06, -4.634628567146137e-06, -4.810083282791311e-06, -4.992139110981952e-06, -5.18104116054019e-06, -5.377046818466624e-06, -5.580420292972121e-06, -5.7914357967092656e-06, -6.010379820509115e-06, -6.237546585907694e-06, -6.473243956861552e-06, -6.717787982779555e-06, -6.971509264985798e-06, -7.234749773488147e-06, -7.507869213441154e-06, -7.79122274252586e-06, -8.085198714979924e-06, -8.390190487261862e-06, -8.706608241482172e-06, -9.034875802171882e-06, -9.37543336476665e-06, -9.728738405101467e-06, -1.0095263860421255e-05, -1.0475499038875569e-05, -1.0869952348002698e-05, -1.1279152204224374e-05, -1.170364430436166e-05, -1.2143996173108462e-05, -1.2600793525052723e-05, -1.3074645721644629e-05, -1.3566185771196615e-05, -1.407606487191515e-05, -1.4604964235331863e-05, -1.5153584172367118e-05, -1.5722653188277036e-05, -1.6312929801642895e-05, -1.692519617790822e-05, -1.7560259948368184e-05, -1.821896330511663e-05, -1.8902181182056665e-05, -1.961081761692185e-05, -2.0345805751276202e-05, -2.110811328748241e-05, -2.1898751583648846e-05, -2.271876292070374e-05, -2.356922414037399e-05, -2.445125392114278e-05, -2.536601823521778e-05, -2.631471215863712e-05, -2.729859079408925e-05, -2.8318936529103667e-05, -2.937708813988138e-05, -3.0474433515337296e-05, -3.161241329507902e-05, -3.2792533602332696e-05, -3.401629510335624e-05, -3.5285305784782395e-05, -3.6601239116862416e-05, -3.796580131165683e-05, -3.93807640648447e-05, -4.084797910763882e-05, -4.2369345464976504e-05, -4.394684583530761e-05, -4.5582521124742925e-05, -4.7278492274926975e-05, -4.9036960263038054e-05, -5.086020610178821e-05, -5.275056173559278e-05, -5.471050462801941e-05, -5.674254134646617e-05, -5.884929851163179e-05, -6.103349005570635e-05, -6.329792086035013e-05, -6.564550858456641e-05, -6.807925819884986e-05, -7.060230564093217e-05, -7.321787416003644e-05, -7.592931797262281e-05, -7.874010771047324e-05, -8.16538231447339e-05, -8.467419684166089e-05, -8.780506323091686e-05, -9.105043864110485e-05, -9.441442671231925e-05, -9.790130570763722e-05, -0.00010151550668524578, -0.00010526160622248426, -0.00010914437734754756, -0.00011316869495203719, -0.00011733966675819829, -0.00012166255328338593, -0.00012614281149581075, -0.0001307860657107085, -0.00013559819490183145, -0.0001405853108735755, -0.0001457534817745909, -0.00015110928507056087, -0.0001566594437463209, -0.00016241084085777402, -0.00016837072325870395, -0.00017454652697779238, -0.00018094586266670376, -0.0001875767484307289, -0.0001944474206538871, -0.00020156640675850213, -0.00020894251065328717, -0.0002165849000448361, -0.00022450297547038645, -0.00023270647216122597, -0.00024120557645801455, -0.0002500107220839709, -0.0002591327065601945, -0.0002685827494133264, -0.00027837237576022744, -0.0002885136636905372, -0.00029901883681304753, -0.0003099008754361421, -0.0003211729053873569, -0.0003328487218823284, -0.00034494249848648906, -0.0003574688162188977, -0.0003704428963828832, -0.00038388039683923125, -0.00039779749931767583, -0.00041221096762456, -0.000427138147642836, -0.0004425969673320651, -0.0004586059367284179, -0.0004751841479446739, -0.0004923514206893742, -0.0005101283313706517, -0.0005285358638502657, -0.0005475958460010588, -0.0005673310370184481, -0.0005877651856280863, -0.0006089214002713561, -0.0006308251759037375, -0.0006535021821036935, -0.0006769790779799223, -0.000701283453963697, -0.0007264434243552387, -0.0007524886750616133, -0.000779449415858835, -0.0008073568460531533, -0.0008362434455193579, -0.0008661425090394914, -0.0008970886119641364, -0.0009291174355894327, -0.000962265650741756, -0.0009965713834390044, -0.0010320736328139901, -0.0010688131442293525, -0.0011068315943703055, -0.0011461720569059253, -0.0011868792353197932, -0.001228999113664031, -0.0012725789565593004, -0.0013176680076867342, -0.0013643164420500398, -0.0014125762972980738, -0.0014625019393861294, -0.001514148199930787, -0.0015675730537623167, -0.0016228348249569535, -0.001679994398728013, -0.0017391154542565346, -0.0018002615543082356, -0.0018634999869391322, -0.0019288999028503895, -0.001996531616896391, -0.0020664685871452093, -0.0021387862507253885, -0.0022135621402412653, -0.002290876116603613, -0.0023708122316747904, -0.002453452441841364, -0.002538885921239853, -0.0026272027753293514, -0.0027184959035366774, -0.002812860766425729, -0.002910395385697484, -0.0030112015083432198, -0.003115383442491293, -0.0032230482902377844, -0.0033343075774610043, -0.003449272597208619, -0.003568061627447605, -0.003690794575959444, -0.0038175953086465597, -0.0039485907182097435, -0.004083911422640085, -0.0042236922308802605, -0.004368070978671312, -0.00451718969270587, -0.004671193193644285, -0.004830232355743647, -0.0049944594502449036, -0.00516403466463089, -0.005339117255061865, -0.005519875790923834, -0.005706479772925377, -0.005899103824049234, -0.0060979281552135944, -0.006303135771304369, -0.006514919921755791, -0.00673346733674407, -0.006958978250622749, -0.007191657088696957, -0.007431708741933107, -0.007679349277168512, -0.007934793829917908, -0.008198265917599201, -0.008469994179904461, -0.008750208653509617, -0.009039148688316345, -0.00933705735951662, -0.009644181467592716, -0.009960774332284927, -0.010287094861268997, -0.010623405687510967, -0.010969975031912327, -0.011327075771987438, -0.01169498823583126, -0.012073992751538754, -0.01246438268572092, -0.01286644209176302, -0.013280472718179226, -0.013706776313483715, -0.014145659282803535, -0.014597431756556034, -0.015062406659126282, -0.015540901571512222, -0.016033237800002098, -0.016539743170142174, -0.017060739919543266, -0.017596561461687088, -0.0181475430727005, -0.01871401071548462, -0.019296307116746902, -0.019894760102033615, -0.020509717985987663, -0.021141504868865013, -0.021790463477373123, -0.022456925362348557, -0.02314123325049877, -0.02384369447827339, -0.024564646184444427, -0.025304406881332397, -0.026063287630677223, -0.026841597631573677, -0.027639636769890785, -0.028457699343562126, -0.029296059161424637, -0.030154984444379807, -0.03103473037481308, -0.03193553537130356, -0.03285762667655945, -0.03380120173096657, -0.03476644679903984, -0.0357535257935524, -0.03676257282495499, -0.03779369965195656, -0.03884698078036308, -0.039922475814819336, -0.04102019965648651, -0.042140115052461624, -0.043282151222229004, -0.04444621130824089, -0.045632150024175644, -0.04683973267674446, -0.04806871339678764, -0.04931877925992012, -0.050589535385370255, -0.051880523562431335, -0.053191229701042175, -0.054521042853593826, -0.05586928501725197, -0.05723518133163452, -0.05861787497997284, -0.06001640483736992, -0.061429671943187714, -0.0628565177321434, -0.06429563462734222, -0.06574559956789017, -0.06720485538244247, -0.06867170333862305, -0.07014428824186325, -0.07162062078714371, -0.07309854030609131, -0.07457569986581802, -0.0760495737195015, -0.07751747220754623, -0.07897647470235825, -0.08042346686124802, -0.08185514807701111, -0.0832679346203804, -0.08465802669525146, -0.08602141588926315, -0.08735378831624985, -0.08865059167146683, -0.08990700542926788, -0.09111791104078293, -0.092277891933918, -0.09338125586509705, -0.09442198276519775, -0.09539372473955154, -0.09628988802433014, -0.09710339456796646, -0.09782698005437851, -0.09845296293497086, -0.09897332638502121, -0.09937969595193863, -0.09966336935758591, -0.09981528669595718, -0.09982600808143616, -0.09968578070402145, -0.09938446432352066, -0.09891166538000107, -0.09825654327869415, -0.09740803390741348, -0.09635474532842636, -0.09508492797613144, -0.09358666092157364, -0.0918477401137352, -0.08985571563243866, -0.08759795129299164, -0.08506164699792862, -0.08223388344049454, -0.07910166680812836, -0.07565189898014069, -0.0718715637922287, -0.06774760037660599, -0.06326707452535629, -0.05841729789972305, -0.05318571254611015, -0.04756005108356476, -0.04152839258313179, -0.03507929667830467, -0.028201788663864136, -0.02088540978729725, -0.013120443560183048, -0.0048978570848703384, 0.0037905953358858824, 0.012952165678143501, 0.022593287751078606, 0.032719243317842484, 0.04333416745066643, 0.05444106087088585, 0.0660414919257164, 0.07813586294651031, 0.09072302281856537, 0.10380038619041443, 0.11736380308866501, 0.13140755891799927, 0.14592428505420685, 0.16090507805347443, 0.17633917927742004, 0.1922142058610916, 0.2085161805152893, 0.22522935271263123, 0.24233628809452057, 0.2598179876804352, 0.27765393257141113, 0.29582199454307556, 0.314298540353775, 0.33305880427360535, 0.3520764708518982, 0.3713241517543793, 0.3907734453678131, 0.4103950560092926, 0.4301586151123047, 0.45003342628479004, 0.46998801827430725, 0.4899906814098358, 0.5100093483924866, 0.5300120115280151, 0.54996657371521, 0.5698413848876953, 0.5896050333976746, 0.6092265844345093, 0.6286759376525879, 0.6479235887527466, 0.6669411659240723, 0.6857013702392578, 0.704177975654602, 0.7223460674285889, 0.7401820421218872, 0.7576637268066406, 0.7747706770896912, 0.7914838194847107, 0.8077858090400696, 0.8236608505249023, 0.839094877243042, 0.8540757298469543, 0.8685925006866455, 0.8826362490653992, 0.896199643611908, 0.9092769026756287, 0.9218640923500061, 0.9339584708213806, 0.9455589652061462, 0.9566658139228821, 0.9672808051109314, 0.9774067401885986, 0.9870477914810181, 0.996209442615509, 1.0048978328704834, 1.013120412826538, 1.0208853483200073, 1.028201699256897, 1.0350792407989502, 1.0415284633636475, 1.0475600957870483, 1.0531858205795288, 1.0584173202514648, 1.0632671117782593, 1.0677475929260254, 1.0718715190887451, 1.0756518840789795, 1.0791016817092896, 1.0822339057922363, 1.0850616693496704, 1.0875979661941528, 1.089855670928955, 1.0918477773666382, 1.0935866832733154, 1.0950849056243896, 1.0963547229766846, 1.0974080562591553, 1.0982564687728882, 1.0989116430282593, 1.0993844270706177, 1.099685788154602, 1.0998259782791138, 1.0998152494430542, 1.0996633768081665, 1.0993796586990356, 1.098973274230957, 1.0984530448913574, 1.0978269577026367, 1.0971033573150635, 1.096289873123169, 1.0953938961029053, 1.0944221019744873, 1.0933812856674194, 1.0922778844833374, 1.0911179780960083, 1.0899070501327515, 1.0886505842208862, 1.0873538255691528, 1.0860214233398438, 1.0846580266952515, 1.0832679271697998, 1.0818551778793335, 1.0804234743118286, 1.0789763927459717, 1.0775173902511597, 1.0760496854782104, 1.0745757818222046, 1.0730985403060913, 1.0716205835342407, 1.0701442956924438, 1.0686718225479126, 1.0672048330307007, 1.0657455921173096, 1.0642956495285034, 1.0628563165664673, 1.061429500579834, 1.0600165128707886, 1.0586180686950684, 1.0572352409362793, 1.0558693408966064, 1.0545209646224976, 1.0531911849975586, 1.0518805980682373, 1.0505893230438232, 1.0493186712265015, 1.0480685234069824, 1.046839714050293, 1.045632004737854, 1.0444459915161133, 1.043282151222229, 1.042140007019043, 1.041020393371582, 1.0399225950241089, 1.0388468503952026, 1.037793755531311, 1.0367624759674072, 1.035753607749939, 1.0347665548324585, 1.0338011980056763, 1.0328576564788818, 1.0319355726242065, 1.0310349464416504, 1.0301547050476074, 1.0292960405349731, 1.0284578800201416, 1.027639627456665, 1.026841402053833, 1.0260632038116455, 1.0253040790557861, 1.0245646238327026, 1.0238438844680786, 1.0231412649154663, 1.0224571228027344, 1.0217901468276978, 1.0211414098739624, 1.0205096006393433, 1.0198948383331299, 1.019296407699585, 1.0187140703201294, 1.0181474685668945, 1.0175966024398804, 1.01706063747406, 1.0165396928787231, 1.0160331726074219, 1.0155407190322876, 1.0150623321533203, 1.0145976543426514, 1.0141452550888062, 1.013706922531128, 1.0132806301116943, 1.0128662586212158, 1.0124644041061401, 1.0120738744735718, 1.0116950273513794, 1.0113271474838257, 1.010969877243042, 1.0106234550476074, 1.0102869272232056, 1.0099611282348633, 1.0096440315246582, 1.0093369483947754, 1.0090389251708984, 1.008750319480896, 1.0084701776504517, 1.0081983804702759, 1.0079345703125, 1.0076794624328613, 1.007431983947754, 1.0071918964385986, 1.006959319114685, 1.0067335367202759, 1.0065147876739502, 1.006303310394287, 1.0060982704162598, 1.0058993101119995, 1.005706548690796, 1.005520224571228, 1.0053389072418213, 1.0051639080047607, 1.00499427318573, 1.0048305988311768, 1.0046714544296265, 1.0045174360275269, 1.0043678283691406, 1.0042238235473633, 1.0040841102600098, 1.0039490461349487, 1.0038176774978638, 1.0036909580230713, 1.003568172454834, 1.0034489631652832, 1.003334879875183, 1.0032232999801636, 1.003114938735962, 1.0030109882354736, 1.0029100179672241, 1.0028125047683716, 1.0027185678482056, 1.0026277303695679, 1.0025393962860107, 1.0024532079696655, 1.0023707151412964, 1.0022906064987183, 1.002213478088379, 1.0021388530731201, 1.002066969871521, 1.001996397972107, 1.0019291639328003, 1.0018635988235474, 1.0017998218536377, 1.0017389059066772, 1.0016796588897705, 1.0016223192214966, 1.001567006111145, 1.0015137195587158, 1.001462459564209, 1.001412272453308, 1.0013642311096191, 1.0013179779052734, 1.0012725591659546, 1.0012292861938477, 1.0011873245239258, 1.0011467933654785, 1.0011067390441895, 1.0010690689086914, 1.001031756401062, 1.0009969472885132, 1.000962734222412, 1.0009291172027588, 1.00089693069458, 1.000866413116455, 1.0008364915847778, 1.000807285308838, 1.000779628753662, 1.0007526874542236, 1.0007264614105225, 1.0007009506225586, 1.0006771087646484, 1.0006532669067383, 1.0006309747695923, 1.0006095170974731, 1.0005877017974854, 1.00056791305542, 1.0005477666854858, 1.0005284547805786, 1.0005100965499878, 1.0004924535751343, 1.0004746913909912, 1.0004589557647705, 1.0004428625106812, 1.0004267692565918, 1.0004125833511353, 1.0003982782363892, 1.0003838539123535, 1.0003703832626343, 1.000357747077942, 1.0003451108932495, 1.0003334283828735, 1.000321626663208, 1.000309705734253, 1.0002987384796143, 1.0002888441085815, 1.0002788305282593, 1.0002686977386475, 1.0002596378326416, 1.0002504587173462, 1.0002412796020508, 1.0002330541610718, 1.0002248287200928, 1.0002164840698242, 1.0002092123031616, 1.000201940536499, 1.0001945495605469, 1.0001870393753052, 1.000180721282959, 1.0001744031906128, 1.000167965888977, 1.0001626014709473, 1.000156044960022, 1.0001506805419922, 1.0001451969146729, 1.000140905380249, 1.0001354217529297, 1.0001310110092163, 1.000126600265503, 1.0001221895217896, 1.0001176595687866, 1.0001131296157837, 1.0001085996627808, 1.0001052618026733, 1.000101923942566, 1.000098466873169, 1.0000938177108765, 1.000091552734375, 1.000088095664978, 1.000084638595581, 1.0000810623168945, 1.000078797340393, 1.000076413154602, 1.0000728368759155, 1.0000704526901245, 1.000068187713623, 1.000065803527832, 1.0000633001327515, 1.0000609159469604, 1.0000585317611694, 1.000057339668274, 1.000054955482483, 1.0000524520874023, 1.0000512599945068, 1.0000487565994263, 1.0000475645065308, 1.0000449419021606, 1.0000437498092651, 1.0000425577163696, 1.0000413656234741, 1.000038743019104, 1.000037431716919, 1.0000361204147339, 1.0000349283218384, 1.0000336170196533, 1.0000323057174683, 1.0000309944152832, 1.0000311136245728, 1.0000298023223877, 1.0000284910202026, 1.0000271797180176, 1.0000258684158325, 1.0000258684158325, 1.0000245571136475, 1.0000231266021729, 1.0000232458114624, 1.0000218152999878, 1.0000205039978027, 1.0000205039978027, 1.0000191926956177, 1.0000191926956177, 1.000017762184143, 1.0000178813934326, 1.000016450881958, 1.000016450881958, 1.0000150203704834, 1.0000150203704834, 1.000015139579773, 1.0000135898590088, 1.0000137090682983, 1.0000137090682983, 1.0000122785568237, 1.0000122785568237, 1.0000122785568237, 1.0000108480453491, 1.0000108480453491, 1.0000108480453491, 1.000009298324585, 1.0000094175338745, 1.0000094175338745, 1.0000094175338745, 1.0000094175338745, 1.0000078678131104, 1.0000079870224, 1.0000079870224, 1.0000079870224, 1.0000079870224, 1.0000064373016357, 1.0000064373016357, 1.0000064373016357, 1.0000064373016357, 1.0000065565109253, 1.0000065565109253, 1.0000048875808716, 1.0000048875808716, 1.0000048875808716, 1.0000050067901611, 1.0000050067901611, 1.0000050067901611, 1.0000050067901611, 1.0000050067901611, 1.0000033378601074, 1.0000033378601074, 1.0000033378601074, 1.0000033378601074, 1.0000033378601074, 1.0000033378601074, 1.000003457069397, 1.000003457069397, 1.000003457069397, 1.000003457069397, 1.000003457069397, 1.000003457069397, 1.000003457069397, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000017881393433, 1.0000019073486328, 1.0000019073486328, 1.0000019073486328, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(silu_fwd_bwd, json.loads(_TESTS), 'triton', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")